# Dynamic Context Ablation Experiment: MedGemma-1.5-4B Medical DeepResearch

## Overview
To stress test the generalizability of **ContextBank** — the context management module powering the Declarative DataFlow Architecture (featuring collision-free tool descriptions and a dynamic context bank) — we built an experimental deep research system that pushes the limits of small language models in orchestrating long-horizon tasks while maintaining coherency and strong alignment.

## Architecture
Our system follows a structured **orchestrator-subagent workflow.** In our deep research architecture, a single orchestrator agent (MedGemma-1.5-4B) delegates to multiple specialized subagents(MedGemma-1.5-4B) acting as such:

| Component | Role |
|-------|----------------------|
| **Orchestrator** | Central coordinator that manages the workflow |
| **Planner sub-agent** | Creates research outline and section plan |
| **Search sub-agent** | Performs web searches for each section |
| **Memory sub-agent** | Persists all research artifacts to disk; enables recovery and human intervention |
| **Writer sub-agent** | Drafts content section by section |
| **Verifier sub-agent** | Checks citations and fact-checks claims |

## Persistent Memory & Fault Tolerance

The **Memory Agent** is the backbone of the system's reliability, designed to scale to hundreds of pages while maintaining complete persistence:

### 📁 What Gets Saved
For each research run, the Memory Agent stores everything to the local file system:
- **Research plan** and section outlines
- **Retrieved web content** (raw HTML, cleaned text, source URLs)
- **Draft writeups** for each section
- **Generated images** with metadata
- **Inline citations** and final references
- **Complete execution logs** of all agent actions

### 🔄 Key Capabilities

| Feature | Benefit |
|---------|---------|
| **Failure Recovery** | If a deep research run crashes (API errors, rate limits, timeouts), the system can restart exactly where it left off — no lost progress |
| **Human-in-the-Loop Editing** | After research completes, a human reviewer can inspect the saved artifacts and instruct the agent to: |
| | • Rewrite specific sections |
| | • Correct factual errors |
| | • Add missing citations |
| | • Regenerate images |
| | • Adjust tone or depth |
| **Scalability** | The file-based persistence scales effortlessly to hundreds of pages — no memory limits, no context window constraints |

This design transforms the system from a fragile linear pipeline into a **resilient, auditable, and human-augmentable research workflow**.

## Results

The experiment showed **promising results**. Despite running entirely on a small language model (**MedGemma-1.5-4B**), our DeepResearch system can:

### What It Does
- Searches the web autonomously given a topic
- Generates section-specific images via the **nanobanana API**
- Produces coherent research reports between **35–50 pages**
- Includes inline citations and references with reasonable accuracy
- Survives failures gracefully and supports human corrections

### Why This Matters
This ablation study suggests that **small language models (4B parameters)** can handle complex, multi-step research tasks when given:
- A clear orchestration structure
- Effective context management (ContextBank) despite running on a maximum context length of 8192 for each agent turn.
- **Persistent memory that decouples execution from model context limits**
- Well-defined subagent roles with checkpointable state

These findings push back against the prevailing assumption that sophisticated research workflows require massive models. Instead, we demonstrate that coherent long-horizon task execution can be achieved through thoughtful **architectural design, persistent memory, and failure recovery** — factors that matter just as much, if not more, than raw parameter count. Hence, small specialised models like MedGemma-1.5-4B can produce impressive deep research reports



In [1]:
from typing import Dict, List, Optional, Any, Callable, Literal, Tuple
from dataclasses import dataclass, field
from enum import Enum

In [2]:
# =============================================================================
# TOOL TYPE CLASSIFICATION
# =============================================================================

class ToolType(Enum):
    RETRIEVAL = "retrieval"
    COMPUTATION = "computation"
    KNOWLEDGE = "knowledge"


# =============================================================================
# EXTRACTION STRATEGY HINTS (universal library)
# =============================================================================

class ArgExtractorType(Enum):
    """
    Extraction strategies that tool authors assign to parameters at registration.
    The framework provides these strategies — tool authors CHOOSE which to use.

    ENUM:       Auto-handled from pipe-separated values in param description
    QUOTED:     Extract quoted strings from user goal
    LANGUAGE:   Map language names to codes (mapping provided by tool author)
    NUMERIC_ID: Extract numeric ID following configurable keywords
    NUMBER:     Extract numbers with configurable units
    """
    ENUM = "enum"
    QUOTED = "quoted"
    LANGUAGE = "language"
    NUMERIC_ID = "numeric_id"
    NUMBER = "number"


# =============================================================================
# TOOL SCHEMA: Self-Describing Tools
# =============================================================================

@dataclass
class ToolSchema:
    """
    Rich tool schema with self-describing metadata.
    All domain knowledge lives HERE, not in the framework.
    """
    # --- Core ---
    name: str
    description: str
    parameters: Dict[str, Any]
    required: List[str]
    example: Dict[str, Any]
    docstring: str

    # --- Self-describing metadata ---
    tool_type: ToolType = ToolType.KNOWLEDGE
    arg_sources: Dict[str, str] = field(default_factory=dict)
    output_keys: Dict[str, str] = field(default_factory=dict)
    explicit_keywords: List[str] = field(default_factory=list)

    # --- Extraction hints per argument ---
    # Maps param_name → (ArgExtractorType, config_dict)
    # Config depends on strategy:
    #   QUOTED:     {}
    #   LANGUAGE:   {"mapping": {"german": "de-DE", ...}, "role": "target|source"}
    #   NUMERIC_ID: {"preceding_words": ["patient", "document"]}
    #   NUMBER:     {"units": ["kg", "m", "mg"]}
    #   ENUM:       {}  (auto-extracted from pipe syntax in param description)
    arg_extractors: Dict[str, tuple] = field(default_factory=dict)

    def get_medgemma_format(self) -> str:
        args_str = json.dumps(self.parameters, indent=2)
        return f"- {self.name}: {self.description}\n  args={args_str}"

    def get_bm25_document(self) -> str:
        return f"{self.name} {self.description} {self.docstring} {json.dumps(self.parameters)}"

    def get_functiongemma_example(self) -> str:
        return json.dumps(self.example, indent=2)

    def get_output_extraction_prompt(self) -> str:
        if not self.output_keys:
            return ""
        lines = [f"- {key} ({type_hint})" for key, type_hint in self.output_keys.items()]
        return "\n".join(lines)


print("✅ ToolType + ArgExtractorType + ToolSchema defined")

✅ ToolType + ArgExtractorType + ToolSchema defined


In [3]:
# ===============================================
# Tool Validation Block: Collision-Free Validation
# ================================================

class ToolSchemaValidationError(Exception):
    """Raised when a tool schema fails validation."""
    pass


class ToolSchemaValidator:
    """Validates ToolSchema objects before registration."""

    @staticmethod
    def validate(schema: ToolSchema):
        errors = []

        # ---------------------------------------------------------
        # 1. required ⊆ parameters
        # ---------------------------------------------------------
        for req in schema.required:
            if req not in schema.parameters:
                errors.append(
                    f"Required parameter '{req}' is not defined in parameters."
                )

        # ---------------------------------------------------------
        # 2. arg_sources keys ⊆ parameters
        # ---------------------------------------------------------
        for param in schema.arg_sources:
            if param not in schema.parameters:
                errors.append(
                    f"arg_sources references unknown parameter '{param}'."
                )

        # ---------------------------------------------------------
        # 3. arg_extractors keys ⊆ parameters
        # ---------------------------------------------------------
        for param in schema.arg_extractors:
            if param not in schema.parameters:
                errors.append(
                    f"arg_extractors references unknown parameter '{param}'."
                )

        # ---------------------------------------------------------
        # 4. output_keys must not collide with arg_sources
        # ---------------------------------------------------------
        for out_key in schema.output_keys:
            if out_key in schema.arg_sources.values():
                errors.append(
                    f"output_key '{out_key}' collides with an arg_source context key."
                )

        # ---------------------------------------------------------
        # 5. ENUM parameters must contain pipe syntax
        # ---------------------------------------------------------
        for param_name, desc in schema.parameters.items():
            if "enum" in desc.lower():
                # heuristic: expecting "a|b|c" in description
                if "|" not in desc:
                    errors.append(
                        f"Parameter '{param_name}' is declared ENUM but description "
                        f"does not contain pipe-separated values."
                    )

        # ---------------------------------------------------------
        # 6. Validate extractor configs
        # ---------------------------------------------------------

        for param, (extractor_type, config) in schema.arg_extractors.items():
            if extractor_type == ArgExtractorType.LANGUAGE:
                if "mapping" not in config:
                    errors.append(
                        f"LANGUAGE extractor for '{param}' must include a 'mapping' dict."
                    )
                if "role" not in config:
                    errors.append(
                        f"LANGUAGE extractor for '{param}' must include a 'role' field."
                    )

            elif extractor_type == ArgExtractorType.NUMERIC_ID:
                if "preceding_words" not in config:
                    errors.append(
                        f"NUMERIC_ID extractor for '{param}' must include 'preceding_words'."
                    )

            elif extractor_type == ArgExtractorType.NUMBER:
                if "units" not in config:
                    errors.append(
                        f"NUMBER extractor for '{param}' must include 'units'."
                    )

            elif extractor_type == ArgExtractorType.QUOTED:
                if config not in ({}, None):
                    errors.append(
                        f"QUOTED extractor for '{param}' should have empty config."
                    )

            elif extractor_type == ArgExtractorType.ENUM:
                # ENUM has no config requirements
                pass


        # ---------------------------------------------------------
        # 7. Validate example payload
        # ---------------------------------------------------------
        if "arguments" in schema.example:
            for arg in schema.example["arguments"]:
                if arg not in schema.parameters:
                    errors.append(
                        f"Example references unknown parameter '{arg}'."
                    )

        # ---------------------------------------------------------
        # 8. Validate tool_type
        # ---------------------------------------------------------
        if not isinstance(schema.tool_type, ToolType):
            errors.append(f"Invalid tool_type: {schema.tool_type}")

        # ---------------------------------------------------------
        # Finalize
        # ---------------------------------------------------------
        if errors:
            formatted = "\n".join(f"- {e}" for e in errors)
            raise ToolSchemaValidationError(
                f"Tool '{schema.name}' failed validation:\n{formatted}"
            )

In [4]:
# =============================================================================
# TOOL REGISTRY: Generalized
# =============================================================================

class ToolRegistry:
    """Domain-agnostic tool registry."""

    def __init__(self):
        self._tools: Dict[str, ToolSchema] = {}
        self._functions: Dict[str, Callable] = {}
        self._bm25_index = None
        self._bm25_tool_names = []

    def register(
        self,
        name: str,
        description: str,
        parameters: Dict[str, Any],
        required: List[str],
        example: Dict[str, Any],
        docstring: str,
        func: Callable,
        tool_type: ToolType = ToolType.KNOWLEDGE,
        arg_sources: Dict[str, str] = None,
        output_keys: Dict[str, str] = None,
        explicit_keywords: List[str] = None,
        arg_extractors: Dict[str, tuple] = None,
    ):
        schema = ToolSchema(
            name=name, description=description, parameters=parameters,
            required=required, example=example, docstring=docstring,
            tool_type=tool_type,
            arg_sources=arg_sources or {},
            output_keys=output_keys or {},
            explicit_keywords=explicit_keywords or [],
            arg_extractors=arg_extractors or {},
        )
        # Validate before storing
        ToolSchemaValidator.validate(schema)  # Uncomment - To void registry validation
        self._tools[name] = schema
        self._functions[name] = func
        self._rebuild_bm25_index()

    def _rebuild_bm25_index(self):
        if not self._tools:
            return
        self._bm25_tool_names = list(self._tools.keys())
        documents = [self._tools[name].get_bm25_document() for name in self._bm25_tool_names]
        tokenized_docs = [doc.lower().split() for doc in documents]
        self._bm25_index = BM25Okapi(tokenized_docs)

    def get_tool(self, name: str) -> Optional[ToolSchema]:
        return self._tools.get(name)

    def get_function(self, name: str) -> Optional[Callable]:
        return self._functions.get(name)

    async def call(self, name: str, arguments: Dict) -> Any:
        func = self._functions.get(name)
        if not func:
            raise ValueError(f"Unknown tool: {name}")
        if asyncio.iscoroutinefunction(func):
            return await func(**arguments)
        return func(**arguments)

    def get_all_medgemma_format(self, exclude: set = None) -> str:
        exclude = exclude or set()
        return "\n".join(s.get_medgemma_format() for n, s in self._tools.items() if n not in exclude)

    def get_all_names(self) -> List[str]:
        return list(self._tools.keys())

    def get_tools_by_type(self, tool_type: ToolType) -> List[ToolSchema]:
        return [t for t in self._tools.values() if t.tool_type == tool_type]

    def get_all_context_keys(self) -> List[str]:
        keys = set()
        for tool in self._tools.values():
            keys.update(tool.arg_sources.values())
            keys.update(tool.output_keys.keys())
        return sorted(keys)

    def get_all_arg_extractors(self) -> Dict[str, tuple]:
        """Collect all arg_extractors across tools, keyed by context_key."""
        extractors = {}
        for tool in self._tools.values():
            for param_name, extractor_spec in tool.arg_extractors.items():
                context_key = tool.arg_sources.get(param_name, param_name)
                if context_key not in extractors:
                    extractors[context_key] = extractor_spec
        return extractors


print("✅ ToolRegistry defined")

✅ ToolRegistry defined


### **SDK Core: LLM Components**

In [5]:
# 1) Install dependencies (Colab)
!pip -q install torch transformers==5.0.0 huggingface_hub rank_bm25
!pip install sentence-transformers -q

In [6]:
from huggingface_hub import notebook_login
notebook_login()
print("✅ Login cell ready")

✅ Login cell ready


In [7]:
import json
import re
import numpy as np
import asyncio
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import Dict, List, Optional, Any, Callable, Literal, Tuple
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod
from rank_bm25 import BM25Okapi

print("✅ Dependencies loaded")

✅ Dependencies loaded


In [8]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, AutoProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRANSLATE_MODEL_ID = "google/medgemma-1.5-4b-it"  # "google/translategemma-4b-it"

translate_pipe = pipeline(
    "image-text-to-text",
    model=TRANSLATE_MODEL_ID,
    device="cuda" if DEVICE == "cuda" else "cpu",
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
)

print(f"✅ Model loaded on {DEVICE}: {TRANSLATE_MODEL_ID}")
if DEVICE == "cuda":
    print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Model loaded on cuda: google/medgemma-1.5-4b-it
   VRAM: 8.6 GB


In [9]:
def parse_function_call(text):
    """Parse FunctionGemma output format: call:name{key1:value1, key2:value2}"""
    match = re.search(r"call:([a-zA-Z0-9_]+)\{(.*)\}", text, re.DOTALL)
    if not match:
        return None

    fn_name = match.group(1)
    raw_args = match.group(2).strip().replace("<escape>", "")

    if not raw_args:
        return {"name": fn_name, "arguments": {}}

    args = {}
    try:
        args = json.loads("{" + raw_args + "}")
        return {"name": fn_name, "arguments": args}
    except json.JSONDecodeError:
        pass

    depth = 0
    current = ""
    pairs = []
    for char in raw_args:
        if char in "{[":
            depth += 1
            current += char
        elif char in "}]":
            depth -= 1
            current += char
        elif char == "," and depth == 0:
            pairs.append(current.strip())
            current = ""
        else:
            current += char
    if current.strip():
        pairs.append(current.strip())

    for pair in pairs:
        if ":" not in pair:
            continue
        key, value = pair.split(":", 1)
        key = key.strip()
        value = value.strip()
        try:
            args[key] = json.loads(value)
        except json.JSONDecodeError:
            if (value.startswith('"') and value.endswith('"')) or \
               (value.startswith("'") and value.endswith("'")):
                value = value[1:-1]
            args[key] = value

    return {"name": fn_name, "arguments": args}


def extract_text(content):
    """Flatten and extract text from multimodal model outputs."""
    flat = []
    for item in content:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)
    return "\n".join(
        c["text"] for c in flat
        if isinstance(c, dict) and c.get("type") == "text"
    )


print("✅ Parser utilities loaded")

✅ Parser utilities loaded


In [10]:
# =============================================================================
# PIPELINE RUNNERS — Dual-mode TranslateGemma
# =============================================================================

def _reformat_messages_for_translate_pipe(messages):
    """
    TranslateGemma's processor REQUIRES source_lang_code + target_lang_code
    on every text content block. When the orchestrator/extractor sends plain
    reasoning prompts, those fields are missing → crash.

    This reformats all content blocks to include en→en as passthrough.
    """
    reformatted = []
    for msg in messages:
        new_msg = {"role": msg["role"]}
        content = msg.get("content")

        if isinstance(content, str):
            new_msg["content"] = [{
                "type": "text",
                "source_lang_code": "en",
                "target_lang_code": "en",
                "text": content,
            }]
        elif isinstance(content, list):
            new_content = []
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    new_content.append({
                        "type": "text",
                        "source_lang_code": block.get("source_lang_code", "en"),
                        "target_lang_code": block.get("target_lang_code", "en"),
                        "text": block.get("text", ""),
                    })
                elif isinstance(block, dict):
                    new_content.append(block)
                elif isinstance(block, str):
                    new_content.append({
                        "type": "text",
                        "source_lang_code": "en",
                        "target_lang_code": "en",
                        "text": block,
                    })
            new_msg["content"] = new_content
        else:
            new_msg["content"] = content

        reformatted.append(new_msg)
    return reformatted


def run_translate_gemma(
    messages=None,
    max_new_tokens=512,
    temperature=0.7,
    text=None,
    source_lang_code=None,
    target_lang_code=None,
    image_url=None,
) -> str:
    """
    Unified MedGemma/TranslateGemma runner — BOTH reasoning engine AND translator.

    Mode 1 (Reasoning): messages=[...] → en→en passthrough for planning/extraction
    Mode 2 (Translation): text=..., source_lang_code=..., target_lang_code=...
    """

    # MODE 1: Reasoning
    if messages is not None:
        reformatted = _reformat_messages_for_translate_pipe(messages)
        try:
            out = translate_pipe(
                text=reformatted,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0),
                temperature=temperature if temperature > 0 else None,
            )
            generated_content = out[0]["generated_text"][-1]["content"]
            if isinstance(generated_content, list):
                return extract_text(generated_content)
            return generated_content
        except Exception as e:
            print(f"[TranslateGemma Mode 1] Error: {e}")
            for msg in messages:
                content = msg.get("content", "")
                if isinstance(content, list):
                    for block in content:
                        if isinstance(block, dict) and "text" in block:
                            return block["text"]
                elif isinstance(content, str):
                    return content
            return ""

    # MODE 2: Translation
    if text is not None or image_url is not None:
        if image_url:
            content = [{
                "type": "image",
                "source_lang_code": source_lang_code or "auto",
                "target_lang_code": target_lang_code or "en",
                "url": image_url,
            }]
        else:
            content = [{
                "type": "text",
                "source_lang_code": source_lang_code or "en",
                "target_lang_code": target_lang_code or "en",
                "text": text,
            }]

        translation_messages = [{"role": "user", "content": content}]
        output = translate_pipe(text=translation_messages, max_new_tokens=max_new_tokens)
        result = output[0]["generated_text"][-1]["content"]
        if isinstance(result, list):
            return extract_text(result)
        return result

    return "Error: provide either messages= (reasoning mode) or text= (translation mode)"


# Compatibility alias for medical agents
run_medgemma = run_translate_gemma

print("✅ run_translate_gemma defined (dual-mode)")
print("   Mode 1: messages=[...] → en→en passthrough for reasoning")
print("   Mode 2: text=..., src=..., tgt=... → real translation")
print("   Alias:  run_medgemma = run_translate_gemma")

✅ run_translate_gemma defined (dual-mode)
   Mode 1: messages=[...] → en→en passthrough for reasoning
   Mode 2: text=..., src=..., tgt=... → real translation
   Alias:  run_medgemma = run_translate_gemma


### **SDK Core: Entity Extraction**

In [11]:
class BaseValueExtractor(ABC):
    @abstractmethod
    def extract_from_goal(self, goal: str, known_keys: List[str] = None) -> Dict[str, Any]:
        pass
    @abstractmethod
    def extract_from_result(self, tool_name: str, result: str, output_keys: Dict[str, str] = None) -> Dict[str, Any]:
        pass

print("✅ Strategy enums and base classes defined")

✅ Strategy enums and base classes defined


In [12]:
# =============================================================================
# LLM VALUE EXTRACTOR — Fully Generic
# =============================================================================

class LLMValueExtractor(BaseValueExtractor):
    """
    Pure LLM-based value extraction.
    NO hardcoded tool branches. Uses tool-declared output_keys to build prompts.
    """

    def __init__(self, llm_caller: Callable, verbose: bool = True):
        self.llm = llm_caller
        self.verbose = verbose

    def _log(self, msg):
        if self.verbose:
            print(f"[LLMExtractor] {msg}")

    def _call_llm(self, prompt: str, max_tokens: int = 512) -> str:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        response = self.llm(messages=messages, max_new_tokens=max_tokens, temperature=0.0)
        cleaned = response
        for start_tag, end_tag in [("<unused94>", "<unused95>"), ("<think>", "</think>")]:
            while start_tag in cleaned:
                start_idx = cleaned.find(start_tag)
                end_idx = cleaned.find(end_tag)
                if end_idx > start_idx:
                    cleaned = cleaned[:start_idx] + cleaned[end_idx + len(end_tag):]
                else:
                    cleaned = cleaned[:start_idx]
        return cleaned.strip()

    def _parse_json_response(self, response: str) -> Dict[str, Any]:
        start = response.find('{')
        end = response.rfind('}')
        if start != -1 and end != -1 and end > start:
            try:
                parsed = json.loads(response[start:end+1])
                return {k: v for k, v in parsed.items() if v is not None and v != "null"}
            except json.JSONDecodeError:
                self._log(f"JSON parse failed: {response[start:end+1][:100]}")
        return {}

    def extract_from_goal(self, goal: str, known_keys: List[str] = None) -> Dict[str, Any]:
        known_keys = known_keys or []
        self._log(f"Extracting from goal: {goal[:80]}...")

        if not known_keys:
            prompt = f"""You are a data extractor. Extract ALL explicit values from this text.
Output as a flat JSON object. Use null for values not explicitly stated.

TEXT: {goal}

JSON:"""
        else:
            keys_list = "\n".join(f"- {key}" for key in known_keys)
            prompt = f"""You are a data extractor. Extract ONLY values that are EXPLICITLY written in the text.

TEXT: {goal}

Extract these fields (use null if not explicitly stated):
{keys_list}

IMPORTANT: Only extract values that are EXPLICITLY written. Do not infer or guess.
Output as JSON:"""

        response = self._call_llm(prompt)
        values = self._parse_json_response(response)
        self._log(f"Extracted from goal: {values}")
        return values

    def extract_from_result(self, tool_name: str, result: str, output_keys: Dict[str, str] = None) -> Dict[str, Any]:
        output_keys = output_keys or {}
        self._log(f"Extracting from {tool_name}: {result[:80]}...")

        if not output_keys:
            prompt = f"""Extract all relevant values from this tool result.

TOOL: {tool_name}
RESULT: {result}

Output as a flat JSON object with descriptive keys.
JSON:"""
        else:
            keys_spec = "\n".join(f"- {key} ({type_hint})" for key, type_hint in output_keys.items())
            prompt = f"""Extract these specific values from the tool result.

TOOL: {tool_name}
RESULT: {result}

Values to extract (use null if not found):
{keys_spec}

Output as JSON:"""

        response = self._call_llm(prompt)
        values = self._parse_json_response(response)
        values[f"_raw_{tool_name}"] = result
        self._log(f"Extracted from {tool_name}: {list(values.keys())}")
        return values


print("✅ LLMValueExtractor defined (fully generic)")

✅ LLMValueExtractor defined (fully generic)


### **SDK Core: Dynamic Context Manager**

In [13]:
# =============================================================================
# CONTEXT BANK — Fully Generic, Schema-Driven
# =============================================================================

class ContextBank:
    """
    Domain-agnostic context bank. ZERO domain knowledge.

    Extraction is driven entirely by:
      1. Enum values parsed from tool parameter descriptions (automatic)
      2. arg_extractors declared by tool authors at registration time
      3. LLM extraction as fallback
    """

    def __init__(self, extractor: BaseValueExtractor, known_keys: List[str] = None, registry=None):
        self.extractor = extractor
        self.known_keys = known_keys or []
        self.registry = registry
        self.user_goal: str = ""
        self.retrieved_data: Dict[str, str] = {}
        self.parsed_values: Dict[str, Any] = {}

    def set_goal(self, goal: str):
        """Set user goal and extract values using schema + LLM."""
        self.user_goal = goal

        # Step 1: Schema-driven extraction (no LLM, reads tool declarations)
        if self.registry:
            schema_values = self._extract_from_schema(goal)
            self.parsed_values.update(schema_values)

        # Step 2: LLM extraction (catches anything schema missed)
        goal_values = self.extractor.extract_from_goal(goal, self.known_keys)
        for k, v in goal_values.items():
            if k not in self.parsed_values:
                self.parsed_values[k] = v

    def _extract_from_schema(self, goal: str) -> Dict[str, Any]:
        """
        Extract values using ONLY tool-declared information.
        No hardcoded domain knowledge whatsoever.
        """
        values = {}
        goal_lower = goal.lower()

        # =============================================================
        # 1. ENUM extraction (automatic from pipe-separated param descs)
        # =============================================================
        for tool in self.registry._tools.values():
            for param_name, param_desc in tool.parameters.items():
                desc_str = str(param_desc)
                for part in desc_str.split():
                    if "|" in part:
                        clean = part.strip(".,;:()[]")
                        enum_values = [v.strip() for v in clean.split("|") if v.strip()]
                        for val in enum_values:
                            if val.lower() in goal_lower:
                                context_key = tool.arg_sources.get(param_name, param_name)
                                if context_key not in values:
                                    values[context_key] = val.lower()
                                break

        # =============================================================
        # 2. Tool-declared arg_extractors
        # =============================================================
        all_extractors = self.registry.get_all_arg_extractors()

        for context_key, extractor_spec in all_extractors.items():
            if context_key in values:
                continue

            ext_type, ext_config = extractor_spec

            # --- QUOTED: extract quoted strings ---
            if ext_type == ArgExtractorType.QUOTED:
                for quote_char in ["'", '"']:
                    parts = goal.split(quote_char)
                    if len(parts) >= 3 and len(parts[1]) > 2:
                        values[context_key] = parts[1]
                        break

            # --- LANGUAGE: map language names → codes ---
            elif ext_type == ArgExtractorType.LANGUAGE:
                mapping = ext_config.get("mapping", {})
                for lang_name, lang_code in mapping.items():
                    if lang_name.lower() in goal_lower:
                        values[context_key] = lang_code
                        break

            # --- NUMERIC_ID: extract ID following specific keywords ---
            elif ext_type == ArgExtractorType.NUMERIC_ID:
                preceding_words = ext_config.get("preceding_words", [])
                words = goal.split()
                for i, word in enumerate(words):
                    clean_word = word.rstrip(".,;:!?")
                    if clean_word.isdigit() and len(clean_word) >= 2 and i > 0:
                        prev = words[i-1].lower().rstrip(":#.,;")
                        if prev in preceding_words:
                            values[context_key] = clean_word
                            break

            # --- NUMBER: extract bare numeric values with units ---
            elif ext_type == ArgExtractorType.NUMBER:
                units = ext_config.get("units", [])
                for unit in units:
                    match = re.search(rf'(\d+\.?\d*)\s*{re.escape(unit)}', goal_lower)
                    if match:
                        values[context_key] = float(match.group(1))
                        break

        return values

    def add_tool_result(self, tool_name: str, result: str, output_keys: Dict[str, str] = None):
        self.retrieved_data[tool_name] = result
        result_values = self.extractor.extract_from_result(tool_name, result, output_keys or {})
        self.parsed_values.update(result_values)

    def has_value(self, key: str) -> bool:
        return key in self.parsed_values

    def get_value(self, key: str, default=None):
        return self.parsed_values.get(key, default)

    def get_grounding_context(self) -> str:
        lines = [f"USER GOAL: {self.user_goal}"]
        if self.parsed_values:
            lines.append("\nEXTRACTED VALUES:")
            for key, value in self.parsed_values.items():
                val_str = str(value)
                if len(val_str) > 120:
                    val_str = val_str[:120] + "..."
                lines.append(f"  {key}: {val_str}")
        if self.retrieved_data:
            lines.append("\nTOOL RESULTS:")
            for tool, result in self.retrieved_data.items():
                lines.append(f"  [{tool}]: {result[:150]}...")
        return "\n".join(lines)

    def reset(self):
        self.user_goal = ""
        self.retrieved_data = {}
        self.parsed_values = {}


print("✅ ContextBank defined (zero domain knowledge)")

✅ ContextBank defined (zero domain knowledge)


## **Medical DeepResearch Agent**

To further evaluate the generalizability of our agent harness, we build a Medical DeepResearch Agent specially designed for long-horizon writing tasks (upto 80-100 pages) using a 4B Model running on a single T4 GPU.

### **Integrated Fixes Summary**

This version of the DeepResearch SDK includes all fixes from the integration report:

| Fix # | Issue | Solution |
|-------|-------|----------|
| 1 | Greedy Decoding (Temperature=0) | `GenerationConfig` with task-specific presets (writing=0.8, planning=0.3) |
| 2 | Factual Scarcity | `FactFilter` with `allow_unverified=True` and confidence threshold |
| 3 | Poor Writing Style | PhD-level system prompt with cohesive devices and sentence variety |
| 4 | No Plagiarism Check | `OverlapChecker` with lexical (Jaccard) + semantic similarity |
| 5 | Context-Free Images | `ContextualImageGenerator` with full content analysis |
| 6 | Template Leakage | `OutputCleaner` with 30+ regex patterns |

**Additional Enhancements:**
- LLM-generated abstract (200-300 words) instead of deterministic patterns
- Page breaks between Abstract/Keywords and Table of Contents
- `allow_image_generation` toggle parameter in orchestrator
- Removed duplicate image labels from output

**Usage:**
```python
# Run with images
result = await run_deep_research(topic_index=0, target_pages=20, allow_image_generation=True)

# Run without images
result = await run_deep_research(topic_index=0, target_pages=20, allow_image_generation=False)
```

In [14]:
# API keys
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY') #TAVILY_API_KEY

##### **Cell 1: Core Infrastructure & Shared State**

In [15]:
# =============================================================================
# DEEP RESEARCH SDK - CORE INFRASTRUCTURE
# =============================================================================
#
# A multi-agent deep research system built on top of the Clinical Agent SDK.
# Capable of producing 80-100 page research documents with citations.
#
# Architecture:
# - Main Orchestrator coordinates sub-agents
# - Each sub-agent is itself an agent instance with specialized tools
# - Shared persistent memory across all agents
# - Incremental file output for fault tolerance
#
# =============================================================================

import os
import json
import hashlib
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Callable
from enum import Enum
from pathlib import Path

"""
# API placeholders
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "your-tavily-api-key")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "your-gemini-api-key")
"""
# =============================================================================
# RESEARCH STATE MODELS
# =============================================================================

class ResearchStatus(Enum):
    NOT_STARTED = "not_started"
    PLANNING = "planning"
    RESEARCHING = "researching"
    WRITING = "writing"
    VERIFYING = "verifying"
    COMPLETE = "complete"
    NEEDS_REVIEW = "needs_review"


@dataclass
class Citation:
    """A single citation/reference."""
    id: int
    title: str
    url: str
    authors: str
    date: str
    accessed_date: str
    snippet: str
    reliability_score: float = 0.0

    def to_vancouver(self) -> str:
        """Format as Vancouver style citation."""
        author_part = f"{self.authors}. " if self.authors and self.authors.strip() else ""
        return f"[{self.id}] {author_part}{self.title}. Available from: {self.url}. Accessed: {self.accessed_date}."
        #return f"[{self.id}] {self.authors}. {self.title}. Available from: {self.url}. Accessed: {self.accessed_date}."


@dataclass
class ResearchFact:
    """A verified fact with citation."""
    fact_id: str
    content: str
    citation_ids: List[int]
    confidence: float
    verified: bool = False
    section: str = ""


@dataclass
class SectionOutline:
    """Outline for a single section/chapter."""
    section_id: str
    title: str
    description: str
    subsections: List[str]
    target_word_count: int
    status: str = "pending"
    file_path: str = ""
    word_count: int = 0


@dataclass
class ResearchOutline:
    """Complete research document outline."""
    topic: str
    thesis: str
    sections: List[SectionOutline]
    total_target_words: int
    created_at: str = ""

    def __post_init__(self):
        if not self.created_at:
            self.created_at = datetime.now().isoformat()


@dataclass
class ResearchState:
    """Complete persistent state for a research project."""
    project_id: str
    topic: str
    status: ResearchStatus
    outline: Optional[ResearchOutline] = None
    citations: List[Citation] = field(default_factory=list)
    facts: List[ResearchFact] = field(default_factory=list)
    completed_sections: List[str] = field(default_factory=list)
    current_section: str = ""
    search_history: List[Dict] = field(default_factory=list)
    error_log: List[str] = field(default_factory=list)
    human_review_notes: List[Dict] = field(default_factory=list)
    created_at: str = ""
    updated_at: str = ""
    output_dir: str = ""

    def __post_init__(self):
        now = datetime.now().isoformat()
        if not self.created_at:
            self.created_at = now
        self.updated_at = now


# =============================================================================
# PERSISTENT MEMORY MANAGER
# =============================================================================

class PersistentMemory:
    """
    Manages persistent state across all agents.
    Saves state to disk after each operation for fault tolerance.
    """

    def __init__(self, base_dir: str = "./research_projects"):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.current_state: Optional[ResearchState] = None
        self._citation_counter = 0

    def create_project(self, topic: str) -> ResearchState:
        """Create a new research project."""
        # Generate unique project ID
        project_id = hashlib.md5(f"{topic}_{datetime.now().isoformat()}".encode()).hexdigest()[:12]

        # Create project directory
        project_dir = self.base_dir / project_id
        project_dir.mkdir(parents=True, exist_ok=True)
        (project_dir / "sections").mkdir(exist_ok=True)
        (project_dir / "images").mkdir(exist_ok=True)

        # Initialize state
        self.current_state = ResearchState(
            project_id=project_id,
            topic=topic,
            status=ResearchStatus.NOT_STARTED,
            output_dir=str(project_dir)
        )

        self._save_state()
        print(f"✅ Created project: {project_id}")
        print(f"   Output directory: {project_dir}")

        return self.current_state

    def load_project(self, project_id: str) -> Optional[ResearchState]:
        """Load existing project from disk."""
        state_file = self.base_dir / project_id / "state.json"

        if not state_file.exists():
            print(f"❌ Project not found: {project_id}")
            return None

        with open(state_file, 'r') as f:
            data = json.load(f)

        # Reconstruct state
        self.current_state = self._dict_to_state(data)
        self._citation_counter = len(self.current_state.citations)

        print(f"✅ Loaded project: {project_id}")
        print(f"   Status: {self.current_state.status.value}")
        print(f"   Completed sections: {len(self.current_state.completed_sections)}")

        return self.current_state

    def _save_state(self):
        """Save current state to disk."""
        if not self.current_state:
            return

        self.current_state.updated_at = datetime.now().isoformat()

        state_file = Path(self.current_state.output_dir) / "state.json"

        with open(state_file, 'w') as f:
            json.dump(self._state_to_dict(self.current_state), f, indent=2)

    def _state_to_dict(self, state: ResearchState) -> Dict:
        """Convert state to JSON-serializable dict."""
        data = {
            "project_id": state.project_id,
            "topic": state.topic,
            "status": state.status.value,
            "created_at": state.created_at,
            "updated_at": state.updated_at,
            "output_dir": state.output_dir,
            "current_section": state.current_section,
            "completed_sections": state.completed_sections,
            "search_history": state.search_history,
            "error_log": state.error_log,
            "human_review_notes": state.human_review_notes,
            "citations": [asdict(c) for c in state.citations],
            "facts": [asdict(f) for f in state.facts],
        }

        if state.outline:
            data["outline"] = {
                "topic": state.outline.topic,
                "thesis": state.outline.thesis,
                "total_target_words": state.outline.total_target_words,
                "created_at": state.outline.created_at,
                "sections": [asdict(s) for s in state.outline.sections]
            }

        return data

    def _dict_to_state(self, data: Dict) -> ResearchState:
        """Reconstruct state from dict."""
        outline = None
        if "outline" in data and data["outline"]:
            outline_data = data["outline"]
            sections = [SectionOutline(**s) for s in outline_data.get("sections", [])]
            outline = ResearchOutline(
                topic=outline_data["topic"],
                thesis=outline_data["thesis"],
                sections=sections,
                total_target_words=outline_data["total_target_words"],
                created_at=outline_data.get("created_at", "")
            )

        return ResearchState(
            project_id=data["project_id"],
            topic=data["topic"],
            status=ResearchStatus(data["status"]),
            outline=outline,
            citations=[Citation(**c) for c in data.get("citations", [])],
            facts=[ResearchFact(**f) for f in data.get("facts", [])],
            completed_sections=data.get("completed_sections", []),
            current_section=data.get("current_section", ""),
            search_history=data.get("search_history", []),
            error_log=data.get("error_log", []),
            human_review_notes=data.get("human_review_notes", []),
            created_at=data.get("created_at", ""),
            updated_at=data.get("updated_at", ""),
            output_dir=data["output_dir"]
        )

    # =========================================================================
    # STATE MUTATION METHODS
    # =========================================================================

    def set_status(self, status: ResearchStatus):
        """Update project status."""
        if self.current_state:
            self.current_state.status = status
            self._save_state()

    def set_outline(self, outline: ResearchOutline):
        """Set the research outline."""
        if self.current_state:
            self.current_state.outline = outline
            self._save_state()

    def add_citation(self, title: str, url: str, authors: str, date: str, snippet: str, reliability_score: float = 0.0) -> Citation:
        """Add a new citation and return it."""
        self._citation_counter += 1

        citation = Citation(
            id=self._citation_counter,
            title=title,
            url=url,
            authors=authors,
            date=date,
            accessed_date=datetime.now().strftime("%Y-%m-%d"),
            snippet=snippet,
            reliability_score=reliability_score
        )

        if self.current_state:
            self.current_state.citations.append(citation)
            self._save_state()

        return citation

    def add_fact(self, content: str, citation_ids: List[int], confidence: float, section: str = "") -> ResearchFact:
        """Add a verified fact."""
        fact_id = hashlib.md5(content.encode()).hexdigest()[:8]

        fact = ResearchFact(
            fact_id=fact_id,
            content=content,
            citation_ids=citation_ids,
            confidence=confidence,
            section=section
        )

        if self.current_state:
            self.current_state.facts.append(fact)
            self._save_state()

        return fact

    def mark_section_complete(self, section_id: str, file_path: str, word_count: int):
        """Mark a section as complete."""
        if self.current_state:
            self.current_state.completed_sections.append(section_id)

            # Update section in outline
            if self.current_state.outline:
                for section in self.current_state.outline.sections:
                    if section.section_id == section_id:
                        section.status = "complete"
                        section.file_path = file_path
                        section.word_count = word_count
                        break

            self._save_state()

    def add_search_record(self, query: str, results_count: int, section: str):
        """Log a search query."""
        if self.current_state:
            self.current_state.search_history.append({
                "query": query,
                "results_count": results_count,
                "section": section,
                "timestamp": datetime.now().isoformat()
            })
            self._save_state()

    def add_human_review_note(self, section: str, note: str, action: str = "review"):
        """Add a human review note."""
        if self.current_state:
            self.current_state.human_review_notes.append({
                "section": section,
                "note": note,
                "action": action,
                "timestamp": datetime.now().isoformat(),
                "resolved": False
            })
            self._save_state()

    def log_error(self, error: str):
        """Log an error."""
        if self.current_state:
            self.current_state.error_log.append(f"[{datetime.now().isoformat()}] {error}")
            self._save_state()

    # =========================================================================
    # QUERY METHODS
    # =========================================================================

    def get_citation_by_id(self, citation_id: int) -> Optional[Citation]:
        """Get a citation by ID."""
        if self.current_state:
            for c in self.current_state.citations:
                if c.id == citation_id:
                    return c
        return None

    def get_facts_for_section(self, section_id: str) -> List[ResearchFact]:
        """Get all facts for a section."""
        if self.current_state:
            return [f for f in self.current_state.facts if f.section == section_id]
        return []

    def get_pending_sections(self) -> List[SectionOutline]:
        """Get sections that haven't been completed."""
        if self.current_state and self.current_state.outline:
            return [s for s in self.current_state.outline.sections if s.status != "complete"]
        return []

    def get_all_citations_formatted(self) -> str:
        """Get all citations formatted as references section."""
        if not self.current_state:
            return ""

        lines = ["# References\n"]
        for citation in sorted(self.current_state.citations, key=lambda c: c.id):
            lines.append(citation.to_vancouver())
            lines.append("")

        return "\n".join(lines)


# Create global memory instance
memory = PersistentMemory()

print("✅ Core infrastructure defined")
print("   - ResearchState with full persistence")
print("   - Citation management (Vancouver style)")
print("   - Fact tracking with verification")
print("   - Section progress tracking")


# =============================================================================
# FIX #1: GENERATION CONFIG FOR TEMPERATURE SCALING
# =============================================================================

@dataclass
class GenerationConfig:
    """
    Configuration for MedGemma text generation.
    Different tasks benefit from different temperature settings.
    """
    temperature: float = 0.7          # Default: balanced creativity
    top_p: float = 0.9                # Nucleus sampling
    top_k: int = 50                   # Top-k sampling
    max_new_tokens: int = 4096       # Maximum output length
    repetition_penalty: float = 1.1   # Penalize repetition
    do_sample: bool = True            # Enable sampling (required for temp > 0)

    @classmethod
    def for_planning(cls) -> 'GenerationConfig':
        """Lower temperature for structured planning output."""
        return cls(temperature=0.3, top_p=0.85, repetition_penalty=1.05)

    @classmethod
    def for_writing(cls) -> 'GenerationConfig':
        """Higher temperature for creative, diverse writing."""
        return cls(temperature=0.8, top_p=0.92, repetition_penalty=1.15)

    @classmethod
    def for_factual(cls) -> 'GenerationConfig':
        """Lower temperature for fact extraction (accuracy over creativity)."""
        return cls(temperature=0.2, top_p=0.8, repetition_penalty=1.0)

    @classmethod
    def for_verification(cls) -> 'GenerationConfig':
        """Very low temperature for verification tasks."""
        return cls(temperature=0.1, top_p=0.75, do_sample=False)


# =============================================================================
# FIX #2: FACT FILTER CONFIG FOR ALLOWING UNVERIFIED FACTS
# =============================================================================

@dataclass
class FactFilterConfig:
    """Configuration for fact filtering in research writing."""
    allow_unverified: bool = True           # Include unverified facts
    min_confidence: float = 0.2             # Minimum confidence threshold
    prefer_verified: bool = True            # Sort verified facts first
    max_unverified_ratio: float = 0.75       # Max 75% unverified facts
    require_citation: bool = False # True   # Facts must not have citation_ids


class FactFilter:
    """
    Filters and prioritizes facts for inclusion in research writing.
    """

    def __init__(self, config: FactFilterConfig = None):
        self.config = config or FactFilterConfig()

    def filter_facts(
        self,
        facts: List[Dict],
        section_id: str = None
    ) -> List[Dict]:
        """Filter facts based on configuration."""
        filtered = []

        for fact in facts:
            # Filter by section if specified
            if section_id and fact.get('section') != section_id:
                continue

            # Check confidence threshold
            confidence = fact.get('confidence', 0)
            if confidence < self.config.min_confidence:
                continue

            # Check citation requirement
            if self.config.require_citation:
                citation_ids = fact.get('citation_ids', [])
                if not citation_ids:
                    continue

            # Check verified status
            is_verified = fact.get('verified', False)
            if not is_verified and not self.config.allow_unverified:
                continue

            filtered.append(fact)

        # Sort: verified first if preferred
        if self.config.prefer_verified:
            filtered.sort(key=lambda f: (
                0 if f.get('verified') else 1,
                -f.get('confidence', 0)
            ))

        # Enforce max unverified ratio
        if self.config.allow_unverified and self.config.max_unverified_ratio < 1.0:
            verified = [f for f in filtered if f.get('verified')]
            unverified = [f for f in filtered if not f.get('verified')]
            total_target = len(verified) + len(unverified)
            max_unverified = int(total_target * self.config.max_unverified_ratio)
            unverified = unverified[:max_unverified]
            filtered = verified + unverified

        return filtered

    def get_facts_for_section(
        self,
        all_facts: List[Dict],
        section_id: str,
        max_facts: int = 20
    ) -> List[Dict]:
        """Get filtered facts for a specific section."""
        section_facts = self.filter_facts(all_facts, section_id)
        return section_facts[:max_facts]


# =============================================================================
# FIX #4: OVERLAP CHECKER FOR PLAGIARISM DETECTION
# =============================================================================

class OverlapChecker:
    """
    Checks for textual overlap and similarity between text sections.
    Uses both lexical (word overlap) and semantic (embedding) approaches.
    """

    def __init__(
        self,
        word_overlap_threshold: float = 0.4,
        semantic_threshold: float = 0.85,
        use_embeddings: bool = True
    ):
        self.word_overlap_threshold = word_overlap_threshold
        self.semantic_threshold = semantic_threshold
        self.use_embeddings = use_embeddings
        self.embedder = None

        if use_embeddings:
            try:
                from sentence_transformers import SentenceTransformer
                self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
                print("[OverlapChecker] ✓ Sentence embeddings enabled")
            except ImportError:
                print("[OverlapChecker] ⚠️ sentence-transformers not available, using lexical only")

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization for word overlap."""
        import re
        words = re.findall(r'\b[a-z]{3,}\b', text.lower())
        stopwords = {
            'the', 'and', 'for', 'are', 'but', 'not', 'you', 'all', 'can',
            'her', 'was', 'one', 'our', 'out', 'has', 'have', 'been', 'were',
            'being', 'there', 'their', 'this', 'that', 'these', 'those', 'with',
            'from', 'which', 'would', 'could', 'should', 'into', 'more', 'some',
            'such', 'than', 'then', 'them', 'also', 'only', 'other', 'very'
        }
        return [w for w in words if w not in stopwords]

    def word_overlap_score(self, text1: str, text2: str) -> float:
        """Calculate Jaccard similarity based on word overlap."""
        words1 = set(self._tokenize(text1))
        words2 = set(self._tokenize(text2))
        if not words1 or not words2:
            return 0.0
        intersection = words1 & words2
        union = words1 | words2
        return len(intersection) / len(union)

    def semantic_similarity(self, text1: str, text2: str) -> float:
        """Calculate semantic similarity using sentence embeddings."""
        if self.embedder is None:
            return 0.0
        embeddings = self.embedder.encode([text1, text2])
        similarity = np.dot(embeddings[0], embeddings[1]) / (
            np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
        )
        return float(similarity)

    def check_overlap(self, text1: str, text2: str, label1: str = "A", label2: str = "B") -> Dict:
        """Check overlap between two text sections."""
        word_score = self.word_overlap_score(text1, text2)
        semantic_score = self.semantic_similarity(text1, text2) if self.embedder else 0.0

        needs_rewrite = False
        reasons = []

        if word_score >= self.word_overlap_threshold:
            needs_rewrite = True
            reasons.append(f"Word overlap: {word_score:.0%}")

        if semantic_score >= self.semantic_threshold:
            needs_rewrite = True
            reasons.append(f"Semantic similarity: {semantic_score:.0%}")

        recommendation = "No significant overlap." if not needs_rewrite else "Rewrite with different vocabulary."

        return {
            'text1_label': label1,
            'text2_label': label2,
            'word_overlap': word_score,
            'semantic_similarity': semantic_score,
            'needs_rewrite': needs_rewrite,
            'reasons': reasons,
            'recommendation': recommendation
        }

    def check_all_subsections(self, subsections: Dict[str, str]) -> List[Dict]:
        """Check all pairwise combinations of subsections."""
        issues = []
        titles = list(subsections.keys())
        for i in range(len(titles)):
            for j in range(i + 1, len(titles)):
                result = self.check_overlap(
                    subsections[titles[i]], subsections[titles[j]],
                    titles[i], titles[j]
                )
                if result['needs_rewrite']:
                    issues.append(result)
        return issues

    def extract_subsections(self, content: str) -> Dict[str, str]:
        """Extract subsections from markdown content."""
        import re
        subsections = {}
        parts = re.split(r'\n### ', content)
        for part in parts[1:]:
            lines = part.split('\n', 1)
            title = lines[0].strip()
            body = lines[1].strip() if len(lines) > 1 else ""
            if title and body:
                subsections[title] = body
        return subsections


# =============================================================================
# FIX #6: OUTPUT CLEANER FOR TEMPLATE LEAK PREVENTION
# =============================================================================
import re
from typing import List


class OutputCleaner:
    """
    Cleans model output to remove prompt template leakage and artifacts.
    """

    # ALWAYS applied - obvious template artifacts
    SAFE_PATTERNS = [
        r'\[Opening paragraph:.*?\]',
        r'\[Topic sentence.*?\]',
        r'\[Continue with.*?\]',
        r'\[Closing paragraph:.*?\]',
        r'\[Insert.*?\]',
        r'\[Add.*?\]',
        r'\[Note:.*?\]',
        r'\[TODO:.*?\]',
        r'\[citation needed\]',
    ]

    # Only applied when aggressive=True - might catch real content
    AGGRESSIVE_PATTERNS = [
        r'\[First Subsection\]',
        r'\[Second Subsection\]',
        r'\[.*?Subsection\]',
        r'\[Write.*?\]',
        r'\[Describe.*?\]',
        r'\[Explain.*?\]',
        r'\[Discuss.*?\]',
        r'^## WRITING REQUIREMENTS.*?(?=^## |\Z)',
        r'^## STRUCTURE TEMPLATE.*?(?=^## |\Z)',
        r'^## SECTION DETAILS.*?(?=^## |\Z)',
        r'^## AVAILABLE CITATIONS.*?(?=^## |\Z)',
        r'^## RESEARCH FACTS.*?(?=^## |\Z)',
        r'^Write the complete section now[\.:].*$',
        r'^Write exactly \d+ words[\.:].*$',
        r'^Target Length:.*$',
        r'^Focus on lexical diversity.*$',
        r'^Subsections to Cover:.*$',
        r'\[specific finding with citation\]',
        r'\[additional supporting point\]',
        r'\[contrasting perspective[^\]]*\]',
        r'\[synthesis of the evidence\]',
        r'\[evidence\]',
        r'\[X\](?!\d)',
    ]

    # Only applied when aggressive=True
    INSTRUCTION_PHRASES = [
        "as per the requirements",
        "following the guidelines",
        "as instructed",
        "according to the prompt",
        "based on the template",
        "as requested above",
        "per the writing requirements",
        "the section should",
        "this section will",
        "we will discuss",
        "we will explore",
        "we will examine",
        "in this section, we",
        "this subsection covers",
    ]

    def __init__(self, verbose: bool = True, aggressive: bool = False):
        self.verbose = verbose
        self.aggressive = aggressive

        # Always compile safe patterns
        self.safe_compiled = [
            re.compile(p, re.MULTILINE | re.IGNORECASE | re.DOTALL)
            for p in self.SAFE_PATTERNS
        ]

        # Compile aggressive patterns only if needed
        if aggressive:
            self.aggressive_compiled = [
                re.compile(p, re.MULTILINE | re.IGNORECASE | re.DOTALL)
                for p in self.AGGRESSIVE_PATTERNS
            ]
        else:
            self.aggressive_compiled = []

    def _log(self, msg: str):
        if self.verbose:
            print(f"[OutputCleaner] {msg}")

    def clean(self, content: str, section_title: str = None) -> str:
        """Clean model output of template leakage and artifacts."""
        original_length = len(content)

        # Apply safe patterns (always)
        for pattern in self.safe_compiled:
            content = pattern.sub('', content)

        # Apply aggressive patterns (only if enabled)
        if self.aggressive:
            for pattern in self.aggressive_compiled:
                content = pattern.sub('', content)

            # Remove instruction phrases (only in aggressive mode)
            for phrase in self.INSTRUCTION_PHRASES:
                content = re.sub(re.escape(phrase), '', content, flags=re.IGNORECASE)

        content = self._clean_whitespace(content)

        if section_title and not content.strip().startswith('## '):
            content = f"## {section_title}\n\n{content}"

        content = self._fix_headings(content)

        cleaned_length = len(content)
        if original_length - cleaned_length > 50:
            self._log(f"Removed {original_length - cleaned_length} chars of template artifacts")

        return content

    def _clean_whitespace(self, content: str) -> str:
        """Clean up whitespace issues."""
        while '\n\n\n' in content:
            content = content.replace('\n\n\n', '\n\n')
        lines = [line.rstrip() for line in content.split('\n')]
        content = '\n'.join(lines)
        return content.strip()

    def _fix_headings(self, content: str) -> str:
        """Ensure proper heading hierarchy."""
        while '####' in content:
            content = content.replace('####', '###')
        return content

    def detect_leakage(self, content: str) -> List[str]:
        """Detect potential template leakage in content."""
        issues = []
        brackets = re.findall(r'\[[^\]]{10,}\]', content)
        for b in brackets:
            if not re.match(r'\[\d+\]', b):
                issues.append(f"Bracketed instruction: {b[:50]}...")

        content_lower = content.lower()
        for phrase in self.INSTRUCTION_PHRASES:
            if phrase.lower() in content_lower:
                issues.append(f"Instruction phrase: '{phrase}'")

        return issues

print("✅ Fix classes added: GenerationConfig, FactFilterConfig, FactFilter, OverlapChecker, OutputCleaner")


✅ Core infrastructure defined
   - ResearchState with full persistence
   - Citation management (Vancouver style)
   - Fact tracking with verification
   - Section progress tracking
✅ Fix classes added: GenerationConfig, FactFilterConfig, FactFilter, OverlapChecker, OutputCleaner


##### **Cell 2: Tavily Web Search Tool**

In [16]:
# =============================================================================
# TAVILY SEARCH TOOL - FIXED WITH QUERY SANITIZATION
# =============================================================================

class TavilySearchTool:
    """
    Web search using Tavily API.
    Includes query sanitization to prevent 400 errors.
    """

    # Maximum query length Tavily accepts reliably
    MAX_QUERY_LENGTH = 200

    def __init__(self, api_key: str = TAVILY_API_KEY, verbose: bool = True):
        self.api_key = api_key
        self.base_url = "https://api.tavily.com"
        self.verbose = verbose

    def _log(self, msg):
        if self.verbose:
            print(f"[Tavily] {msg}")

    def _sanitize_query(self, query: str) -> str:
        """
        Sanitize query for Tavily API.
        - Truncate to max length
        - Remove special characters that cause issues
        - Keep it search-engine friendly
        """
        # Remove problematic characters
        sanitized = query.replace('"', '').replace("'", "")
        sanitized = sanitized.replace('\n', ' ').replace('\r', ' ')

        # Remove excessive whitespace
        sanitized = ' '.join(sanitized.split())

        # Truncate to max length (at word boundary)
        if len(sanitized) > self.MAX_QUERY_LENGTH:
            truncated = sanitized[:self.MAX_QUERY_LENGTH]
            # Cut at last space to avoid partial words
            last_space = truncated.rfind(' ')
            if last_space > self.MAX_QUERY_LENGTH // 2:
                truncated = truncated[:last_space]
            sanitized = truncated
            self._log(f"Query truncated to {len(sanitized)} chars")

        return sanitized.strip()

    def search(
        self,
        query: str,
        search_depth: str = "advanced",
        max_results: int = 10,
        include_answer: bool = True,
        include_raw_content: bool = False
    ) -> Dict[str, Any]:
        """Perform a search query with sanitization."""

        # Sanitize query before sending
        clean_query = self._sanitize_query(query)

        self._log(f"Searching: {clean_query[:60]}...")

        endpoint = f"{self.base_url}/search"

        payload = {
            "api_key": self.api_key,
            "query": clean_query,
            "search_depth": search_depth,
            "max_results": max_results,
            "include_answer": include_answer,
            "include_raw_content": include_raw_content
        }

        try:
            response = requests.post(endpoint, json=payload, timeout=30)

            # Handle specific error codes
            if response.status_code == 400:
                self._log(f"⚠️ Bad request, trying simplified query...")
                # Try with even shorter query
                words = clean_query.split()[:10]  # Just first 5 words
                simplified_query = ' '.join(words)
                payload["query"] = simplified_query
                self._log(f"Simplified to: {simplified_query}")
                response = requests.post(endpoint, json=payload, timeout=30)

            response.raise_for_status()

            data = response.json()

            results_count = len(data.get("results", []))
            response_time = data.get("response_time", 0)
            self._log(f"✓ Found {results_count} results in {response_time:.1f}s")

            return data

        except requests.exceptions.RequestException as e:
            self._log(f"❌ Request error: {e}")
            return {"error": str(e), "results": []}

    def search_for_research(
        self,
        query: str,
        topic_context: str = "",
        num_sources: int = 5
    ) -> List[Dict]:
        """Search optimized for research."""

        # Build query - keep it concise
        if topic_context:
            enhanced_query = f"{query} {topic_context}"
        else:
            enhanced_query = query

        self._log(f"Research query: {enhanced_query[:60]}...")

        results = self.search(
            query=enhanced_query,
            search_depth="advanced",
            max_results=num_sources * 2,
            include_answer=True
        )

        if "error" in results:
            return []

        sources = []

        for result in results.get("results", [])[:num_sources]:
            reliability = self._assess_reliability(result.get("url", ""))

            sources.append({
                "title": result.get("title", "Unknown Title"),
                "url": result.get("url", ""),
                "content": result.get("content", ""),
                "score": result.get("score", 0),
                "reliability": reliability,
                "published_date": "Unknown",
            })

        sources.sort(key=lambda x: x["reliability"] + x["score"], reverse=True)

        self._log(f"Returning {len(sources)} sources")
        return sources

    def _assess_reliability(self, url: str) -> float:
        """Assess source reliability based on domain."""
        url_lower = url.lower()

        high_reliability = [
            "pubmed", "nih.gov", "who.int", "cdc.gov", "nejm.org",
            "thelancet.com", "nature.com", "science.org", "bmj.com",
            "jamanetwork.com", "cochrane.org", ".edu", "springer.com",
            "wiley.com", "elsevier.com", "oxford", "cambridge"
        ]

        medium_reliability = [
            "wikipedia.org", "medscape.com", "webmd.com", "mayoclinic.org",
            "healthline.com", "medicalnewstoday.com", ".gov",
            "ibm.com"
        ]

        for domain in high_reliability:
            if domain in url_lower:
                return 0.9

        for domain in medium_reliability:
            if domain in url_lower:
                return 0.6

        if any(x in url_lower for x in ["news", "times", "post", "guardian"]):
            return 0.5

        return 0.4


# Recreate search tool
tavily_search = TavilySearchTool(verbose=True)

print("✅ TavilySearchTool with query sanitization")

✅ TavilySearchTool with query sanitization


In [17]:
# Test the API key
import requests
import json
def test_tavily_connection():
    """Test if Tavily API is working."""
    print("\n🔍 Testing Tavily API connection...")


    # Simple test query
    results = tavily_search.search("artificial intelligence in medicine", max_results=3)

    if "error" in results:
        print(f"\n❌ TAVILY TEST FAILED: {results['error']}")
        print("\nTo fix:")
        print("1. Get API key from https://tavily.com")
        print("2. Set environment variable: export TAVILY_API_KEY='your-key'")
        print("3. Or pass directly: TavilySearchTool(api_key='your-key')")
        return False
    else:
        print(f"\n✅ TAVILY TEST PASSED: {len(results.get('results', []))} results")
        return True


# Run test
is_true = test_tavily_connection()
is_true


🔍 Testing Tavily API connection...
[Tavily] Searching: artificial intelligence in medicine...
[Tavily] ✓ Found 3 results in 2.6s

✅ TAVILY TEST PASSED: 3 results


True

##### **Cell 3: Image Generation Tool(Gemini NanoBanana API)**

In [18]:
# =============================================================================
# GEMINI 3 PRO IMAGE GENERATOR (FIXED - FROM USER)
# =============================================================================

import requests
import json
import base64
from typing import Optional, Dict, Any


class Gemini3ProImageGenerator:
    """
    Gemini 3 Pro Image Preview (REST API, no SDK).
    Uses generateContent (non-streaming) for maximum compatibility.
    """

    def __init__(self, api_key: str = GEMINI_API_KEY, verbose: bool = True):
        self.api_key = api_key
        self.base_url = "https://generativelanguage.googleapis.com/v1beta"
        self.model = "gemini-3-pro-image-preview"
        self.method = "generateContent"
        self.verbose = verbose

        # Style enhancement dictionaries
        self.style_prompts = {
            "scientific": "Create a clean, professional scientific diagram with clear labels. Use a white background with blue and gray tones suitable for academic publication.",
            "medical": "Create a medical illustration suitable for academic publication. Use anatomically accurate representations with professional labeling.",
            "minimalist": "Create a minimalist infographic with simple shapes, clear hierarchy, and modern design aesthetic."
        }

        self.type_prompts = {
            "diagram": "Create a technical diagram showing",
            "illustration": "Create an educational illustration depicting",
            "infographic": "Create an infographic explaining",
            "flowchart": "Create a flowchart showing the process of"
        }

    def _log(self, msg):
        if self.verbose:
            print(f"[ImageGen] {msg}")

    def generate_image(
        self,
        prompt: str,
        image_type: str = "diagram",
        style: str = "scientific",
        aspect_ratio: str = "16:9",
        resolution: str = "1K",
        save_path: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Generate an image using Gemini 3 Pro.

        Args:
            prompt: Description of image to generate
            image_type: "diagram", "illustration", "infographic", "flowchart"
            style: "scientific", "medical", "minimalist"
            aspect_ratio: "1:1", "16:9", "4:3", etc.
            resolution: "1K", "2K", "4K"
            save_path: Optional path to save the image

        Returns:
            Dict with success status, image data, and metadata
        """
        self._log(f"Generating {image_type} ({style}): {prompt[:50]}...")

        # Build enhanced prompt
        type_prefix = self.type_prompts.get(image_type, "Create an image of")
        style_suffix = self.style_prompts.get(style, "")
        full_prompt = f"{type_prefix} {prompt}. {style_suffix}"

        endpoint = f"{self.base_url}/models/{self.model}:{self.method}?key={self.api_key}"

        payload = {
            "contents": [
                {
                    "parts": [{"text": full_prompt}]
                }
            ],
            "generationConfig": {
                "responseModalities": ["IMAGE", "TEXT"],
                "imageConfig": {
                    "aspectRatio": aspect_ratio,
                    "imageSize": resolution
                }
            }
        }

        try:
            response = requests.post(
                endpoint,
                json=payload,
                headers={"Content-Type": "application/json"},
                timeout=90
            )

            if response.status_code != 200:
                self._log(f"❌ API error: {response.status_code}")
                return {
                    "success": False,
                    "error": f"API error {response.status_code}",
                    "details": response.text
                }

            result = response.json()

            # Extract inlineData (handles both camelCase and snake_case)
            try:
                part = result["candidates"][0]["content"]["parts"][0]

                if "inlineData" in part:
                    b64_img = part["inlineData"]["data"]
                elif "inline_data" in part:
                    b64_img = part["inline_data"]["data"]
                else:
                    raise KeyError("No inlineData or inline_data found")

            except Exception as e:
                self._log(f"❌ No image in response: {e}")
                return {
                    "success": False,
                    "error": "No image returned by Gemini",
                    "raw_response": result
                }

            # Save image if path provided
            if save_path:
                self._save_image(b64_img, save_path)
                self._log(f"✓ Saved to: {save_path}")

            self._log(f"✓ Generated successfully ({len(b64_img)} bytes)")

            return {
                "success": True,
                "prompt": full_prompt,
                "image_type": image_type,
                "style": style,
                "aspect_ratio": aspect_ratio,
                "resolution": resolution,
                "image_base64": b64_img,
                "save_path": save_path
            }

        except requests.exceptions.Timeout:
            self._log("❌ Request timeout")
            return {"success": False, "error": "Request timeout (90s)"}

        except Exception as e:
            self._log(f"❌ Error: {e}")
            return {"success": False, "error": str(e)}

    def _save_image(self, b64_data: str, save_path: str):
        """Save base64 image data to file."""
        try:
            # Ensure directory exists
            save_dir = os.path.dirname(save_path)
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)

            img_bytes = base64.b64decode(b64_data)
            with open(save_path, "wb") as f:
                f.write(img_bytes)
        except Exception as e:
            self._log(f"Error saving image: {e}")

    def generate_for_section(
        self,
        section_title: str,
        section_content: str,
        output_dir: str
    ) -> Optional[str]:
        """
        Analyze section content and generate appropriate image if relevant.

        Args:
            section_title: Title of the section
            section_content: Full text content of the section
            output_dir: Directory to save the image

        Returns:
            Path to generated image, or None if no image needed
        """
        content = section_content.lower()

        image_needed = False
        image_type = "diagram"
        prompt = ""

        # Detect content type and determine image needs
        if any(w in content for w in ["process", "pathway", "mechanism", "cycle", "workflow", "pipeline"]):
            image_needed = True
            image_type = "flowchart"
            prompt = f"the process described in: {section_title}"

        elif any(w in content for w in ["anatomy", "structure", "organ", "tissue", "cell", "body"]):
            image_needed = True
            image_type = "illustration"
            prompt = f"anatomical structure of: {section_title}"

        elif any(w in content for w in ["statistics", "data", "percentage", "rate", "number", "survey", "study found"]):
            image_needed = True
            image_type = "infographic"
            prompt = f"key statistics about: {section_title}"

        elif any(w in content for w in ["comparison", "versus", "difference", "compare", "contrast"]):
            image_needed = True
            image_type = "diagram"
            prompt = f"comparison diagram for: {section_title}"

        elif any(w in content for w in ["algorithm", "model", "neural network", "machine learning", "ai system"]):
            image_needed = True
            image_type = "diagram"
            prompt = f"AI/ML architecture diagram for: {section_title}"

        if not image_needed:
            self._log(f"No image needed for: {section_title}")
            return None

        # Create safe filename
        safe_title = "".join(c for c in section_title if c.isalnum() or c in " -_")[:50]
        image_filename = f"{safe_title.replace(' ', '_')}.png"
        image_path = os.path.join(output_dir, "images", image_filename)

        # Ensure images directory exists
        os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)

        result = self.generate_image(
            prompt=prompt,
            image_type=image_type,
            style="scientific",
            save_path=image_path
        )

        if result.get("success"):
            return image_path
        else:
            self._log(f"Image generation failed: {result.get('error')}")
            return None


# Create image generator instance
image_generator = Gemini3ProImageGenerator(verbose=True)

print("✅ Gemini3ProImageGenerator configured")

✅ Gemini3ProImageGenerator configured


In [ ]:
# =============================================================================
# TEST IMAGE GENERATION INTEGRATION
# =============================================================================

def test_image_generation_for_sections():
    """Test image generation for different section types."""

    print("="*70)
    print("Testing Image Generation for Research Sections")
    print("="*70)

    test_cases = [
        {
            "title": "Machine Learning Pipeline Architecture",
            "content": "The ML pipeline involves several process steps including data preprocessing, feature extraction, model training, and evaluation. The workflow begins with raw data collection and ends with model deployment.",
            "expected_type": "flowchart"
        },
        {
            "title": "Global AI Adoption Statistics",
            "content": "Recent surveys show that 75% of hospitals have adopted AI in some form. The percentage of radiologists using AI tools has increased from 20% to 65% over three years. Data indicates significant growth.",
            "expected_type": "infographic"
        },
        {
            "title": "Neural Network vs Traditional ML",
            "content": "When comparing neural networks versus traditional machine learning approaches, several key differences emerge. The comparison shows that deep learning excels at unstructured data.",
            "expected_type": "diagram"
        },
        {
            "title": "Literature Review Methods",
            "content": "This section reviews the methodology used in systematic literature reviews, focusing on search strategies and inclusion criteria.",
            "expected_type": None  # Should not generate image
        }
    ]



    # Create test output directory
    test_dir = "./test_images"
    os.makedirs(f"{test_dir}/images", exist_ok=True)

    results = []

    for case in test_cases:
        print(f"\n--- {case['title']} ---")
        print(f"Expected: {case['expected_type'] or 'No image'}")

        image_path = image_generator.generate_for_section(
            case["title"],
            case["content"],
            test_dir
        )

        if image_path:
            print(f"✓ Generated: {os.path.basename(image_path)}")
            results.append({"title": case["title"], "success": True, "path": image_path})
        else:
            print(f"✓ No image generated (as expected)" if case["expected_type"] is None else "⚠️ No image generated")
            results.append({"title": case["title"], "success": case["expected_type"] is None, "path": None})

    print(f"\n{'='*70}")
    print(f"Results: {sum(1 for r in results if r['success'])}/{len(results)} as expected")

    return results


# Run test
test_results = test_image_generation_for_sections()

Testing Image Generation for Research Sections

--- Machine Learning Pipeline Architecture ---
Expected: flowchart
[ImageGen] Generating flowchart (scientific): the process described in: Machine Learning Pipelin...
[ImageGen] ❌ API error: 503
[ImageGen] Image generation failed: API error 503
⚠️ No image generated

--- Global AI Adoption Statistics ---
Expected: infographic
[ImageGen] Generating infographic (scientific): key statistics about: Global AI Adoption Statistic...
[ImageGen] ❌ API error: 503
[ImageGen] Image generation failed: API error 503
⚠️ No image generated

--- Neural Network vs Traditional ML ---
Expected: diagram
[ImageGen] Generating illustration (scientific): anatomical structure of: Neural Network vs Traditi...
[ImageGen] ❌ API error: 503
[ImageGen] Image generation failed: API error 503
⚠️ No image generated

--- Literature Review Methods ---
Expected: No image
[ImageGen] Generating infographic (scientific): key statistics about: Literature Review Methods...
[Image

##### **Cell 4: File Manager Tool**

In [19]:
# =============================================================================
# UPDATED FILE MANAGER WITH IMAGE SUPPORT (FIX #5: No duplicate labels)
# =============================================================================

class FileManager:
    """
    Manages research output files.
    Handles section files, images, citations, and final assembly.
    """

    def __init__(self, memory: PersistentMemory):
        self.memory = memory

    def save_section(
        self,
        section_id: str,
        title: str,
        content: str,
        citations_used: List[int],
        image_path: Optional[str] = None
    ) -> str:
        """
        Save a section to markdown file.
        Returns the file path.
        """
        if not self.memory.current_state:
            raise ValueError("No active project")

        output_dir = Path(self.memory.current_state.output_dir)
        sections_dir = output_dir / "sections"
        sections_dir.mkdir(parents=True, exist_ok=True)

        # Build markdown content
        md_lines = [
            f"# {title}\n",
        ]

        # Add image if present (with relative path for portability)
        # FIX #5: Removed duplicate label - only include the image reference
        # The ResearchPaperFormatter will add proper "Figure X." captions
        if image_path and os.path.exists(image_path):
            try:
                relative_path = os.path.relpath(image_path, sections_dir)
                md_lines.append(f"\n![{title}]({relative_path})\n")
                # REMOVED: md_lines.append(f"*Figure: Visual representation of {title}*\n")
            except ValueError:
                md_lines.append(f"\n![{title}]({image_path})\n")

        # Add main content
        md_lines.append("\n")
        md_lines.append(content)

        # ================== version 2b =====================================
        # New: Extract citations actually used in content
        import re
        inline_citations = set(re.findall(r'\[(\d+)\]', content))
        inline_citation_ids = [int(c) for c in inline_citations if c.isdigit()]

        # Combine passed citation_ids with those found in content
        all_citation_ids = set(citations_used) | set(inline_citation_ids)

        # Add section references
        if all_citation_ids:
            md_lines.append("\n\n---\n")
            md_lines.append("### Section References\n")

            references_added = 0
            for cid in sorted(all_citation_ids):
                citation = self.memory.get_citation_by_id(cid)
                if citation:
                    md_lines.append(f"- {citation.to_vancouver()}\n")
                    references_added += 1
                else:
                    # Debug: Citation not found
                    print(f"[FileManager] Warning: Citation [{cid}] not found in memory")

            if references_added == 0:
                # Remove the empty references section
                md_lines = md_lines[:-2]
                print(f"[FileManager] Warning: No valid citations found for IDs: {all_citation_ids}")


        # ====================================================================
        """
        # Add section citations
        if citations_used:
            md_lines.append("\n\n---\n")
            md_lines.append("### Section References\n")
            for cid in citations_used:
                citation = self.memory.get_citation_by_id(cid)
                if citation:
                    md_lines.append(f"- {citation.to_vancouver()}\n")
        """
        # Write file
        safe_title = self._sanitize_filename(title)
        file_path = sections_dir / f"{section_id}_{safe_title}.md"

        with open(file_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(md_lines))

        # Calculate word count
        word_count = len(content.split())

        # Update memory
        self.memory.mark_section_complete(section_id, str(file_path), word_count)

        print(f"✅ Saved section: {file_path}")
        print(f"   Words: {word_count}")
        if image_path:
            print(f"   Image: {os.path.basename(image_path)}")

        return str(file_path)

    def _sanitize_filename(self, name: str) -> str:
        """Convert title to safe filename."""
        safe = "".join(c for c in name if c.isalnum() or c in " -_")
        return safe.replace(" ", "_")[:50]

    def assemble_final_document(self) -> str:
        """
        Combine all sections into final document.
        Returns path to final document.
        """
        if not self.memory.current_state:
            raise ValueError("No active project")

        state = self.memory.current_state
        output_dir = Path(state.output_dir)

        # Build final document
        final_lines = [
            f"# {state.topic}\n",
            f"*Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n",
            f"*Research conducted using Deep Research SDK with MedGemma-1.5-4B*\n",
            "\n---\n",
        ]

        # Add thesis/abstract if present
        if state.outline and state.outline.thesis:
            final_lines.append("## Abstract\n")
            final_lines.append(f"{state.outline.thesis}\n")
            final_lines.append("\n---\n")

        # Table of contents
        final_lines.append("## Table of Contents\n")
        if state.outline:
            for i, section in enumerate(state.outline.sections, 1):
                anchor = self._sanitize_filename(section.title).lower()
                final_lines.append(f"{i}. [{section.title}](#{anchor})\n")
        final_lines.append(f"{len(state.outline.sections) + 1 if state.outline else 1}. [References](#references)\n")
        final_lines.append("\n---\n\n")

        # Add each section
        sections_dir = output_dir / "sections"
        total_words = 0
        images_included = 0

        for section_file in sorted(sections_dir.glob("*.md")):
            with open(section_file, 'r', encoding='utf-8') as f:
                section_content = f.read()

            final_lines.append(section_content)
            final_lines.append("\n\n---\n\n")
            total_words += len(section_content.split())

            if "![" in section_content:
                images_included += 1

        # Add references section
        final_lines.append("## References\n\n")
        final_lines.append(self.memory.get_all_citations_formatted())

        # Write final document
        final_path = output_dir / "FINAL_REPORT.md"

        with open(final_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(final_lines))

        pages = total_words / 500

        print(f"\n{'='*70}")
        print(f"✅ FINAL DOCUMENT ASSEMBLED")
        print(f"{'='*70}")
        print(f"📄 Output: {final_path}")
        print(f"📊 Statistics:")
        print(f"   - Total words: {total_words:,}")
        print(f"   - Estimated pages: {pages:.1f}")
        print(f"   - Total citations: {len(state.citations)}")
        print(f"   - Images included: {images_included}")
        print(f"   - Sections: {len(state.completed_sections)}")

        return str(final_path)

    def get_progress_report(self) -> str:
        """Generate a progress report."""
        if not self.memory.current_state:
            return "No active project"

        state = self.memory.current_state

        lines = [
            "# Research Progress Report",
            "",
            f"**Project ID:** {state.project_id}",
            f"**Topic:** {state.topic}",
            f"**Status:** {state.status.value}",
            f"**Started:** {state.created_at}",
            f"**Last Updated:** {state.updated_at}",
            "",
            "## Progress",
        ]

        if state.outline:
            total = len(state.outline.sections)
            completed = len(state.completed_sections)
            progress_pct = (completed / total * 100) if total > 0 else 0
            lines.append(f"- Sections: {completed}/{total} ({progress_pct:.0f}%)")

            total_words = sum(s.word_count for s in state.outline.sections if s.status == "complete")
            target_words = state.outline.total_target_words
            word_pct = (total_words / target_words * 100) if target_words > 0 else 0
            lines.append(f"- Words: {total_words:,}/{target_words:,} ({word_pct:.0f}%)")
            lines.append(f"- Estimated Pages: {total_words / 500:.1f}")

        lines.append(f"- Citations: {len(state.citations)}")
        lines.append(f"- Verified Facts: {len([f for f in state.facts if f.verified])}/{len(state.facts)}")
        lines.append(f"- Search Queries: {len(state.search_history)}")

        if state.output_dir:
            images_dir = Path(state.output_dir) / "images"
            if images_dir.exists():
                image_count = len(list(images_dir.glob("*.png")))
                lines.append(f"- Images Generated: {image_count}")

        if state.error_log:
            lines.append(f"\n## Recent Errors ({len(state.error_log)})")
            for error in state.error_log[-3:]:
                lines.append(f"- {error}")

        if state.human_review_notes:
            unresolved = [n for n in state.human_review_notes if not n.get("resolved")]
            if unresolved:
                lines.append(f"\n## Pending Review ({len(unresolved)})")
                for note in unresolved[:5]:
                    lines.append(f"- [{note['section']}] {note['note'][:50]}...")

        return "\n".join(lines)


# Recreate file manager
file_manager = FileManager(memory)

print("✅ FileManager updated - removed duplicate image labels")


✅ FileManager updated - removed duplicate image labels


##### **Cell 5: Sub-Agent Tool Registries**

In [20]:
# =============================================================================
# PLANNER AGENT TOOLS
# =============================================================================

planner_registry = ToolRegistry()

planner_registry.register(
    name="generate_outline",
    description="Generate a comprehensive research outline with chapters and sections",
    parameters={
        "topic": "string - The main research topic",
        "target_pages": "number - Target number of pages (default 80)",
        "depth": "string - 'comprehensive' or 'overview'"
    },
    required=["topic"],
    example={
        "name": "generate_outline",
        "arguments": {"topic": "AI in Healthcare", "target_pages": 80, "depth": "comprehensive"}
    },
    docstring="Creates a structured outline for the research document including chapters, sections, and subsections. Estimates word counts for each section.",
    func=lambda topic, target_pages=80, depth="comprehensive": f"OUTLINE_PLACEHOLDER:{topic}:{target_pages}:{depth}"
)

planner_registry.register(
    name="refine_scope",
    description="Refine and narrow the research scope based on available sources",
    parameters={
        "current_scope": "string - Current topic/scope description",
        "available_sources": "number - Number of quality sources found",
        "feedback": "string - Any feedback from initial research"
    },
    required=["current_scope"],
    example={
        "name": "refine_scope",
        "arguments": {"current_scope": "AI in medicine", "available_sources": 50, "feedback": "Many sources on diagnostics"}
    },
    docstring="Helps narrow or adjust the research scope based on source availability and quality.",
    func=lambda current_scope, available_sources=0, feedback="": f"SCOPE_REFINED:{current_scope}"
)

planner_registry.register(
    name="estimate_resources",
    description="Estimate research resources needed (searches, time, sections)",
    parameters={
        "outline": "string - JSON string of the outline",
    },
    required=["outline"],
    example={
        "name": "estimate_resources",
        "arguments": {"outline": "{}"}
    },
    docstring="Analyzes outline and estimates resources needed for completion.",
    func=lambda outline: "RESOURCES_ESTIMATED"
)

print("✅ Planner registry: 3 tools")

✅ Planner registry: 3 tools


In [21]:
# =============================================================================
# WEB SEARCH AGENT TOOLS
# =============================================================================

search_registry = ToolRegistry()

search_registry.register(
    name="search_academic",
    description="Search for academic and peer-reviewed sources on a topic",
    parameters={
        "query": "string - Search query",
        "topic_context": "string - Broader topic for context",
        "num_sources": "number - Number of sources to retrieve (default 5)"
    },
    required=["query"],
    example={
        "name": "search_academic",
        "arguments": {"query": "machine learning diagnosis accuracy", "topic_context": "AI in healthcare", "num_sources": 5}
    },
    docstring="Searches for high-quality academic sources using Tavily. Prioritizes peer-reviewed journals, .edu, and .gov domains.",
    func=lambda query, topic_context="", num_sources=5: tavily_search.search_for_research(query, topic_context, num_sources)
)

search_registry.register(
    name="search_recent_news",
    description="Search for recent news and developments on a topic",
    parameters={
        "query": "string - Search query",
        "days_back": "number - How many days back to search (default 30)"
    },
    required=["query"],
    example={
        "name": "search_recent_news",
        "arguments": {"query": "FDA AI approval 2024", "days_back": 30}
    },
    docstring="Searches for recent news articles and press releases.",
    func=lambda query, days_back=30: tavily_search.search(f"{query} news last {days_back} days", max_results=5)
)

search_registry.register(
    name="extract_key_facts",
    description="Extract key facts from search results for citation",
    parameters={
        "search_results": "string - JSON string of search results",
        "focus_area": "string - What aspect to focus on"
    },
    required=["search_results", "focus_area"],
    example={
        "name": "extract_key_facts",
        "arguments": {"search_results": "[]", "focus_area": "statistics"}
    },
    docstring="Analyzes search results and extracts citable facts.",
    func=lambda search_results, focus_area: "FACTS_EXTRACTED"
)

print("✅ Search registry: 3 tools")

✅ Search registry: 3 tools


In [22]:
# =============================================================================
# SECTION RESEARCHER AGENT TOOLS
# =============================================================================

researcher_registry = ToolRegistry()

researcher_registry.register(
    name="deep_research_section",
    description="Conduct deep research on a specific section/topic",
    parameters={
        "section_title": "string - Title of the section",
        "section_description": "string - What the section should cover",
        "target_words": "number - Target word count",
        "existing_facts": "string - Facts already gathered (JSON)"
    },
    required=["section_title", "section_description"],
    example={
        "name": "deep_research_section",
        "arguments": {"section_title": "Introduction", "section_description": "Overview of AI in healthcare", "target_words": 2000}
    },
    docstring="Conducts comprehensive research on a section, gathering sources and facts.",
    func=lambda section_title, section_description, target_words=2000, existing_facts="[]": f"RESEARCH:{section_title}"
)

researcher_registry.register(
    name="synthesize_findings",
    description="Synthesize research findings into coherent narrative",
    parameters={
        "facts": "string - JSON array of facts to synthesize",
        "section_context": "string - Context of the section",
        "writing_style": "string - 'academic', 'review', or 'technical'"
    },
    required=["facts", "section_context"],
    example={
        "name": "synthesize_findings",
        "arguments": {"facts": "[]", "section_context": "Introduction", "writing_style": "academic"}
    },
    docstring="Combines facts into a coherent written section with proper citations.",
    func=lambda facts, section_context, writing_style="academic": "SYNTHESIZED"
)

researcher_registry.register(
    name="identify_gaps",
    description="Identify gaps in current research coverage",
    parameters={
        "section_outline": "string - What should be covered",
        "current_facts": "string - What we have so far"
    },
    required=["section_outline", "current_facts"],
    example={
        "name": "identify_gaps",
        "arguments": {"section_outline": "AI diagnosis methods", "current_facts": "[]"}
    },
    docstring="Identifies what's missing from the research and suggests additional queries.",
    func=lambda section_outline, current_facts: "GAPS_IDENTIFIED"
)

print("✅ Researcher registry: 3 tools")

✅ Researcher registry: 3 tools


In [23]:
# =============================================================================
# VERIFIER AGENT TOOLS
# =============================================================================

verifier_registry = ToolRegistry()

verifier_registry.register(
    name="verify_fact",
    description="Verify a fact against multiple sources",
    parameters={
        "fact": "string - The fact to verify",
        "original_source": "string - URL of original source",
        "min_confirmations": "number - Minimum sources to confirm (default 2)"
    },
    required=["fact", "original_source"],
    example={
        "name": "verify_fact",
        "arguments": {"fact": "AI accuracy is 95%", "original_source": "https://...", "min_confirmations": 2}
    },
    docstring="Cross-references a fact against multiple sources to verify accuracy.",
    func=lambda fact, original_source, min_confirmations=2: {"verified": True, "confidence": 0.85, "sources": 3}
)

verifier_registry.register(
    name="check_source_credibility",
    description="Assess the credibility of a source",
    parameters={
        "url": "string - URL to check",
        "content_snippet": "string - Sample of content"
    },
    required=["url"],
    example={
        "name": "check_source_credibility",
        "arguments": {"url": "https://pubmed.ncbi.nlm.nih.gov/...", "content_snippet": "..."}
    },
    docstring="Evaluates source credibility based on domain, publication type, and content quality.",
    func=lambda url, content_snippet="": {"credibility": 0.9, "type": "academic", "peer_reviewed": True}
)

verifier_registry.register(
    name="detect_contradictions",
    description="Detect contradictions between facts",
    parameters={
        "facts": "string - JSON array of facts to check"
    },
    required=["facts"],
    example={
        "name": "detect_contradictions",
        "arguments": {"facts": "[]"}
    },
    docstring="Analyzes a set of facts for contradictions or inconsistencies.",
    func=lambda facts: {"contradictions": [], "warnings": []}
)

verifier_registry.register(
    name="flag_for_review",
    description="Flag content for human review",
    parameters={
        "section": "string - Section identifier",
        "issue": "string - Description of the issue",
        "severity": "string - 'low', 'medium', or 'high'"
    },
    required=["section", "issue"],
    example={
        "name": "flag_for_review",
        "arguments": {"section": "chapter_3", "issue": "Conflicting statistics", "severity": "medium"}
    },
    docstring="Flags content for human review with issue description.",
    func=lambda section, issue, severity="medium": memory.add_human_review_note(section, issue, severity)
)

print("✅ Verifier registry: 4 tools")

✅ Verifier registry: 4 tools


In [24]:
# =============================================================================
# WRITER AGENT TOOLS
# =============================================================================

writer_registry = ToolRegistry()

writer_registry.register(
    name="write_section",
    description="Write a complete section with proper formatting",
    parameters={
        "section_id": "string - Unique section identifier",
        "title": "string - Section title",
        "content_brief": "string - What to write about",
        "facts_to_include": "string - JSON array of facts with citations",
        "target_words": "number - Target word count"
    },
    required=["section_id", "title", "content_brief"],
    example={
        "name": "write_section",
        "arguments": {"section_id": "ch1_intro", "title": "Introduction", "content_brief": "Overview of topic", "target_words": 2000}
    },
    docstring="Writes a complete section using provided facts and citations.",
    func=lambda section_id, title, content_brief, facts_to_include="[]", target_words=2000: f"WRITTEN:{section_id}"
)

writer_registry.register(
    name="format_citations",
    description="Format inline citations in Vancouver style",
    parameters={
        "text": "string - Text with citation placeholders",
        "citation_map": "string - JSON mapping placeholders to citation IDs"
    },
    required=["text", "citation_map"],
    example={
        "name": "format_citations",
        "arguments": {"text": "Studies show [CITE1] that...", "citation_map": '{"CITE1": 1}'}
    },
    docstring="Replaces citation placeholders with properly formatted citations.",
    func=lambda text, citation_map: text
)

writer_registry.register(
    name="add_image_placeholder",
    description="Add an image placeholder to content",
    parameters={
        "content": "string - Current section content",
        "image_description": "string - What image should show",
        "position": "string - 'start', 'middle', or 'end'"
    },
    required=["content", "image_description"],
    example={
        "name": "add_image_placeholder",
        "arguments": {"content": "...", "image_description": "Flowchart of AI pipeline", "position": "middle"}
    },
    docstring="Inserts an image placeholder that will be filled by image generator.",
    func=lambda content, image_description, position="middle": content
)

writer_registry.register(
    name="save_section_to_file",
    description="Save completed section to markdown file",
    parameters={
        "section_id": "string - Section identifier",
        "title": "string - Section title",
        "content": "string - Full section content",
        "citation_ids": "string - JSON array of citation IDs used"
    },
    required=["section_id", "title", "content"],
    example={
        "name": "save_section_to_file",
        "arguments": {"section_id": "ch1", "title": "Introduction", "content": "...", "citation_ids": "[1,2,3]"}
    },
    docstring="Saves the section to a markdown file in the project directory.",
    func=lambda section_id, title, content, citation_ids="[]": file_manager.save_section(
        section_id, title, content, json.loads(citation_ids) if citation_ids else []
    )
)

print("✅ Writer registry: 4 tools")

✅ Writer registry: 4 tools


In [25]:
# =============================================================================
# MEMORY AGENT TOOLS
# =============================================================================

memory_registry = ToolRegistry()

memory_registry.register(
    name="get_project_state",
    description="Get current project state and progress",
    parameters={},
    required=[],
    example={
        "name": "get_project_state",
        "arguments": {}
    },
    docstring="Returns the current state of the research project.",
    func=lambda: file_manager.get_progress_report()
)

memory_registry.register(
    name="get_all_citations",
    description="Get all citations collected so far",
    parameters={
        "format": "string - 'full' for details or 'brief' for just titles"
    },
    required=[],
    example={
        "name": "get_all_citations",
        "arguments": {"format": "brief"}
    },
    docstring="Returns all citations in the current project.",
    func=lambda format="brief": memory.get_all_citations_formatted() if memory.current_state else "No project"
)

memory_registry.register(
    name="get_facts_for_topic",
    description="Get verified facts related to a topic",
    parameters={
        "topic": "string - Topic to search for",
        "section": "string - Optionally filter by section"
    },
    required=["topic"],
    example={
        "name": "get_facts_for_topic",
        "arguments": {"topic": "AI accuracy", "section": "chapter_2"}
    },
    docstring="Searches the fact bank for relevant verified facts.",
    func=lambda topic, section="": [asdict(f) for f in memory.current_state.facts if topic.lower() in f.content.lower()] if memory.current_state else []
)

memory_registry.register(
    name="search_previous_research",
    description="Search what's been researched so far",
    parameters={
        "query": "string - Search query",
    },
    required=["query"],
    example={
        "name": "search_previous_research",
        "arguments": {"query": "machine learning"}
    },
    docstring="Searches across all gathered facts and sources.",
    func=lambda query: "SEARCH_RESULTS"
)

memory_registry.register(
    name="get_pending_reviews",
    description="Get items flagged for human review",
    parameters={},
    required=[],
    example={
        "name": "get_pending_reviews",
        "arguments": {}
    },
    docstring="Returns all items that need human review.",
    func=lambda: [n for n in memory.current_state.human_review_notes if not n.get("resolved")] if memory.current_state else []
)

print("✅ Memory registry: 5 tools")

✅ Memory registry: 5 tools


##### **Cell 6: Sub-Agent Implementations**

In [26]:
# =============================================================================
# SUB-AGENT IMPLEMENTATIONS (BaseClass)
# =============================================================================

# Each sub-agent is an instance of our ConfigurableOrchestrator
# with its own specialized registry

class SubAgent:
    """
    Base class for sub-agents.
    Wraps ConfigurableOrchestrator with sub-agent specific behavior.
    """

    def __init__(
        self,
        name: str,
        registry: ToolRegistry,
        med_gemma_caller: Callable,
        system_prompt: str,
        verbose: bool = True,
        gen_config: GenerationConfig = None
    ):
        self.name = name
        self.registry = registry
        self.med_gemma = med_gemma_caller
        self.system_prompt = system_prompt
        self.verbose = verbose
        self.gen_config = gen_config or GenerationConfig()
        self.debug = False
        # Create extractor (using LLM for flexibility)
        self.extractor = LLMValueExtractor(med_gemma_caller, verbose=False)

        # Context bank for this agent
        self.context = ContextBank(self.extractor, known_keys=self.registry.get_all_context_keys(), registry=self.registry)

    def _log(self, msg):
        if self.verbose:
            print(f"[{self.name}] {msg}")

    async def execute(self, task: str, context: Dict = None) -> str:
        """
        Execute a task using this sub-agent.

        Args:
            task: The task description
            context: Optional context from other agents

        Returns:
            Result of the sub-agent execution
        """
        self._log(f"Task: {task[:80]}...")

        # Build prompt with system context
        full_prompt = f"""{self.system_prompt}

TASK: {task}

{f"CONTEXT: {json.dumps(context)}" if context else ""}

Think through this task step by step, then execute the appropriate tools."""

        # Get MedGemma's plan
        messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]

        response = self.med_gemma(
            messages=messages,
            max_new_tokens=self.gen_config.max_new_tokens,
            temperature=self.gen_config.temperature,
            # top_p=self.gen_config.top_p,
            # top_k=self.gen_config.top_k,
            # repetition_penalty=self.gen_config.repetition_penalty,
            # do_sample=self.gen_config.do_sample
        )

        # Clean response
        response = self._clean_response(response)

        self._log(f"Response: {response[:200]}...")

        return response

    def _clean_response(self, text: str) -> str:
        """Remove thinking blocks from response."""
        # Remove <unused94>...<unused95> blocks
        cleaned = text
        while "<unused94>" in cleaned:
            start = cleaned.find("<unused94>")
            end = cleaned.find("<unused95>")
            if end > start:
                cleaned = cleaned[:start] + cleaned[end + len("<unused95>"):]
            else:
                cleaned = cleaned[:start]
        return cleaned.strip()

In [27]:
# =============================================================================
# SPECIALIZED SUB-AGENTS
# =============================================================================

class PlannerAgentOld(SubAgent):
    """Agent specialized in research planning and outlining."""

    def __init__(self, med_gemma_caller: Callable, verbose: bool = True):
        system_prompt = """You are a Research Planner Agent. Your role is to:
1. Create comprehensive research outlines
2. Break down topics into logical chapters and sections
3. Estimate resource requirements
4. Ensure complete coverage of the topic

You specialize in medical and healthcare research. Create outlines that are:
- Logically structured
- Comprehensive (covering all important aspects)
- Balanced (appropriate depth for each section)
- Academic in style"""

        super().__init__(
            name="PlannerAgent",
            registry=planner_registry,
            med_gemma_caller=med_gemma_caller,
            system_prompt=system_prompt,
            verbose=verbose
        )

    async def create_outline(self, topic: str, target_pages: int = 80) -> ResearchOutline:
        """Create a complete research outline."""

        target_words = target_pages * 500  # ~500 words per page

        prompt = f"""Create a detailed research outline for the following topic:

TOPIC: {topic}
TARGET LENGTH: {target_pages} pages (~{target_words} words)

Create a JSON outline with:
1. A thesis statement (2-3 sentences summarizing the research)
2. 12-18 sections/chapters that comprehensively cover the topic
3. Each section should have:
   - A clear title
   - A brief description (what it covers)
   - 3-5 subsections
   - Estimated word count (total should sum to ~{target_words})

The outline should follow academic structure:
- Introduction/Background
- Literature Review sections
- Main topic sections (the bulk)
- Discussion/Analysis
- Future Directions
- Conclusion

Output as valid JSON:
{{
    "thesis": "...",
    "sections": [
        {{
            "section_id": "sec_01",
            "title": "...",
            "description": "...",
            "subsections": ["...", "..."],
            "target_word_count": 3000
        }},
        ...
    ]
}}

JSON:"""

        response = await self.execute(prompt)

        # Parse JSON from response
        try:
            # Find JSON block
            start = response.find('{')
            end = response.rfind('}') + 1
            if start >= 0 and end > start:
                json_str = response[start:end]
                data = json.loads(json_str)

                sections = []
                for s in data.get("sections", []):
                    sections.append(SectionOutline(
                        section_id=s.get("section_id", f"sec_{len(sections)+1:02d}"),
                        title=s.get("title", "Untitled"),
                        description=s.get("description", ""),
                        subsections=s.get("subsections", []),
                        target_word_count=s.get("target_word_count", 2500)
                    ))

                outline = ResearchOutline(
                    topic=topic,
                    thesis=data.get("thesis", ""),
                    sections=sections,
                    total_target_words=target_words
                )

                self._log(f"Created outline with {len(sections)} sections")
                return outline
        except json.JSONDecodeError as e:
            self._log(f"JSON parse error: {e}")

        # Fallback: create basic outline
        return self._create_fallback_outline(topic, target_words)

    def _create_fallback_outline(self, topic: str, target_words: int) -> ResearchOutline:
        """Create a basic outline if LLM fails."""
        sections = [
            SectionOutline("sec_01", "Introduction", f"Introduction to {topic}", ["Background", "Objectives", "Scope"], 3000),
            SectionOutline("sec_02", "Literature Review", "Review of existing research", ["Historical Context", "Current State", "Key Studies"], 5000),
            SectionOutline("sec_03", "Methodology", "Research methods and approach", ["Data Sources", "Analysis Methods"], 3000),
            SectionOutline("sec_04", "Main Analysis", f"Core analysis of {topic}", ["Key Findings", "Data Analysis", "Interpretation"], 8000),
            SectionOutline("sec_05", "Discussion", "Discussion of findings", ["Implications", "Limitations", "Comparisons"], 5000),
            SectionOutline("sec_06", "Future Directions", "Future research opportunities", ["Emerging Trends", "Research Gaps"], 3000),
            SectionOutline("sec_07", "Conclusion", "Summary and conclusions", ["Key Takeaways", "Recommendations"], 2000),
        ]

        return ResearchOutline(
            topic=topic,
            thesis=f"A comprehensive analysis of {topic}.",
            sections=sections,
            total_target_words=target_words
        )



In [28]:
!pip install -q json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.7 MB/s eta 0:00:00


In [29]:
# =============================================================================
# ENHANCEMENT PATCH: Planner Diversity + Writer Creativity + Reference Fix
# =============================================================================
#
# 3 targeted fixes for GemmaDeepResearchV2.ipynb
#   Fix A: Planner temperature scaling + diversity prompt engineering
#   Fix B: Writer creativity boost + rhetorical prompt hacks
#   Fix C: Remove "Unknown" from references
#
# Each fix shows:  CELL → WHAT TO REPLACE → REPLACEMENT CODE
# =============================================================================


# ▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
# FIX A — CELL 39: PlannerAgent with temperature scaling + diversity injection
# ▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
#
# PROBLEM: PlannerAgent uses default GenerationConfig (temp=0.7) and a rigid
#          prompt template. Running twice on the same topic → near-identical outlines.
#
# ROOT CAUSE:
#   1. No gen_config passed → defaults to temp=0.7 (too low for diversity)
#   2. The prompt is 100% deterministic — same topic always produces same structure
#   3. The "academic structure" instruction locks the model into one skeleton
#
# FIX:
#   1. Add GenerationConfig.for_planning_diverse() at temp=0.85
#   2. Inject a randomized "structural angle" into the prompt each run
#   3. Explicitly instruct the model to AVOID standard textbook ordering
# =============================================================================

import random
from json_repair import repair_json

# --- ADD this new GenerationConfig classmethod to Cell 21 ---

# In the GenerationConfig class (Cell 21), ADD this method:
#
#

# --- REPLACE entire PlannerAgent class in Cell 39 with: ---

class PlannerAgent(SubAgent):
    """Agent specialized in research planning and outlining.

    ENHANCED:
    - Higher temperature (0.85) for structural diversity
    - Randomized framing angles to avoid repetitive outlines
    - Explicit anti-template instructions
    """

    # Structural angles injected randomly to force diverse outlines.
    # Each angle reframes HOW the model organizes the same topic.
    STRUCTURAL_ANGLES = [
        {
            "frame": "chronological-evolutionary",
            "instruction": (
                "Organize the outline CHRONOLOGICALLY — trace how this field evolved over time. "
                "Start from foundational discoveries, move through paradigm shifts, and arrive at "
                "cutting-edge developments. Each section should represent a distinct era or breakthrough."
            )
        },
        {
            "frame": "problem-solution",
            "instruction": (
                "Organize the outline around PROBLEMS AND SOLUTIONS. Each section should identify "
                "a specific clinical/technical challenge, then examine how research has addressed it. "
                "Build from micro-level problems (individual techniques) to macro-level (systemic adoption)."
            )
        },
        {
            "frame": "stakeholder-centric",
            "instruction": (
                "Organize the outline around STAKEHOLDERS AND PERSPECTIVES. Dedicate sections to how "
                "this topic impacts different groups: patients, clinicians, researchers, regulators, "
                "industry, and society. Cross-cut with technical depth within each perspective."
            )
        },
        {
            "frame": "systems-thinking",
            "instruction": (
                "Organize the outline using SYSTEMS THINKING — from components to emergent behavior. "
                "Start with fundamental building blocks, then examine how they interact, then analyze "
                "emergent properties of the whole system, feedback loops, and unintended consequences."
            )
        },
        {
            "frame": "controversy-debate",
            "instruction": (
                "Organize the outline around KEY DEBATES AND CONTROVERSIES in the field. Each section "
                "should present competing viewpoints, methodological disputes, or unresolved questions. "
                "Structure as claim → evidence → counterclaim → synthesis."
            )
        },
        {
            "frame": "translational-pipeline",
            "instruction": (
                "Organize the outline as a TRANSLATIONAL PIPELINE — from bench to bedside. Start with "
                "basic science foundations, move through preclinical validation, clinical trials, "
                "regulatory pathways, clinical deployment, and real-world impact assessment."
            )
        },
        {
            "frame": "comparative-analytical",
            "instruction": (
                "Organize the outline as a COMPARATIVE ANALYSIS. Each section should compare and contrast "
                "different approaches, methodologies, or schools of thought. Use explicit comparison "
                "frameworks rather than sequential description."
            )
        },
        {
            "frame": "case-study-driven",
            "instruction": (
                "Organize the outline around LANDMARK CASE STUDIES and pivotal examples. Each section "
                "anchors to one or more real-world cases that illustrate key principles, then extracts "
                "generalizable insights. Weave theory around concrete examples."
            )
        },
    ]

    def __init__(self, med_gemma_caller: Callable, verbose: bool = True):
        system_prompt = """You are a Research Architect Agent. Your role is to design
ORIGINAL, non-formulaic research structures that reveal unexpected connections
and illuminate topics from fresh angles.

You reject cookie-cutter outlines. Every outline you produce should feel like it
was designed by a domain expert who deeply understands the intellectual landscape —
not generated from a template.

Principles:
- Each section title should be SPECIFIC and DESCRIPTIVE (not generic like "Literature Review")
- Subsections should ask QUESTIONS or make CLAIMS, not just label topics
- The ordering of sections should tell a STORY, not just catalog information
- Adjacent sections should have clear intellectual connections
- At least 2-3 sections should cover angles that a generic outline would miss"""

        # ENHANCED: Use higher temperature for diversity
        gen_config = GenerationConfig(
            temperature=0.85,
            max_new_tokens=4096,
        )

        super().__init__(
            name="PlannerAgent",
            registry=planner_registry,
            med_gemma_caller=med_gemma_caller,
            system_prompt=system_prompt,
            verbose=verbose,
            gen_config=gen_config  # ← NOW ACTUALLY PASSED
        )


    async def _parse_outline_json(self, response: str, topic: str, target_words: int, max_retries: int = 2) -> ResearchOutline:
        """
        Parse outline JSON from LLM response with 3-tier fallback:
        1. json_repair (fixes missing quotes, trailing commas, brackets)
        2. LLM retry (ask model to fix its own broken JSON)
        3. Return None (caller decides what to do)
        """

        # ─────────────────────────────────────────────────────
        # Tier 1: Extract JSON substring + json_repair
        # ─────────────────────────────────────────────────────
        start = response.find('{')
        end = response.rfind('}') + 1

        if start >= 0 and end > start:
            raw_json = response[start:end]

            # Try json_repair — handles missing quotes, trailing commas,
            # single quotes, truncated brackets, Python True/False/None
            try:
                data = repair_json(raw_json, return_objects=True)
                outline = self._build_outline(data, topic, target_words)
                if outline:
                    self._log(f"✅ Outline parsed successfully (json_repair)")
                    return outline

            except Exception as e:
                self._log(f"json_repair error: {e}")

        # ─────────────────────────────────────────────────────
        # Tier 2: LLM retry — ask model to fix its own JSON
        # ─────────────────────────────────────────────────────
        for attempt in range(max_retries):
            self._log(f"🔄 LLM JSON fix attempt {attempt + 1}/{max_retries}")
            fix_prompt = f"""The following JSON is broken. Fix it and return ONLY valid JSON. No explanation.
The JSON should have this structure:
{{
    "thesis": "string",
    "sections": [
        {{
            "section_id": "sec_01",
            "title": "string",
            "description": "string",
            "subsections": ["string", "string"],
            "target_word_count": 3000
        }}
    ]
}}

BROKEN JSON:
{response[start:end] if start >= 0 and end > start else response[-2000:]}

FIXED JSON:"""

            fix_response = await self.execute(fix_prompt)
            # Try json_repair on the fixed response
            fix_start = fix_response.find('{')
            fix_end = fix_response.rfind('}') + 1

            if fix_start >= 0 and fix_end > fix_start:
                try:
                    data = repair_json(fix_response[fix_start:fix_end], return_objects=True)
                    outline = self._build_outline(data, topic, target_words)
                    if outline:
                        self._log(f"✅ Outline parsed after LLM fix (attempt {attempt + 1})")
                        return outline
                except Exception as e:
                    self._log(f"⚠️ LLM fix attempt {attempt + 1} failed: {e}")

        self._log(f"❌ Outline parsing failed after {max_retries} attempts")
        return None

    def _build_outline(self, data: dict, topic: str, target_words: int) -> ResearchOutline:
      """
      Build ResearchOutline from parsed dict
      Returns None if data is invalid
      """
      if not isinstance(data, dict):
          return None

      sections_data = data.get("sections", [])
      if not sections_data or len(sections_data) < 2:
          return None

      sections = []
      for s in sections_data:
          if not isinstance(s, dict):
              continue
          title = s.get("title", "").strip()

          if not title:
              continue
          sections.append(SectionOutline(
              section_id=s.get("section_id", f"sec_{len(sections)+1:02d}"),
              title=title,
              description=s.get("description", ""),
              subsections=s.get("subsections", []),
              target_word_count=s.get("target_word_count", 2500)
          ))

      if len(sections) < 2:
          return None

      return ResearchOutline(
          topic=topic,
          thesis=data.get("thesis", ""),
          sections=sections,
          total_target_words=target_words
      )

    async def create_outline(self, topic: str, target_pages: int = 80) -> ResearchOutline:
        """Create a research outline with randomized structural framing."""
        target_words = target_pages * 500
        angle = random.choice(self.STRUCTURAL_ANGLES)
        self._log(f"Structural angle: {angle['frame']}")

        depth_styles = [
          "Prioritize DEPTH over breadth — fewer sections with richer subsections.",
          "Balance breadth and depth — moderate section count with substantive subsections.",
          "Prioritize BREADTH — more sections covering diverse facets, each tightly focused.",
        ]
        depth_choice = random.choice(depth_styles)
        min_sections = random.choice([8, 9, 10])
        max_sections = min_sections + random.randint(4, 8)
        prompt = f"""Design an original research outline for:
TOPIC: {topic}
TARGET LENGTH: {target_pages} pages (~{target_words} words)

## STRUCTURAL FRAMEWORK
{angle['instruction']}

## DEPTH STYLE
{depth_choice}

## REQUIREMENTS
1. Write a thesis statement (2-3 sentences) that takes a POSITION, not just describes the topic
2. Create {min_sections}-{max_sections} sections that follow the structural framework above
3. Each section needs:
   - A SPECIFIC, descriptive topic specific/related title (NOT generic labels like "Background" or "Literature Review")
   - A 1-2 sentence description of what it covers and WHY it matters
   - 3-5 subsections phrased as questions or claims where possible
   - Estimated word count (must sum to ~{target_words})
4. AVOID these generic section titles: "Introduction", "Literature Review", "Methodology",
   "Discussion", "Conclusion"

Output as valid JSON:
{{
    "thesis": "...",
    "sections": [
        {{
            "section_id": "sec_01",
            "title": "...",
            "description": "...",
            "subsections": ["...", "..."],
            "target_word_count": 3000
        }}
    ]
}}

JSON: """

        response = await self.execute(prompt)

        # Robust 3-tier parsing: json_repair -> LLM fix -> None
        outline = await self._parse_outline_json(response, topic, target_words)
        if outline:
            self._log(f"Created outline with {len(outline.sections)} sections (angle: {angle['frame']})")
            return outline

        # Last resort — only if all 3 tiers fail
        self._log("❌ Outline parsing failed after 3 attempts")
        self._log("⚠️ Trying Deterministic outline fallback...")
        return self._create_fallback_outline(topic, target_words)


    async def create_outline_old(self, topic: str, target_pages: int = 80) -> ResearchOutline:
        """Create a research outline with randomized structural framing."""

        target_words = target_pages * 500

        # Pick a random structural angle for diversity
        angle = random.choice(self.STRUCTURAL_ANGLES)
        self._log(f"Structural angle: {angle['frame']}")

        # Pick random style modifiers
        depth_styles = [
            "Prioritize DEPTH over breadth — fewer sections with richer subsections.",
            "Balance breadth and depth — moderate section count with substantive subsections.",
            "Prioritize BREADTH — more sections covering diverse facets, each tightly focused.",
        ]
        depth_choice = random.choice(depth_styles)

        # Random section count range for variety
        min_sections = random.choice([8, 9, 10])
        max_sections = min_sections + random.randint(4, 8)

        prompt = f"""Design an original research outline for:

TOPIC: {topic}
TARGET LENGTH: {target_pages} pages (~{target_words} words)

## STRUCTURAL FRAMEWORK
{angle['instruction']}

## DEPTH STYLE
{depth_choice}

## REQUIREMENTS
1. Write a thesis statement (2-3 sentences) that takes a POSITION, not just describes the topic
2. Create {min_sections}-{max_sections} sections that follow the structural framework above
3. Each section needs:
   - A SPECIFIC, descriptive title (NOT generic labels like "Background" or "Literature Review")
   - A 1-2 sentence description of what it covers and WHY it matters
   - 3-5 subsections phrased as questions or claims where possible
   - Estimated word count (must sum to ~{target_words})
4. AVOID these generic section titles: "Introduction", "Literature Review", "Methodology",
   "Discussion", "Conclusion" — instead use descriptive titles that preview the content
5. Section ordering should create a narrative arc, not just a list of topics

## ANTI-PATTERNS (DO NOT DO THESE)
- Do NOT follow a standard textbook chapter structure
- Do NOT use numbered subsection titles like "3.1 Overview of X"
- Do NOT repeat the topic name in every section title
- Do NOT start every section description with "This section covers..."

Output as valid JSON:
{{
    "thesis": "...",
    "sections": [
        {{
            "section_id": "sec_01",
            "title": "...",
            "description": "...",
            "subsections": ["...", "..."],
            "target_word_count": 3000
        }}
    ]
}}

JSON:"""

        response = await self.execute(prompt)

        # Parse JSON from response
        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start >= 0 and end > start:
                json_str = response[start:end]
                data = json.loads(json_str)

                sections = []
                for s in data.get("sections", []):
                    sections.append(SectionOutline(
                        section_id=s.get("section_id", f"sec_{len(sections)+1:02d}"),
                        title=s.get("title", "Untitled"),
                        description=s.get("description", ""),
                        subsections=s.get("subsections", []),
                        target_word_count=s.get("target_word_count", 2500)
                    ))

                outline = ResearchOutline(
                    topic=topic,
                    thesis=data.get("thesis", ""),
                    sections=sections,
                    total_target_words=target_words
                )

                self._log(f"Created outline with {len(sections)} sections (angle: {angle['frame']})")
                return outline
        except json.JSONDecodeError as e:
            self._log(f"JSON parse error: {e}")

        return self._create_fallback_outline(topic, target_words)

    def _create_fallback_outline(self, topic: str, target_words: int) -> ResearchOutline:
        """Create a basic outline if LLM fails."""
        sections = [
            SectionOutline("sec_01", "Introduction", f"Introduction to {topic}",
                          ["Background", "Objectives", "Scope"], 3000),
            SectionOutline("sec_02", f"Literature Review of {topic}", "Review of existing research",
                          ["Historical Context", "Current State", "Key Studies"], 5000),
            SectionOutline("sec_03", "Methodology", "Research methods and approach",
                          ["Data Sources", "Analysis Methods"], 3000),
            SectionOutline("sec_04", "Main Analysis", f"Core analysis of {topic}",
                          ["Key Findings", "Data Analysis", "Interpretation"], 8000),
            SectionOutline("sec_05", "Discussion", "Discussion of findings",
                          ["Implications", "Limitations", "Comparisons"], 5000),
            SectionOutline("sec_06", "Future Directions", "Future research opportunities",
                          ["Emerging Trends", "Research Gaps"], 3000),
            SectionOutline("sec_07", "Conclusion", "Summary and conclusions",
                          ["Key Takeaways", "Recommendations"], 2000),
        ]

        return ResearchOutline(
            topic=topic,
            thesis=f"A comprehensive analysis of {topic}.",
            sections=sections,
            total_target_words=target_words
        )


print("✅ PlannerAgent enhanced with temperature scaling (0.85) and structural diversity injection")

✅ PlannerAgent enhanced with temperature scaling (0.85) and structural diversity injection


In [30]:
# =============================================================================
# CONTENT DEDUPLICATION — Cycle Detection on Paragraph Sequences
# =============================================================================
#
# Standalone function. Place in its own cell before the WriterAgent cell.
#
# Pattern the model produces:
#   [P0, P1, P2, P3, P4, P0, P1, P2, P3, P4, P0, P1, P2, P3 <truncated>]
#    |--- first cycle ---|  |--- second cycle --|  |--- cut off ---------|
#
# Usage in WriterAgent.write_section():
#
#     response = await self.execute(write_prompt)
#     content = deduplicate_content(response, verbose=self.verbose)
#     content = self._clean_section(content, section.title)
#     content = self.output_cleaner.clean(content, section.title)
#
# =============================================================================

import re


def _paragraph_similarity(a: str, b: str) -> float:
    """Word-level Jaccard similarity between two text blocks."""
    words_a = set(a.lower().split())
    words_b = set(b.lower().split())
    if not words_a or not words_b:
        return 0.0
    intersection = len(words_a & words_b)
    union = len(words_a | words_b)
    return intersection / union if union else 0.0


def _find_cycle_period(paragraphs: list, threshold: float = 0.65) -> int | None:
    """
    Find the smallest period k where para[k:2k] ≈ para[0:k].
    Returns k if found, None otherwise.
    """
    n = len(paragraphs)

    for k in range(2, n // 2 + 1):
        # Quick check: does para[k] match para[0]?
        if _paragraph_similarity(paragraphs[k], paragraphs[0]) < threshold:
            continue

        # Verify: check how many paragraphs in [k:2k] match [0:k]
        checks = min(k, n - k)
        matches = sum(
            1 for i in range(checks)
            if _paragraph_similarity(paragraphs[i], paragraphs[k + i]) >= threshold
        )

        if checks > 0 and matches / checks >= 0.6:
            return k

    return None


def _find_partial_repeat(paragraphs: list, threshold: float = 0.65) -> int | None:
    """
    Detect a partial trailing repeat: [P0..Pk, P0..Pj<cut>] where j < k.
    Returns the start index of the repeat, or None.
    """
    n = len(paragraphs)

    for start in range(n // 2, n - 1):
        tail_len = n - start
        if tail_len < 2:
            break

        matches = sum(
            1 for i in range(tail_len)
            if _paragraph_similarity(paragraphs[start + i], paragraphs[i]) >= threshold
        )

        if matches / tail_len >= 0.6:
            return start

    return None


def _trim_broken_sentence(text: str) -> str:
    """Trim a trailing incomplete sentence caused by token cutoff."""
    if not text or text[-1] in '.!?:)"\'':
        return text

    last_period = max(text.rfind('. '), text.rfind('.\n'), text.rfind('.'))

    # Only trim if the last sentence boundary is in the back half
    if last_period > len(text) * 0.3:
        return text[:last_period + 1]

    return text


def deduplicate_content(content: str, verbose: bool = True, threshold: float = 0.65) -> str:
    """
    Detect and remove full-cycle repetition in LLM output.

    Args:
        content:    Raw LLM output string
        verbose:    Print dedup diagnostics
        threshold:  Word-overlap ratio to consider two paragraphs identical (0-1)

    Returns:
        Single clean copy of the content with broken trailing sentence trimmed.
    """
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', content) if p.strip()]

    if len(paragraphs) < 4:
        return content

    original_count = len(paragraphs)

    # Pass 1: Full cycle detection
    period = _find_cycle_period(paragraphs, threshold)

    if period is not None:
        paragraphs = paragraphs[:period]
        if verbose:
            print(
                f"  🔄 [Dedup] Full cycle: period={period}, "
                f"had {original_count} paragraphs → kept {period}"
            )
    else:
        # Pass 2: Partial trailing repeat
        repeat_start = _find_partial_repeat(paragraphs, threshold)

        if repeat_start is not None:
            trimmed = len(paragraphs) - repeat_start
            paragraphs = paragraphs[:repeat_start]
            if verbose:
                print(
                    f"  🔄 [Dedup] Partial repeat at paragraph {repeat_start}, "
                    f"trimmed {trimmed} trailing paragraphs"
                )

    # Pass 3: Trim broken last sentence
    if paragraphs:
        original_last = paragraphs[-1]
        paragraphs[-1] = _trim_broken_sentence(paragraphs[-1])
        if verbose and paragraphs[-1] != original_last:
            print(f"  ✂️ [Dedup] Trimmed broken trailing sentence")

    return '\n\n'.join(paragraphs)


print("✅ deduplicate_content() defined — standalone cycle detection dedup")

✅ deduplicate_content() defined — standalone cycle detection dedup


In [31]:
!pip install -q pytest
# test_deduplicate.py
import re
# from dedupe_module import deduplicate_content  # change to your module name

def normalize(s: str) -> str:
    return re.sub(r'\s+', ' ', s.strip())

def test_no_repeat_short():
    text = "Intro paragraph.\n\nAnother paragraph."
    out = deduplicate_content(text, verbose=False)
    assert normalize(out) == normalize(text)

def test_exact_full_cycle():
    # P0,P1,P2,P0,P1,P2
    p0 = "Paragraph zero with unique content."
    p1 = "Paragraph one with other content."
    p2 = "Paragraph two concluding."
    content = "\n\n".join([p0, p1, p2, p0, p1, p2])
    out = deduplicate_content(content, verbose=False)
    assert normalize(out) == normalize("\n\n".join([p0, p1, p2]))

def test_partial_repeat_trim():
    # P0,P1,P2,P0,P1 (cut)
    p0 = "First paragraph."
    p1 = "Second paragraph."
    p2 = "Third paragraph."
    content = "\n\n".join([p0, p1, p2, p0, p1])
    out = deduplicate_content(content, verbose=False)
    assert normalize(out) == normalize("\n\n".join([p0, p1, p2]))

def test_similar_but_not_repeat():
    # Similar vocabulary but different ordering should not be removed
    a = "The model learns patterns from labeled examples."
    b = "Labeled examples help the model learn different patterns."
    content = "\n\n".join([a, b])
    out = deduplicate_content(content, verbose=False)
    assert normalize(out) == normalize(content)

def test_trim_broken_last_sentence():
    # Last paragraph cut mid-sentence
    p0 = "Complete paragraph one."
    p1 = "This paragraph ends properly."
    p2 = "This paragraph was cut off mid"
    content = "\n\n".join([p0, p1, p2])
    out = deduplicate_content(content, verbose=False)
    assert out.endswith('.')  # last paragraph should end with punctuation

test_no_repeat_short()
test_exact_full_cycle()
test_partial_repeat_trim()
test_similar_but_not_repeat()

In [32]:
# Cell 2: quick examples
examples = {
    "exact_cycle": "A\n\nB\n\nC\n\nA\n\nB\n\nC",
    "partial": "A\n\nB\n\nC\n\nA\n\nB",
    "no_repeat": "Intro\n\nBody\n\nConclusion",
    "cut_off": "Start\n\nMiddle\n\nThis ends mid"
}

for name, txt in examples.items():
    print("=== Example:", name)
    cleaned = deduplicate_content(txt, verbose=True)
    print(cleaned)
    print()

=== Example: exact_cycle
  🔄 [Dedup] Full cycle: period=3, had 6 paragraphs → kept 3
A

B

C

=== Example: partial
  🔄 [Dedup] Partial repeat at paragraph 3, trimmed 2 trailing paragraphs
A

B

C

=== Example: no_repeat
Intro

Body

Conclusion

=== Example: cut_off
Start

Middle

This ends mid



In [33]:
# =============================================================================
# ENHANCED WRITER AGENT - PHD-LEVEL ACADEMIC STYLE (FIX #3)
# =============================================================================

import random

# PhD-level system prompt for academic writing
PHD_WRITER_SYSTEM_PROMPT_OLD = """You are an Expert Academic Research Writer with PhD-level expertise in scientific communication. Your writing must demonstrate:

## SENTENCE STRUCTURE VARIETY
- Use a sophisticated mix of sentence types:
  * Simple sentences for key assertions
  * Compound sentences with coordinating conjunctions (and, but, or, nor, for, yet, so)
  * Complex sentences with subordinating conjunctions (although, because, since, while, whereas, despite)
  * Compound-complex sentences for nuanced arguments
- Vary sentence length: mix short punchy sentences (8-12 words) with longer elaborative ones (25-35 words)
- NEVER repeat the same sentence structure in consecutive sentences

## COHESIVE DEVICES (Use extensively)
Additive: furthermore, moreover, in addition, additionally, equally important
Adversative: however, nevertheless, nonetheless, conversely, on the contrary, in contrast
Causal: consequently, therefore, thus, hence, as a result, accordingly
Temporal: subsequently, meanwhile, previously, simultaneously, hitherto
Exemplifying: for instance, specifically, notably, particularly, in particular
Summarizing: in summary, to conclude, ultimately, overall, in essence

## ACADEMIC HEDGING (Required for claims)
- "Evidence suggests that..." rather than "It is true that..."
- "This may indicate..." rather than "This proves..."
- "Studies have demonstrated..." with appropriate citations
- "It appears that..." / "The data imply..." / "One interpretation is..."

## PARAGRAPH STRUCTURE
1. Topic sentence (main claim)
2. Supporting evidence with citations [X]
3. Analysis/interpretation of evidence
4. Transition to next paragraph

## LEXICAL SOPHISTICATION
- Use precise academic vocabulary appropriate to the field
- Avoid colloquialisms and informal language
- Employ field-specific terminology accurately
- Vary word choice - NEVER use the same key term more than twice per paragraph

## CRITICAL RULES
- Every factual claim MUST have inline citation [X]
- Maintain third-person perspective throughout
- Use passive voice strategically for objectivity
- Create logical flow between ideas with explicit transitions
- Each subsection must have distinct content - NO repetition across subsections
"""

# ~143 tokens — 58% SMALLER than original, 80% smaller than v1 bloat
PHD_WRITER_SYSTEM_PROMPT = """You are a brilliant academic writer contributing a section with inline citations
 for a research paper/publication.

RULES:
1. Every factual claim MUST have [X] citation. No exceptions.
2. NEVER open with "This section discusses..." or "In recent years..." — hook the reader.
3. NEVER start two consecutive sentences the same way. Vary length and structure.
4. Each subsection must have UNIQUE content. No repetition.
5. Hedge claims: "evidence suggests", "data indicate", "appears to" — not "proves" or "shows".
6. Use analogies, rhetorical questions, and concrete examples to make abstract ideas vivid.
7. Vary verbs: reveal, underscore, complicate, challenge, corroborate — not just "shows".
8. Third person throughout. No filler phrases like "It is important to note that"."""


# Creative seeds — injected into write_prompt (ONE per section, ~15 tokens each)
WRITING_STYLE_SEEDS = [
    "Open with a surprising statistic or counterintuitive finding.",
    "Open with a brief clinical scenario that makes the abstract concrete.",
    "Open by identifying a paradox or tension this section will untangle.",
    "Open with a provocative question the section will answer with evidence.",
    "Open by describing what goes wrong when this problem is ignored.",
    "Open with a bold claim, then marshal evidence for and against it.",
]


class WriterAgent(SubAgent):
    """Agent specialized in academic writing with proper citations - Enhanced with PhD-level prompts."""

    def __init__(self, med_gemma_caller: Callable, verbose: bool = True, debug:bool=False ):
        # system_prompt = PHD_WRITER_SYSTEM_PROMPT
        # gen_config = GenerationConfig.for_writing() # temp=0.8

        gen_config = GenerationConfig(
            temperature=0.92,
            max_new_tokens=4096,
        )

        super().__init__(
            name="WriterAgent",
            registry=writer_registry,
            med_gemma_caller=med_gemma_caller,
            system_prompt=PHD_WRITER_SYSTEM_PROMPT,
            verbose=verbose,
            gen_config=gen_config
        )
        self.debug = debug

        # Use higher temperature for writing (FIX #1)
        self.gen_config = gen_config # GenerationConfig.for_writing()

        # Initialize fact filter (FIX #2)
        self.fact_filter = FactFilter(FactFilterConfig(
            allow_unverified=True,
            min_confidence=0.2   # reduce from 0.6
        ))

        # Initialize output cleaner (FIX #6)
        self.output_cleaner = OutputCleaner(verbose=False, aggressive=False) # Safe Mode

    async def execute_old(self, prompt: str) -> str:
        """Direct LLM call for writing — bypasses SubAgent's generic wrapper."""
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        response = self.med_gemma(
            messages=messages,
            max_new_tokens=self.gen_config.max_new_tokens,
            temperature=self.gen_config.temperature,
        )
        return self._clean_response(response)

    async def write_section(
        self,
        section: SectionOutline,
        facts: List[Dict],
        citations: List[Citation],
        allow_unverified: bool = True
    ) -> str:
        """Write a section with proper academic formatting and all fixes applied."""

        self._log(f"Writing: {section.title}")

        # FIX #2: Filter facts with configurable settings
        self.fact_filter.config.allow_unverified = allow_unverified
        # Fix V2b
        self.fact_filter.config.require_citation = False  # Don't filter out facts without citation_ids


        # Convert facts to dict format if needed
        facts_as_dicts = []
        for f in facts:
            if isinstance(f, dict):
                facts_as_dicts.append(f)
            else:
                facts_as_dicts.append({
                    'fact': getattr(f, 'content', str(f)),
                    'confidence': getattr(f, 'confidence', 0.7),
                    'verified': getattr(f, 'verified', False),
                    'citation_ids': getattr(f, 'citation_ids', []),
                    'section': getattr(f, 'section', section.section_id),
                    #'source': {'url': ''} # comment out in version 2b
                })

        # Filter facts: V2B (broken — filters on a key that doesn't exist yet)
        # filtered_facts = self.fact_filter.filter_facts(facts_as_dicts, section.section_id)
        # Filter facts: V2B.1 (correct — skip section filtering, facts are already section-scoped)
        filtered_facts = self.fact_filter.filter_facts(facts_as_dicts)
        # ====================== V2B FIXES ==================================================
        # V2b fix: If section filter returns nothing, use all facts (section_id might not match)
        if not filtered_facts and facts_as_dicts:
            self._log(" [WriterAgent] All facts filtered out by confidence/citation rules. Using unfiltered.")
            filtered_facts = facts_as_dicts

        verified_count = len([f for f in filtered_facts if f.get('verified')])
        unverified_count = len(filtered_facts) - verified_count
        self._log(f"  Using {len(filtered_facts)} facts ({verified_count} verified, {unverified_count} unverified)")

        # Build citation lookup
        citation_by_id = {c.id: c for c in citations}

        # Build citation reference for prompt
        citation_ref = []
        for c in citations[:15]:
            citation_ref.append(f"[{c.id}] {c.title[:80]}... (URL: {c.url[:50]})")

        # New: Build facts with Explicit citation mapping
        facts_ref = []
        for f in filtered_facts[:25]:
            citation_ids = f.get('citation_ids', [])
            if citation_ids:
                # Use the mapped citation IDs
                cite_str = ", ".join(str(cid) for cid in citation_ids)
            else:
                # No citation - mark as needing hedging
                cite_str = "No Citation - use hedging language"

            verified_mark = "✓" if f.get('verified') else "○"
            confidence = f.get('confidence', 0)
            fact_text = f.get('fact', f.get('content', ''))[:150]
            facts_ref.append(f"{verified_mark} CITE AS [{cite_str}]: {fact_text} (confidence: {confidence:.0%})")

        # Creative Writing Prompt Hack
        creative_seed = random.choice(WRITING_STYLE_SEEDS)

        # Build writing prompt with EXPLICIT citation instructions
        write_prompt_older_long = f"""{PHD_WRITER_SYSTEM_PROMPT} Write an academic research section. Start DIRECTLY with the content.

SECTION: {section.title}
DESCRIPTION: {section.description}
SUBSECTIONS: {', '.join(section.subsections)}
TARGET: {section.target_word_count} words

AVAILABLE CITATIONS:
{chr(10).join(citation_ref) if citation_ref else "None — use hedging language."}

FACTS WITH CITATION MAPPING((✓=verified, ○=unverified)):
Each fact below shows which citation number(s) to use. Follow this mapping EXACTLY.
{chr(10).join(facts_ref) if facts_ref else "No specific facts - write general overview."}

CREATIVE DIRECTION: {creative_seed}


CITATION RULES:
1. When using a fact, cite it with the EXACT citation number shown above
2. Format: EVERY factual claim must have a vancouver citation map to its fact number provided above.
3. vancouver citation examples: to cit single source is writen for example [1]. but for multiple source use a list of numbers  as such [4,5,7].
4. It is good practice to put the citation number in square brackets after the sentence/statement containing factual claim before a full-stop
4. If a fact says "NO CITATION", use hedging ("may", "suggests", "appears to")
5. Do NOT invent citation numbers - only use [1] through [{len(citations)}]

FORMAT:
- Start with: ## {section.title}
- Use ### for each subsection
- Include [X] citations matching the mapping above
- Write {section.target_word_count} words minimum
- Do not repeat/re-write any given subsection again and again. Just end if finished with all sections.

BEGIN YOUR RESPONSE WITH "## {section.title}" - NO other preamble:"""

        # new write prompt
        write_prompt = f"""{PHD_WRITER_SYSTEM_PROMPT}

Write section "## {section.title}" about: {section.description}
Subsections: {', '.join(section.subsections)}
Target: {section.target_word_count} words

FACTS — use the [X] citation shown for each:
{chr(10).join(facts_ref) if facts_ref else "No facts — use hedging."}

CITE EVERY FACT using [X] from above. Example: "AI improves accuracy [1]."

FORMAT:
- Start with: ## {section.title}
- Use ### for each subsection
- Provide citations to improve factual rigour
- Write {section.target_word_count} words minimum
- Do not repeat/re-write any given subsection again and again. Just end if finished with all sections.

BEGIN YOUR RESPONSE WITH "## {section.title}" - NO other preamble:"""


        # Execute
        response = await self.execute(write_prompt)
        content = deduplicate_content(response, verbose=self.verbose)

        # Clean section
        content = self._clean_section(content, section.title)
        # FIX #6: Clean output for template leakage
        content = self.output_cleaner.clean(content, section.title)

        # Check for template leakage
        leakage = self.output_cleaner.detect_leakage(content)
        if leakage:
            self._log(f"  ⚠️ Detected {len(leakage)} potential leakage issues")


        word_count = len(content.split())
        self._log(f"  Written {word_count} words")

        return content
        # =================================================================================
        ############################# Original version 2a ###########################################
        """
        # Build citation reference (ID -> fact mapping)
        citation_ref = []
        for c in citations[:15]:
            citation_ref.append(f"[{c.id}] {c.title[:60]}...")

        # Build facts with citation IDs
        facts_ref = []
        for i, f in enumerate(filtered_facts[:20]):
            source_url = f.get("source", {}).get("url", "")
            cite_id = "?"
            for c in citations:
                if c.url == source_url:
                    cite_id = str(c.id)
                    break
            verified_mark = "✓" if f.get('verified') else "○"
            confidence = f.get('confidence', 0)
            facts_ref.append(f"{verified_mark} [{cite_id}] (conf:{confidence:.0%}) {f.get('fact', f.get('content', ''))[:100]}")

        write_prompt = fWrite an academic research section following strict formatting guidelines.

## SECTION DETAILS
Title: {section.title}
Description: {section.description}
Subsections to Cover: {', '.join(section.subsections)}
Target Length: {section.target_word_count} words

## AVAILABLE CITATIONS
{chr(10).join(citation_ref) if citation_ref else "No citations available - write based on general knowledge with appropriate hedging."}

## FACTS TO INCORPORATE (✓=verified, ○=unverified with confidence)
{chr(10).join(facts_ref) if facts_ref else "No specific facts - write general overview with hedging language."}

## FORMATTING REQUIREMENTS
1. Start with "## {section.title}" heading
2. Use "### Subsection" for subsections
3. EVERY factual claim must have a vancouver citation
4. Write {section.target_word_count} words minimum
5. Use sophisticated sentence variety (simple, compound, complex, compound-complex)
6. Include cohesive devices: furthermore, however, consequently, nevertheless, etc.
7. Apply academic hedging: "suggests", "indicates", "may", "appears to"
8. Each subsection MUST have distinct content - NO repetition

Write the complete section now with proper academic style:

        # FIX #1: Use configured temperature
        response = await self.execute(write_prompt)

        # Clean response
        content = self._clean_section(response, section.title)

        # FIX #6: Clean output for template leakage
        content = self.output_cleaner.clean(content, section.title)

        # Check for template leakage
        leakage = self.output_cleaner.detect_leakage(content)
        if leakage:
            self._log(f"  ⚠️ Detected {len(leakage)} potential leakage issues")

        word_count = len(content.split())
        self._log(f"  Written {word_count} words")

        return content"""

    def _clean_section_old(self, text: str, expected_title: str) -> str:
        """Clean section output."""
        # Remove any preamble before the section heading
        lines = text.split('\n')
        start_idx = 0

        for i, line in enumerate(lines):
            if line.strip().startswith('## '):
                start_idx = i
                break

        text = '\n'.join(lines[start_idx:])

        # Ensure it starts with ## heading
        if not text.strip().startswith('## '):
            text = f"## {expected_title}\n\n{text}"

        # Fix heading levels
        while '####' in text:
            text = text.replace('####', '###')

        return text.strip()

    def _clean_section(self, text: str, expected_title: str) -> str:
        """Safer cleaning that preserves content."""
        lines = text.split('\n')

        # Find where real content starts
        content_start = 0
        for i, line in enumerate(lines):
            stripped = line.strip()
            # Found heading = content starts here
            if stripped.startswith('## ') or stripped.startswith('### '):
                content_start = i
                break
            # Found substantial paragraph (not meta-commentary)
            elif len(stripped) > 100 and not any(skip in stripped.lower() for skip in
                ['i will', 'let me', 'here is', "i'll write", 'the following', 'as requested']):
                content_start = i
                break

        text = '\n'.join(lines[content_start:])

        # Add heading if missing
        if not text.strip().startswith('## ') and not text.strip().startswith('### '):
            text = f"## {expected_title}\n\n{text}"

        # Fix #### -> ###
        while '####' in text:
            text = text.replace('####', '###')

        return text.strip()


print("✅ WriterAgent enhanced with PhD-level prompts (Fix #3) and integrated fixes #1, #2, #6")


✅ WriterAgent enhanced with PhD-level prompts (Fix #3) and integrated fixes #1, #2, #6


In [34]:
class VerifierAgent(SubAgent):
    """Agent specialized in fact verification."""

    def __init__(self, med_gemma_caller: Callable, verbose: bool = True):
        system_prompt = """You are a Research Verifier Agent. Your role is to:
1. Verify facts against multiple sources
2. Check for contradictions
3. Assess claim confidence
4. Flag uncertain or problematic content

Verification standards:
- Facts should be confirmed by at least 2 reliable sources
- Statistical claims need primary source verification
- Medical claims should align with current guidelines
- Flag any controversial or contested claims"""

        super().__init__(
            name="VerifierAgent",
            registry=verifier_registry,
            med_gemma_caller=med_gemma_caller,
            system_prompt=system_prompt,
            verbose=verbose
        )

        self.tavily = tavily_search

    async def verify_fact(self, fact: str, original_source: str) -> Dict:
        """
        Verify a fact by cross-referencing with other sources.

        Returns verification result with confidence score.
        """
        # Search for verification
        verification_query = f"verify: {fact}"

        results = self.tavily.search_for_research(verification_query, num_sources=3)

        if not results:
            return {
                "verified": False,
                "confidence": 0.3,
                "reason": "Could not find additional sources",
                "sources": 1
            }

        # Ask LLM to compare
        verify_prompt = f"""Verify this fact by comparing with the search results.

FACT TO VERIFY: {fact}
ORIGINAL SOURCE: {original_source}

ADDITIONAL SOURCES FOUND:
{json.dumps([{"title": r["title"], "content": r["content"][:500]} for r in results], indent=2)}

Analyze:
1. Do the additional sources support this fact?
2. Are there any contradictions?
3. What is your confidence level (0-1)?

Output JSON:
{{"verified": true/false, "confidence": 0.0-1.0, "reason": "...", "supporting_sources": 0}}

JSON:"""

        response = await self.execute(verify_prompt)

        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start >= 0 and end > start:
                return json.loads(response[start:end])
        except:
            pass

        return {
            "verified": True,
            "confidence": 0.6,
            "reason": "Could not fully verify",
            "sources": len(results) + 1
        }

    async def verify_section(self, content: str, citations: List[Citation]) -> Dict:
        """Verify an entire section's claims."""

        verify_prompt = f"""Review this section for accuracy and flag any issues.

SECTION CONTENT:
{content[:3000]}

CITATIONS USED:
{json.dumps([{"id": c.id, "title": c.title} for c in citations], indent=2)}

Check for:
1. Unsupported claims (statements without citations)
2. Potential inaccuracies
3. Contradictions
4. Missing context or nuance

Output JSON:
{{
    "overall_quality": "good/fair/poor",
    "issues": [
        {{"type": "unsupported_claim", "text": "...", "severity": "low/medium/high"}},
        ...
    ],
    "suggestions": ["..."]
}}

JSON:"""

        response = await self.execute(verify_prompt)

        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            if start >= 0 and end > start:
                return json.loads(response[start:end])
        except:
            pass

        return {"overall_quality": "fair", "issues": [], "suggestions": []}


In [35]:
# =============================================================================
# FIXED SEARCH AGENT - GENERATES CONCISE QUERIES
# =============================================================================

class SearchAgent(SubAgent):
    """Agent specialized in web research - generates concise search queries."""

    def __init__(self, med_gemma_caller: Callable, search_tool, verbose: bool = True):
        system_prompt = """You are a Research Search Agent. Your role is to:
1. Generate SHORT, effective search queries (3-6 words each)
2. Find high-quality academic sources
3. Extract key facts and statistics

IMPORTANT: Keep search queries SHORT and simple like Google searches.
BAD: "Review of Artificial Intelligence and Machine Learning definitions and applications in medical diagnosis"
GOOD: "AI medical diagnosis accuracy"

Always use concise, keyword-focused queries."""

        super().__init__(
            name="SearchAgent",
            registry=search_registry,
            med_gemma_caller=med_gemma_caller,
            system_prompt=system_prompt,
            verbose=verbose
        )

        self.search_tool = search_tool

    async def research_topic(self, topic: str, context: str = "", num_queries: int = 3) -> List[Dict]:
        """Research a topic using multiple concise search queries."""
        self._log(f"Researching: {topic}")

        # Generate SHORT search queries
        query_prompt = f"""Generate {num_queries} SHORT search queries (3-6 words each) for researching:

TOPIC: {topic}
CONTEXT: {context}

Rules:
- Each query must be 3-6 words ONLY
- Use keywords, not full sentences
- No quotation marks or special characters

Examples of GOOD queries:
- "AI radiology accuracy studies"
- "deep learning cancer detection"
- "machine learning diagnosis 2024"

Output as JSON array of short strings:
["short query 1", "short query 2", "short query 3"]

JSON:"""

        queries_response = await self.execute(query_prompt)

        # Parse queries
        queries = []
        try:
            start = queries_response.find('[')
            end = queries_response.rfind(']') + 1
            if start >= 0 and end > start:
                parsed_queries = json.loads(queries_response[start:end])
                # Validate and clean each query
                for q in parsed_queries:
                    if isinstance(q, str):
                        # Ensure query is short
                        words = q.split()[:6]  # Max 6 words
                        clean_query = ' '.join(words)
                        if len(clean_query) > 5:  # Minimum viable query
                            queries.append(clean_query)
        except:
            pass

        # Fallback: generate simple queries from topic
        if not queries:
            # Extract key terms from topic
            topic_words = [w for w in topic.split() if len(w) > 3][:4]
            base_query = ' '.join(topic_words)

            queries = [
                base_query,
                f"{base_query} research",
                f"{base_query} studies 2024"
            ]

        self._log(f"Using {len(queries)} queries: {queries}")

        # Execute searches
        all_sources = []
        seen_urls = set()

        for query in queries[:num_queries]:
            self._log(f"  Query: {query}")

            sources = self.search_tool.search_for_research(query, context, num_sources=4)

            for source in sources:
                url = source.get("url", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    all_sources.append(source)

            # Record search
            if memory.current_state:
                memory.add_search_record(query, len(sources), topic)

        self._log(f"Found {len(all_sources)} unique sources")

        return all_sources

    async def extract_facts(self, sources: List[Dict], section_topic: str) -> List[Dict]:
        """Extract citable facts from sources."""

        if not sources:
            self._log("No sources to extract from")
            return []

        facts = []

        for source in sources:
            content = source.get("content", "")
            if not content:
                continue

            extract_prompt = f"""Extract 3-5 specific, citable facts from this source.

SOURCE: {source.get('title', 'Unknown')}
CONTENT: {content[:1500]}

Output JSON array of facts:
[
    {{"fact": "specific fact with numbers/data", "confidence": 0.9}},
    {{"fact": "another specific fact", "confidence": 0.85}}
]

JSON:"""

            response = await self.execute(extract_prompt)

            try:
                start = response.find('[')
                end = response.rfind(']') + 1
                if start >= 0 and end > start:
                    extracted = json.loads(response[start:end])
                    for item in extracted:
                        if isinstance(item, dict) and "fact" in item:
                            item["source"] = source
                            facts.append(item)
            except json.JSONDecodeError:
                # Fallback: create basic fact from content
                facts.append({
                    "fact": f"Research indicates developments in {section_topic}",
                    "confidence": 0.6,
                    "source": source
                })

        self._log(f"Extracted {len(facts)} facts")
        return facts


print("✅ SearchAgent with concise query generation")

✅ SearchAgent with concise query generation


In [36]:
# =============================================================================
# TEST FIXED SEARCH AGENT
# =============================================================================

async def test_fixed_search():
    """Test that SearchAgent now generates working queries."""

    print("="*70)
    print("TEST: Fixed SearchAgent Query Generation")
    print("="*70)

    searcher = SearchAgent(run_medgemma, tavily_search, verbose=True)

    test_topics = [
        "Introduction: The Intersection of AI and Medical Diagnosis",
        "Foundational Concepts in AI for Healthcare",
        "Applications Across Medical Specialties"
    ]

    for topic in test_topics:
        print(f"\n--- Topic: {topic[:50]}... ---")

        sources = await searcher.research_topic(topic, context="healthcare AI", num_queries=2)

        if sources:
            print(f"✅ Found {len(sources)} sources")
            for s in sources[:2]:
                print(f"   - {s['title'][:50]}...")
        else:
            print(f"❌ No sources found")

    print("\n" + "="*70)
    print("Test complete!")


await test_fixed_search()

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST: Fixed SearchAgent Query Generation

--- Topic: Introduction: The Intersection of AI and Medical D... ---
[SearchAgent] Researching: Introduction: The Intersection of AI and Medical Diagnosis
[SearchAgent] Task: Generate 2 SHORT search queries (3-6 words each) for researching:

TOPIC: Introd...
[SearchAgent] Response: ```json
[
  "AI medical diagnosis applications",
  "AI diagnosis accuracy studies"
]
```...
[SearchAgent] Using 2 queries: ['AI medical diagnosis applications', 'AI diagnosis accuracy studies']
[SearchAgent]   Query: AI medical diagnosis applications
[Tavily] Research query: AI medical diagnosis applications healthcare AI...
[Tavily] Searching: AI medical diagnosis applications healthcare AI...
[Tavily] ✓ Found 8 results in 2.9s
[Tavily] Returning 4 sources
[SearchAgent]   Query: AI diagnosis accuracy studies
[Tavily] Research query: AI diagnosis accuracy studies healthcare AI...
[Tavily] Searching: AI diagnosis accuracy studies healthcare AI...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Tavily] ✓ Found 8 results in 4.1s
[Tavily] Returning 4 sources
[SearchAgent] Found 8 unique sources
✅ Found 8 sources
   - AI in Healthcare: Enhancing Patient Care and Diagn...
   - Artificial Intelligence for Medical Diagnostics—Ex...

--- Topic: Foundational Concepts in AI for Healthcare... ---
[SearchAgent] Researching: Foundational Concepts in AI for Healthcare
[SearchAgent] Task: Generate 2 SHORT search queries (3-6 words each) for researching:

TOPIC: Founda...
[SearchAgent] Response: ```json
[
  "AI healthcare concepts",
  "Machine learning healthcare diagnosis"
]
```...
[SearchAgent] Using 2 queries: ['AI healthcare concepts', 'Machine learning healthcare diagnosis']
[SearchAgent]   Query: AI healthcare concepts
[Tavily] Research query: AI healthcare concepts healthcare AI...
[Tavily] Searching: AI healthcare concepts healthcare AI...
[Tavily] ✓ Found 8 results in 2.8s
[Tavily] Returning 4 sources
[SearchAgent]   Query: Machine learning healthcare diagnosis
[Tavily] Research q

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Tavily] ✓ Found 8 results in 2.3s
[Tavily] Returning 4 sources
[SearchAgent] Found 7 unique sources
✅ Found 7 sources
   - How is AI being Used in the Healthcare Industry...
   - AI, Health, and Health Care Today and Tomorrow - J...

--- Topic: Applications Across Medical Specialties... ---
[SearchAgent] Researching: Applications Across Medical Specialties
[SearchAgent] Task: Generate 2 SHORT search queries (3-6 words each) for researching:

TOPIC: Applic...
[SearchAgent] Response: ```json
[
  "AI healthcare applications",
  "AI medical specialties review"
]
```...
[SearchAgent] Using 2 queries: ['AI healthcare applications', 'AI medical specialties review']
[SearchAgent]   Query: AI healthcare applications
[Tavily] Research query: AI healthcare applications healthcare AI...
[Tavily] Searching: AI healthcare applications healthcare AI...
[Tavily] ✓ Found 8 results in 2.3s
[Tavily] Returning 4 sources
[SearchAgent]   Query: AI medical specialties review
[Tavily] Research query: AI medi

In [37]:
# =============================================================================
# TAVILY INTEGRATION TEST
# =============================================================================
from IPython.display import display, Markdown

async def test_full_pipeline():
    """Test the full research pipeline with real API."""

    print("="*70)
    print("INTEGRATION TEST: Full Research Pipeline")
    print("="*70)

    # Step 1: Test search
    print("\n[1/4] Testing search...")
    sources = tavily_search.search_for_research(
        "artificial intelligence medical diagnosis",
        topic_context="healthcare AI",
        num_sources=3
    )

    if not sources:
        print("❌ Search failed - no sources returned")
        return False

    print(f"✓ Found {len(sources)} sources:")
    for s in sources:
        print(f"  - {s['title'][:50]}... (reliability: {s['reliability']})")

    # Step 2: Test citation creation
    print("\n[2/4] Testing citation creation...")
    # test_project = memory.create_project("Integration Test") # image vs section Folder mismatch bug
    test_memory = PersistentMemory(base_dir="./research_projects_test")
    test_project = test_memory.create_project("Integration Test")

    for source in sources:
        citation = memory.add_citation(
            title=source["title"],
            url=source["url"],
            authors="",
            date=source["published_date"],
            snippet=source["content"][:200],
            reliability_score=source["reliability"]
        )
        print(f"  ✓ Citation [{citation.id}]: {citation.title[:40]}...")

    # Step 3: Test fact extraction
    print("\n[3/4] Testing fact extraction...")
    searcher = SearchAgent(run_medgemma, tavily_search, verbose=False)
    facts = await searcher.extract_facts(sources, "AI in medical diagnosis")

    print(f"✓ Extracted {len(facts)} facts:")
    for f in facts[:3]:
        print(f"  - {f['fact'][:60]}... (confidence: {f.get('confidence', 'N/A')})")

    # Step 4: Test writing
    print("\n[4/4] Testing section writing...")
    writer = WriterAgent(run_medgemma, verbose=False)

    test_section = SectionOutline(
        section_id="test_01",
        title="Test Section: AI in Diagnosis",
        description="Brief test of writing capabilities",
        subsections=["Introduction", "Key Points"],
        target_word_count=500
    )

    citations = [memory.get_citation_by_id(i) for i in range(1, len(sources)+1)]
    citations = [c for c in citations if c]  # Filter None

    content = await writer.write_section(test_section, facts, citations)

    word_count = len(content.split())
    print(f"✓ Written {word_count} words")
    print(f"  Preview: {content[:200]}...")
    print("="*70)
    display(Markdown(content))

    # Check for citations in content
    citation_count = content.count('[')
    print(f"  Citations found: ~{citation_count}")

    print("\n" + "="*70)
    print("✅ INTEGRATION TEST PASSED")
    print("="*70)

    return True


# Run integration test
await test_full_pipeline()

INTEGRATION TEST: Full Research Pipeline

[1/4] Testing search...
[Tavily] Research query: artificial intelligence medical diagnosis healthcare AI...
[Tavily] Searching: artificial intelligence medical diagnosis healthcare AI...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Tavily] ✓ Found 6 results in 1.4s
[Tavily] Returning 3 sources
✓ Found 3 sources:
  - Artificial Intelligence for Medical Diagnostics—Ex... (reliability: 0.9)
  - Artificial Intelligence in Medical Diagnostics: Re... (reliability: 0.4)
  - 7 ways AI is transforming healthcare... (reliability: 0.4)

[2/4] Testing citation creation...
✅ Created project: bb18a6171ccc
   Output directory: research_projects_test/bb18a6171ccc
  ✓ Citation [1]: Artificial Intelligence for Medical Diag...
  ✓ Citation [2]: Artificial Intelligence in Medical Diagn...
  ✓ Citation [3]: 7 ways AI is transforming healthcare...

[3/4] Testing fact extraction...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Extracted 15 facts:
  - GAI projects like OpenAI's DeepQA, IBM's Watson, and Google'... (confidence: 0.95)
  - The goal of GAI for medical diagnostics is to improve the ac... (confidence: 0.9)
  - AI algorithms analyze vast amounts of medical data to identi... (confidence: 0.85)

[4/4] Testing section writing...
✓ Written 584 words
  Preview: ## Test Section: AI in Diagnosis

### Introduction

The intricate landscape of medical diagnosis often presents formidable challenges, particularly when confronting the vast and often elusive realm of...


## Test Section: AI in Diagnosis

### Introduction

The intricate landscape of medical diagnosis often presents formidable challenges, particularly when confronting the vast and often elusive realm of rare diseases. For decades, clinicians have navigated complex pathways, frequently encountering diagnostic delays that can significantly impact patient outcomes. This narrative underscores the persistent need for innovative tools capable of augmenting human expertise. Emerging from the confluence of artificial intelligence and machine learning, AI systems are beginning to reshape this landscape, offering potential solutions to some of medicine's most persistent problems. This section provides a brief exploration of AI's burgeoning role in diagnostic processes, highlighting key capabilities and their implications. The integration of these technologies promises to revolutionize how we identify and manage health conditions, potentially ushering in an era of more precise, efficient, and accessible healthcare.

### Key Points

The capabilities of AI in medical diagnosis are multifaceted, revealing significant potential across various domains. One notable advancement involves the detection of subtle anomalies indicative of disease onset. Evidence suggests that certain AI machine learning models possess the capacity to identify the presence of specific diseases even before patients experience noticeable symptoms. This predictive power could represent a paradigm shift, enabling earlier intervention. Furthermore, the diagnostic odyssey associated with rare diseases appears to be shortening. Data indicate that the time required to reach a diagnosis for these conditions has been substantially reduced, moving from lengthy years to more manageable months. This acceleration is crucial for patient well-being and timely treatment initiation.

Beyond clinical symptom detection, AI demonstrates remarkable proficiency in interpreting complex medical imaging. For instance, an AI tool has been shown to successfully detect 64% of epilepsy brain lesions that were previously overlooked by human radiologists. This capability highlights AI's potential to enhance the accuracy of visual diagnostics, potentially reducing misdiagnoses and improving patient care. The development of foundational AI models like AlphaFold, which predicts 3D protein structures with high fidelity, also underpins advancements in understanding disease mechanisms and developing targeted therapies. The training data for these sophisticated systems is often extensive, with AI tools like the one mentioned being trained on MRI scans of over 1,100 adults and children globally, ensuring broad applicability.

The impact of AI extends to the broader ecosystem of medical diagnostics. GAI projects, including those developed by major tech companies like OpenAI, IBM, and Google, are actively employed in medical diagnostics, demonstrating the practical application of these technologies. AI-driven genomics has also emerged as a powerful tool, particularly in the challenging field of diagnosing rare diseases, offering new avenues for identification. The overall impact of these AI systems appears to be substantial, as they are widely credited with improving diagnostic accuracy, significantly speeding up the diagnostic process, and potentially reducing the associated costs. The potential for AI to improve these metrics is a central theme in ongoing research and development.

Moreover, AI's influence is not confined to static imaging or genomic analysis; it demonstrates adaptability in real-time clinical decision support. An AI model deployed in Yorkshire, for example, successfully predicted the need for hospital transfer in 80% of patients, highlighting its utility in complex, dynamic clinical scenarios. This real-time predictive capability could empower clinicians, providing valuable insights to guide immediate care decisions. The integration of these diverse AI applications underscores the technology's versatility and its potential to become an indispensable partner in the diagnostic process. The ongoing development and refinement of these systems promise further enhancements, potentially leading to even more sophisticated and widely accessible diagnostic tools in the future.

  Citations found: ~0

✅ INTEGRATION TEST PASSED


True

In [ ]:
# =============================================================================
# TAVILY INTEGRATION TEST — FIXED WITH CITATION ENRICHMENT
# =============================================================================
from IPython.display import display, Markdown

async def test_full_pipeline_v2():
    """Test the full research pipeline with real API."""

    print("=" * 70)
    print("INTEGRATION TEST: Full Research Pipeline")
    print("=" * 70)

    # ─────────────────────────────────────────────────────────────────────
    # Step 1: Search
    # ─────────────────────────────────────────────────────────────────────
    print("\n[1/5] Testing search...")
    sources = tavily_search.search_for_research(
        "artificial intelligence medical diagnosis",
        topic_context="healthcare AI",
        num_sources=3
    )

    if not sources:
        print("❌ Search failed - no sources returned")
        return False

    print(f"✓ Found {len(sources)} sources:")
    for s in sources:
        print(f"  - {s['title'][:50]}... (reliability: {s['reliability']})")

    # ─────────────────────────────────────────────────────────────────────
    # Step 2: Create citations (isolated test memory)
    # ─────────────────────────────────────────────────────────────────────
    print("\n[2/5] Testing citation creation...")
    test_memory = PersistentMemory(base_dir="./research_projects_test")
    test_project = test_memory.create_project("Integration Test")

    citations = []
    for source in sources:
        citation = test_memory.add_citation(
            title=source["title"],
            url=source["url"],
            authors="",
            date=source.get("published_date", ""),
            snippet=source["content"][:200],
            reliability_score=source.get("reliability", 0.5)
        )
        citations.append(citation)
        print(f"  ✓ Citation [{citation.id}]: {citation.title[:40]}...")

    # ─────────────────────────────────────────────────────────────────────
    # Step 3: Extract facts
    # ─────────────────────────────────────────────────────────────────────
    print("\n[3/5] Testing fact extraction...")
    searcher = SearchAgent(run_medgemma, tavily_search, verbose=False)
    facts = await searcher.extract_facts(sources, "AI in medical diagnosis")

    print(f"✓ Extracted {len(facts)} facts:")
    for f in facts[:3]:
        print(f"  - {f['fact'][:60]}... (confidence: {f.get('confidence', 'N/A')})")

    # ─────────────────────────────────────────────────────────────────────
    # Step 4: Enrich facts with citation_ids (THE FIX)
    # ─────────────────────────────────────────────────────────────────────
    print("\n[4/5] Enriching facts with citation IDs...")
    enriched = 0
    for fact in facts:
        source_url = fact.get("source", {}).get("url", "")
        matching = [c for c in citations if c.url == source_url]
        fact["citation_ids"] = [c.id for c in matching]
        if fact["citation_ids"]:
            enriched += 1

    print(f"✓ Enriched {enriched}/{len(facts)} facts with citation IDs")

    # Show the mapping
    for f in facts[:5]:
        cite_ids = f.get("citation_ids", [])
        status = f"→ [{', '.join(str(i) for i in cite_ids)}]" if cite_ids else "→ ⚠️ no match"
        print(f"  {status} {f['fact'][:60]}...")

    # ─────────────────────────────────────────────────────────────────────
    # Step 5: Write section
    # ─────────────────────────────────────────────────────────────────────
    print("\n[5/5] Testing section writing...")
    writer = WriterAgent(run_medgemma, verbose=True)

    test_section = SectionOutline(
        section_id="test_01",
        title="Test Section: AI in Diagnosis",
        description="Brief test of writing capabilities",
        subsections=["Introduction", "Key Points"],
        target_word_count=500
    )

    content = await writer.write_section(test_section, facts, citations)

    # ─────────────────────────────────────────────────────────────────────
    # Display results
    # ─────────────────────────────────────────────────────────────────────
    word_count = len(content.split())

    print(f"\n{'=' * 70}")
    print(f"📝 WRITTEN CONTENT ({word_count} words)")
    print(f"{'=' * 70}")
    display(Markdown(content))

    # Citation count in output
    import re
    inline_citations = re.findall(r'\[(\d+)\]', content)
    unique_citations = set(inline_citations)
    print(f"\n{'=' * 70}")
    print(f"📚 CITATION ANALYSIS")
    print(f"{'=' * 70}")
    print(f"  Inline citations found: {len(inline_citations)} total, {len(unique_citations)} unique")
    if unique_citations:
        print(f"  Citation IDs used: [{', '.join(sorted(unique_citations))}]")
    else:
        print(f"  ⚠️ NO inline citations found in output!")

    # Display references
    print(f"\n{'=' * 70}")
    print(f"📖 REFERENCES")
    print(f"{'=' * 70}")
    for c in citations:
        used = "✓" if str(c.id) in unique_citations else "✗"
        author_part = f"{c.authors}. " if c.authors and c.authors.strip() else ""
        print(f"  [{used}] [{c.id}] {author_part}{c.title}")
        print(f"        URL: {c.url[:70]}...")
        print(f"        Reliability: {c.reliability_score}")
        print()

    print(f"{'=' * 70}")
    print(f"✅ INTEGRATION TEST {'PASSED' if unique_citations else 'FAILED — no citations in output'}")
    print(f"{'=' * 70}")

    # Cleanup
    import shutil
    test_dir = Path("./research_projects_test")
    if test_dir.exists():
        shutil.rmtree(test_dir)
        print("🧹 Test artifacts cleaned up")

    return bool(unique_citations)


# Run integration test
await test_full_pipeline_v2()

INTEGRATION TEST: Full Research Pipeline

[1/5] Testing search...
[Tavily] Research query: artificial intelligence medical diagnosis healthcare AI...
[Tavily] Searching: artificial intelligence medical diagnosis healthcare AI...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Tavily] ✓ Found 6 results in 1.9s
[Tavily] Returning 3 sources
✓ Found 3 sources:
  - Artificial Intelligence for Medical Diagnostics—Ex... (reliability: 0.9)
  - Transforming diagnosis through artificial intellig... (reliability: 0.9)
  - Artificial Intelligence (AI) in Healthcare & Medic... (reliability: 0.4)

[2/5] Testing citation creation...
✅ Created project: 13438460099b
   Output directory: research_projects_test/13438460099b
  ✓ Citation [1]: Artificial Intelligence for Medical Diag...
  ✓ Citation [2]: Transforming diagnosis through artificia...
  ✓ Citation [3]: Artificial Intelligence (AI) in Healthca...

[3/5] Testing fact extraction...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingf

✓ Extracted 17 facts:
  - AI algorithms can analyze medical images (e.g., X-rays, MRIs... (confidence: 0.95)
  - AI can analyze large amounts of patient data, including medi... (confidence: 0.9)
  - The goal of GAI or general AI, which is being used by differ... (confidence: 0.85)

[4/5] Enriching facts with citation IDs...
✓ Enriched 17/17 facts with citation IDs
  → [1] AI algorithms can analyze medical images (e.g., X-rays, MRIs...
  → [1] AI can analyze large amounts of patient data, including medi...
  → [1] The goal of GAI or general AI, which is being used by differ...
  → [1] The goal of GAI for medical diagnostics is to improve the ac...
  → [1] By using AI algorithms to analyze vast amounts of medical da...

[5/5] Testing section writing...
[WriterAgent] Writing: Test Section: AI in Diagnosis
[WriterAgent]   Using 12 facts (0 verified, 12 unverified)
[WriterAgent] Task: You are a brilliant academic writer contributing a section with inline citations...
[WriterAgent] Response:

## Test Section: AI in Diagnosis

### Introduction

The integration of artificial intelligence into healthcare represents a paradigm shift, poised to fundamentally alter stroke care processes [2]. This technological advancement, driven by sophisticated machine learning algorithms, significantly reshaped healthcare by enhancing medical diagnosis and treatment [3]. The hyperacute stroke context, characterized by its high stakes, underscores the critical need for precise diagnostic capabilities, making AI-induced changes on the diagnostic process and its outcomes warrant urgent further investigation [2]. While AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2], the potential impact necessitates careful consideration and validation. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. Furthermore, the emergence of generative AI presents a new frontier, potentially revolutionizing aspects of medical practice [3]. AI-driven algorithms are designed to gain insights from clinical data to assist rapid, accurate diagnostic problem-solving and treatment decision-making [2].

### Key Points

Augmented diagnostic models are proving particularly useful in hyperacute stroke [2]. These systems leverage AI to analyze medical images, including X-rays, MRIs, ultrasounds, CT scans, and DXAs, assisting healthcare providers in identifying and diagnosing conditions more effectively [1]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2]. The development of AI algorithms for medical image analysis has been a significant area of research and application [1]. Generative AI is emerging as a powerful force in healthcare, offering potential solutions for various challenges [3]. The integration of AI into stroke care processes is likely to fundamentally impact stroke care processes [2]. The potential impact necessitates careful consideration and validation [2]. The field of AI in healthcare is not nascent; foundational concepts have been explored for over half a century [3]. The ability of machine learning processes to analyze vast amounts of clinical data is central to these advancements [3]. This capacity allows AI to identify subtle patterns and predict medical outcomes with increasing accuracy [3]. The application of AI in healthcare extends beyond image analysis; AI-driven algorithms are increasingly used in healthcare settings to support clinicians with diagnosis, treatment, and patient outcome prediction [2].


📚 CITATION ANALYSIS
  Inline citations found: 188 total, 3 unique
  Citation IDs used: [1, 2, 3]

📖 REFERENCES
  [✓] [1] Artificial Intelligence for Medical Diagnostics—Existing and Future AI ...
        URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC9955430/...
        Reliability: 0.9

  [✓] [2] Transforming diagnosis through artificial intelligence - Nature
        URL: https://www.nature.com/articles/s41746-025-01460-1...
        Reliability: 0.9

  [✓] [3] Artificial Intelligence (AI) in Healthcare & Medical Field
        URL: https://www.foreseemed.com/artificial-intelligence-in-healthcare...
        Reliability: 0.4

✅ INTEGRATION TEST PASSED
🧹 Test artifacts cleaned up


True

In [38]:
# Vanceuver citation extractor regex
import re
from typing import List

BRACKETED_CITATION_RE = re.compile(
    r'\[\s*([0-9]+(?:\s*(?:,|[-\u2013\u2014])\s*[0-9]+)*)\s*\]'
)

def parse_citation_group(group: str) -> List[int]:
    """
    Parse a single bracket content like "1, 3-5,7" -> [1,3,4,5,7]
    """
    items = re.split(r'\s*,\s*', group)
    result = []
    for it in items:
        # normalize different dash characters to hyphen
        it_norm = re.sub(r'[\u2013\u2014]', '-', it.strip())
        if '-' in it_norm:
            parts = it_norm.split('-', 1)
            try:
                start = int(parts[0])
                end = int(parts[1])
            except ValueError:
                continue  # skip malformed parts
            if end >= start:
                result.extend(range(start, end + 1))
            else:
                # handle reversed ranges conservatively (swap)
                result.extend(range(end, start + 1))
        else:
            try:
                result.append(int(it_norm))
            except ValueError:
                continue
    return result

def extract_vancouver_citations(text: str, dedupe: bool = True) -> List[int]:
    """
    Extract all Vancouver citations from text in document order.
    Returns a flat list of integers. If dedupe=True, preserves first-seen order.
    """
    seen = set()
    out = []
    for m in BRACKETED_CITATION_RE.finditer(text):
        group = m.group(1)
        nums = parse_citation_group(group)
        for n in nums:
            if dedupe:
                if n not in seen:
                    seen.add(n)
                    out.append(n)
            else:
                out.append(n)
    return out

text = "As shown previously [1], and later studies [2-4, 6], the effect is clear. See also [10–12]."
print(extract_vancouver_citations(text))
# -> [1, 2, 3, 4, 6, 10, 11, 12]

text2 = "Multiple lists: [1,2,3] and single [4]."
print(extract_vancouver_citations(text2))
# -> [1, 2, 3, 4]

[1, 2, 3, 4, 6, 10, 11, 12]
[1, 2, 3, 4]


##### **Cell 6B: Fixed Research Paper Formatter**

In [39]:
# =============================================================================
# RESEARCH PAPER FORMATTER - ENHANCED WITH LLM ABSTRACT GENERATION
# =============================================================================

class ResearchPaperFormatter:
    """
    Formats research output into proper academic paper structure.
    Converts raw sections into publication-ready markdown.

    Enhancements:
    - LLM-generated abstract (200-300 words) instead of deterministic patterns
    - Page breaks between Abstract/Keywords and Table of Contents
    - Proper figure numbering without duplicate labels
    """

    def __init__(self, memory: PersistentMemory, med_gemma_caller: Callable = None):
        self.memory = memory
        self.med_gemma = med_gemma_caller  # LLM caller for abstract generation
        self.figure_counter = 0

    def format_final_document(self) -> str:
        """
        Create properly formatted research paper.
        Returns path to formatted document.
        """
        if not self.memory.current_state:
            raise ValueError("No active project")

        state = self.memory.current_state
        output_dir = Path(state.output_dir)
        sections_dir = output_dir / "sections"

        # Validate sections exist
        section_files = sorted(sections_dir.glob("*.md"))
        if not section_files:
            raise FileNotFoundError(
                f"No section files found in {sections_dir}. "
                f"Sections may have been written to a different project folder. "
                f"Check research_projects/ for the correct folder."
            )

        print(f"📄 Formatting {len(section_files)} sections from: {sections_dir}")

        # Reset figure counter
        self.figure_counter = 0

        # =================================================================
        # BUILD DOCUMENT
        # =================================================================

        doc_lines = []

        # -----------------------------------------------------------------
        # TITLE PAGE
        # -----------------------------------------------------------------
        doc_lines.extend([
            f"# {state.topic}",
            "",
            "---",
            "",
            "**Author:** Gemma DeepResearch Agent (MedGemma-1.5-4B)",
            "",
            f"**Date:** {datetime.now().strftime('%B %d, %Y')}",
            "",
            "---",
            "",
        ])

        # -----------------------------------------------------------------
        # ABSTRACT PAGE (200-300 words using LLM)
        # -----------------------------------------------------------------
        doc_lines.append("## Abstract")
        doc_lines.append("")

        # Generate abstract using LLM
        abstract = self._generate_abstract_with_llm(state, sections_dir)
        doc_lines.append(abstract)
        doc_lines.append("")

        # Keywords
        keywords = self._extract_keywords(state.topic)
        doc_lines.append(f"**Keywords:** {', '.join(keywords)}")
        doc_lines.append("")

        # PAGE BREAK after Abstract and Keywords
        doc_lines.append("")
        doc_lines.append('<div style="page-break-after: always;"></div>')
        doc_lines.append("")
        doc_lines.append("---")
        doc_lines.append("")

        # -----------------------------------------------------------------
        # TABLE OF CONTENTS (New Page)
        # -----------------------------------------------------------------
        doc_lines.append("## Table of Contents")
        doc_lines.append("")

        if state.outline:
            for i, section in enumerate(state.outline.sections, 1):
                anchor = self._make_anchor(section.title)
                doc_lines.append(f"{i}. [{section.title}](#{anchor})")

        doc_lines.append(f"{len(state.outline.sections) + 1 if state.outline else 1}. [References](#references)")
        doc_lines.append("")

        # PAGE BREAK after Table of Contents
        doc_lines.append("")
        doc_lines.append('<div style="page-break-after: always;"></div>')
        doc_lines.append("")
        doc_lines.append("---")
        doc_lines.append("")

        # -----------------------------------------------------------------
        # MAIN CONTENT (Sections)
        # -----------------------------------------------------------------
        section_files = sorted(sections_dir.glob("*.md"))

        for section_file in section_files:
            formatted_section = self._format_section(section_file)
            doc_lines.append(formatted_section)
            doc_lines.append("")
            doc_lines.append("---")
            doc_lines.append("")

        # -----------------------------------------------------------------
        # REFERENCES (Single consolidated list)
        # -----------------------------------------------------------------
        doc_lines.append("## References")
        doc_lines.append("")

        for citation in sorted(state.citations, key=lambda c: c.id):
            author_part = f"{citation.authors}. " if citation.authors and citation.authors.strip() else ""
            doc_lines.append(f"[{citation.id}] {author_part}" +
                   f"*{citation.title}*. " +
                   f"Available from: {citation.url}. " +
                   f"Accessed: {citation.accessed_date}.")
            doc_lines.append("")

        # -----------------------------------------------------------------
        # WRITE FINAL DOCUMENT
        # -----------------------------------------------------------------
        final_content = "\n".join(doc_lines)
        final_path = output_dir / "RESEARCH_PAPER.md"

        with open(final_path, 'w', encoding='utf-8') as f:
            f.write(final_content)

        # Calculate statistics
        word_count = len(final_content.split())
        pages = word_count / 500

        print(f"\n{'='*70}")
        print("✅ RESEARCH PAPER FORMATTED")
        print(f"{'='*70}")
        print(f"📄 Output: {final_path}")
        print(f"📊 Words: {word_count:,} (~{pages:.1f} pages)")
        print(f"📚 Citations: {len(state.citations)}")
        print(f"🖼️  Figures: {self.figure_counter}")

        return str(final_path)

    def _generate_abstract_with_llm(self, state: ResearchState, sections_dir: Path) -> str:
        """
        Generate a 200-300 word abstract using the LLM.
        Takes section summaries as input for comprehensive abstract generation.
        """
        # Gather section summaries
        section_summaries = []
        section_files = sorted(sections_dir.glob("*.md"))

        for section_file in section_files[:10]:  # First 10 sections for context
            with open(section_file, 'r', encoding='utf-8') as f:
                content = f.read()

            # Extract title and first paragraph as summary
            lines = content.split('\n')
            title = ""
            first_para = ""

            for line in lines:
                if line.startswith('# '):
                    title = line.replace('# ', '').strip()
                elif line.strip() and not line.startswith('#') and not line.startswith('!') and not line.startswith('*Figure'):
                    first_para = line.strip()[:300]
                    break

            if title and first_para:
                section_summaries.append(f"- {title}: {first_para}")

        # Get thesis if available
        thesis = state.outline.thesis if state.outline else ""

        # If we have LLM caller, use it for abstract generation
        if self.med_gemma:
            try:
                abstract_prompt = f"""Write a comprehensive academic abstract (200-300 words) for a research paper.

RESEARCH TOPIC: {state.topic}

THESIS: {thesis if thesis else "Not specified - infer from sections below."}

SECTION SUMMARIES:
{chr(10).join(section_summaries) if section_summaries else "No section summaries available."}

TOTAL CITATIONS: {len(state.citations)}
TOTAL VERIFIED FACTS: {len([f for f in state.facts if f.verified])}

REQUIREMENTS:
1. Write exactly 200-300 words
2. Include: Background, Objectives, Methods overview, Key findings, Conclusions
3. Use formal academic tone
4. Do NOT use first person
5. Include specific numbers/statistics if available in the sections
6. End with implications or future directions

Write the abstract now (no heading, just the text):"""

                # Call MedGemma for abstract generation
                messages = [{"role": "user", "content": [{"type": "text", "text": abstract_prompt}]}]

                response = self.med_gemma(
                    messages=messages,
                    max_new_tokens=500,
                    temperature=0.5  # Lower temperature for coherent abstract
                )

                # Clean response
                abstract = response.strip()

                # Remove any thinking blocks
                while "<unused94>" in abstract:
                    start = abstract.find("<unused94>")
                    end = abstract.find("<unused95>")
                    if end > start:
                        abstract = abstract[:start] + abstract[end + len("<unused95>"):]
                    else:
                        abstract = abstract[:start]

                # Remove any preamble
                if "Abstract:" in abstract:
                    abstract = abstract.split("Abstract:")[-1].strip()
                if "abstract:" in abstract.lower():
                    abstract = abstract.split("abstract:", 1)[-1].strip()

                # Ensure reasonable length
                words = abstract.split()
                if len(words) > 350:
                    abstract = " ".join(words[:300]) + "..."
                elif len(words) < 100:
                    # Fallback to deterministic if LLM output is too short
                    abstract = self._generate_abstract_deterministic(state)

                return abstract

            except Exception as e:
                print(f"[Formatter] ⚠️ LLM abstract generation failed: {e}, using fallback")
                return self._generate_abstract_deterministic(state)
        else:
            return self._generate_abstract_deterministic(state)

    def _generate_abstract_deterministic(self, state: ResearchState) -> str:
        """Fallback deterministic abstract generation (legacy method)."""
        # Start with thesis
        thesis = state.outline.thesis if state.outline else ""

        # Add scope from sections
        if state.outline and state.outline.sections:
            section_titles = [s.title for s in state.outline.sections[:5]]
            scope = f"This comprehensive review examines: {', '.join(section_titles)}."
        else:
            scope = ""

        # Add key statistics from facts
        key_facts = [f.content for f in state.facts if f.confidence > 0.8][:3]
        findings = " ".join(key_facts) if key_facts else ""

        # Add conclusion hint
        topic_short = state.topic.split(':')[0] if ':' in state.topic else state.topic[:50]
        conclusion = f"This research synthesizes current evidence on {topic_short} and identifies key areas for future investigation and clinical application."

        # Combine into structured abstract
        abstract_parts = [thesis, scope, findings, conclusion]
        abstract = " ".join([p for p in abstract_parts if p])

        # Ensure reasonable length
        words = abstract.split()
        if len(words) > 300:
            abstract = " ".join(words[:280]) + "..."
        elif len(words) < 150:
            # Pad with generic academic language
            abstract += f" The findings presented herein contribute to the growing body of knowledge in this domain and offer practical implications for researchers and practitioners alike."

        return abstract

    def _extract_keywords_deterministic(self, topic: str) -> List[str]:
        """Extract keywords from topic."""
        # Common stopwords
        stopwords = {'a', 'an', 'the', 'in', 'on', 'of', 'for', 'to', 'and', 'or', 'with', 'by', 'recent', 'advances', 'comprehensive', 'review'}

        # Extract meaningful words
        words = topic.lower().replace(':', '').replace('-', ' ').split()
        keywords = [w.capitalize() for w in words if w not in stopwords and len(w) > 3]

        # Add standard medical AI keywords
        standard_keywords = ["Machine Learning", "Healthcare", "Clinical Decision Support"]

        # Combine and deduplicate
        all_keywords = keywords[:4] + [k for k in standard_keywords if k.lower() not in topic.lower()]

        return all_keywords[:6]

    def _extract_keywords(self, topic: str) -> List[str]:
        """Extract keywords using LLM."""
        if self.med_gemma:
            try:
                prompt = f"""Extract 5-8 academic keywords for this research topic.

TOPIC: {topic}

Return ONLY a JSON array of strings. No other text.
Example: ["Artificial Intelligence", "Medical Imaging", "Deep Learning"]

JSON:"""

                messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
                response = self.med_gemma(
                    messages=messages,
                    max_new_tokens=1024,
                    temperature=0.1,
                )

                # Clean thinking blocks
                cleaned = response.strip()
                while "<unused94>" in cleaned:
                    start = cleaned.find("<unused94>")
                    end = cleaned.find("<unused95>")
                    if end > start:
                        cleaned = cleaned[:start] + cleaned[end + len("<unused95>"):]
                    else:
                        cleaned = cleaned[:start]

                # Parse JSON array
                start = cleaned.find('[')
                end = cleaned.rfind(']') + 1
                if start >= 0 and end > start:
                    keywords = json.loads(cleaned[start:end])
                    if isinstance(keywords, list) and len(keywords) >= 3:
                        return [str(k).strip() for k in keywords[:8]]

            except Exception as e:
                print(f"[Keywords] LLM extraction failed: {e}")

        # Fallback: deterministic
        stopwords = {'a', 'an', 'the', 'in', 'on', 'of', 'for', 'to', 'and', 'or', 'with', 'by', 'recent', 'advances', 'comprehensive', 'review'}
        words = topic.lower().replace(':', '').replace('-', ' ').split()
        keywords = [w.capitalize() for w in words if w not in stopwords and len(w) > 3]
        return keywords[:6]

    def _make_anchor(self, title: str) -> str:
        """Create markdown anchor from title."""
        return title.lower().replace(' ', '-').replace(':', '').replace(',', '')

    def _format_section(self, section_file: Path) -> str:
        """Format a single section properly - no duplicate figure labels."""

        with open(section_file, 'r', encoding='utf-8') as f:
            content = f.read()

        lines = content.split('\n')
        formatted_lines = []
        skip_next_figure_caption = False

        for line in lines:
            # Fix heading hierarchy (ensure ## for sections, ### for subsections)
            if line.startswith('# ') and not line.startswith('## '):
                line = '#' + line  # Convert # to ##

            # Format figures with proper numbering
            if line.startswith('!['):
                self.figure_counter += 1
                # Extract alt text
                alt_match = line[2:line.find(']')]
                img_path = line[line.find('(')+1:line.find(')')]

                formatted_lines.append(f"![Figure {self.figure_counter}]({img_path})")
                formatted_lines.append("")
                formatted_lines.append(f"*Figure {self.figure_counter}. {alt_match}*")
                skip_next_figure_caption = True  # Skip the next line if it's a duplicate caption
                continue

            # Skip duplicate figure captions (FIX #5)
            if skip_next_figure_caption:
                if line.startswith('*Figure:') or line.startswith('*Figure '):
                    skip_next_figure_caption = False
                    continue
                skip_next_figure_caption = False

            # Remove per-section references (we consolidate at end)
            if line.startswith('### Section References') or (line.startswith('- [') and '] ' in line and 'Available from:' in line):
                continue

            formatted_lines.append(line)

        # Clean up extra blank lines
        cleaned = '\n'.join(formatted_lines)
        while '\n\n\n' in cleaned:
            cleaned = cleaned.replace('\n\n\n', '\n\n')

        return cleaned.strip()


# Create formatter (will be recreated with med_gemma_caller in orchestrator)
paper_formatter = ResearchPaperFormatter(memory)

print("✅ ResearchPaperFormatter enhanced with LLM abstract generation and page breaks")


✅ ResearchPaperFormatter enhanced with LLM abstract generation and page breaks


##### **Cell 7: Main Deep Research Orchestrator**

In [40]:
# =============================================================================
# DEEP RESEARCH ORCHESTRATOR
# =============================================================================

class DeepResearchOrchestrator:
    """
    Main orchestrator for deep research.
    Coordinates all sub-agents to produce comprehensive research documents.
    """

    def __init__(
        self,
        med_gemma_caller: Callable,
        memory: PersistentMemory,
        file_manager: FileManager,
        verbose: bool = True
    ):
        self.med_gemma = med_gemma_caller
        self.memory = memory
        self.file_manager = file_manager
        self.verbose = verbose

        # Initialize sub-agents
        self.planner = PlannerAgent(med_gemma_caller, verbose)
        self.searcher = SearchAgent(med_gemma_caller, verbose)
        self.writer = WriterAgent(med_gemma_caller, verbose)
        self.verifier = VerifierAgent(med_gemma_caller, verbose)

        self._log("Deep Research Orchestrator initialized")

    def _log(self, msg):
        if self.verbose:
            print(f"[Orchestrator] {msg}")

    async def research(self, topic: str, target_pages: int = 80) -> str:
        """
        Conduct comprehensive research on a topic.

        Args:
            topic: Research topic
            target_pages: Target document length in pages

        Returns:
            Path to final document
        """
        self._log(f"Starting research: {topic}")
        self._log(f"Target: {target_pages} pages")

        # =====================================================================
        # PHASE 1: PROJECT INITIALIZATION
        # =====================================================================
        self._log("\n" + "="*70)
        self._log("PHASE 1: PROJECT INITIALIZATION")
        self._log("="*70)

        state = self.memory.create_project(topic)
        self.memory.set_status(ResearchStatus.PLANNING)

        # =====================================================================
        # PHASE 2: PLANNING
        # =====================================================================
        self._log("\n" + "="*70)
        self._log("PHASE 2: PLANNING")
        self._log("="*70)

        outline = await self.planner.create_outline(topic, target_pages)
        self.memory.set_outline(outline)

        self._log(f"Outline created: {len(outline.sections)} sections")
        for section in outline.sections:
            self._log(f"  - {section.title} ({section.target_word_count} words)")

        # =====================================================================
        # PHASE 3: RESEARCH AND WRITING (Section by Section)
        # =====================================================================
        self._log("\n" + "="*70)
        self._log("PHASE 3: RESEARCH AND WRITING")
        self._log("="*70)

        self.memory.set_status(ResearchStatus.RESEARCHING)

        for i, section in enumerate(outline.sections):
            self._log(f"\n--- Section {i+1}/{len(outline.sections)}: {section.title} ---")

            # Skip if already complete (for resume capability)
            if section.status == "complete":
                self._log("  Already complete, skipping")
                continue

            # Update current section
            if self.memory.current_state:
                self.memory.current_state.current_section = section.section_id

            await self._process_section(section)

        # =====================================================================
        # PHASE 4: VERIFICATION
        # =====================================================================
        self._log("\n" + "="*70)
        self._log("PHASE 4: VERIFICATION")
        self._log("="*70)

        self.memory.set_status(ResearchStatus.VERIFYING)

        await self._verify_document()

        # =====================================================================
        # PHASE 5: FINAL ASSEMBLY
        # =====================================================================
        self._log("\n" + "="*70)
        self._log("PHASE 5: FINAL ASSEMBLY")
        self._log("="*70)

        final_path = self.file_manager.assemble_final_document()

        self.memory.set_status(ResearchStatus.COMPLETE)

        self._log(f"\n✅ RESEARCH COMPLETE!")
        self._log(f"Output: {final_path}")

        return final_path

    async def _process_section(self, section: SectionOutline):
        """Process a single section: research, write, verify."""

        # STEP 1: Research the topic
        self._log(f"  [1/4] Researching...")

        sources = await self.searcher.research_topic(
            section.title,
            context=section.description,
            num_queries=3
        )

        # Add sources as citations
        section_citations = []
        source_to_citation = {}  # NEW_v2b: Map source URL to citation for fast lookup

        for source in sources:
            citation = self.memory.add_citation(
                title=source["title"],
                url=source["url"],
                authors="",  # Would need additional parsing
                date=source.get("published_date", ""),
                snippet=source.get("content", "")[:500],
                reliability_score=source.get("reliability", 0.5)
            )
            section_citations.append(citation)
            source_to_citation[source["url"]] = citation  # NEW_v2b: Store mapping

        self._log(f"  [1/4] Added {len(section_citations)} citations")

        # STEP 2: Extract facts
        self._log(f"  [2/4] Extracting facts...")

        facts = await self.searcher.extract_facts(sources, section.title)
        # ===================== version v_2b references/citation fixes ===========
        # NEW: Enrich facts with citation_ids BEFORE storing and passing to writer
        enriched_facts = []
        for fact_data in facts:
            source_url = fact_data.get("source", {}).get("url", "")
            # Find matching citation
            citation = source_to_citation.get(source_url)

            if citation:
                citation_ids = [citation.id]  # NEW: Use the citation ID
            else:
                # Fallback: try partial URL match
                citation_ids = []
                for url, cit in source_to_citation.items():
                    # match by domain or partial path
                    if source_url and (source_url in url or url in source_url):
                        citation_ids = [cit.id]
                        break

            # Create enriched fact
            enriched_fact = {
            'fact': fact_data.get('fact', ''),
            'content': fact_data.get('fact', ''),  # Alias for compatibility
            'confidence': fact_data.get('confidence', 0.7),
            'verified': False,
            'citation_ids': citation_ids,
            'section': section.section_id,
            'source': fact_data.get('source', {})
            }
            enriched_facts.append(enriched_fact)

            # Store to memory
            self.memory.add_fact(
                content=enriched_fact['fact'],
                citation_ids=citation_ids,
                confidence=enriched_fact['confidence'],
                section=section.section_id
            )

        self._log(f"  [2/4b] Extracted {len(enriched_facts)} facts, {len([f for f in enriched_facts if f['citation_ids']])} with citations")
        ########################### end of v_2b ##################################
        """ # v_2a original code
        # Store facts
        for fact_data in facts:
            matching_citations = [c for c in section_citations if c.url == fact_data["source"]["url"]]
            citation_ids = [c.id for c in matching_citations]

            self.memory.add_fact(
                content=fact_data["fact"],
                citation_ids=citation_ids,
                confidence=fact_data.get("confidence", 0.7),
                section=section.section_id
            )

        self._log(f"  [2/4] Extracted {len(facts)} facts")
        """

        # STEP 3: Write section - use fact in v2a; use enriched_facts in v2b
        self._log(f"  [3/4] Writing...")

        self.memory.set_status(ResearchStatus.WRITING)
        # in second argument, change to enriched_facts instead of facts used in vanilla 2a
        content = await self.writer.write_section(section, enriched_facts, section_citations)

        # STEP 4: Quick verification
        self._log(f"  [4/4] Quick verification...")

        verification = await self.verifier.verify_section(content, section_citations)

        if verification.get("overall_quality") == "poor":
            self.memory.add_human_review_note(
                section.section_id,
                f"Quality flagged as poor: {verification.get('issues', [])}",
                "review"
            )

        # Check for images
        image_path = None
        if image_generator:
            image_path = image_generator.generate_for_section(
                section.title,
                content,
                self.memory.current_state.output_dir if self.memory.current_state else "."
            )

        # Save section
        citation_ids = [c.id for c in section_citations]
        file_path = self.file_manager.save_section(
            section.section_id,
            section.title,
            content,
            citation_ids,
            image_path
        )

        self._log(f"  ✓ Section complete: {file_path}")

    async def _verify_document(self):
        """Verify the complete document."""

        if not self.memory.current_state:
            return

        # Sample some facts for verification
        all_facts = self.memory.current_state.facts
        sample_size = min(10, len(all_facts))

        if sample_size == 0:
            self._log("No facts to verify")
            return

        import random
        sample_facts = random.sample(all_facts, sample_size)

        verified_count = 0

        for fact in sample_facts:
            self._log(f"  Verifying: {fact.content[:50]}...")

            # Get original source
            original_source = ""
            if fact.citation_ids:
                citation = self.memory.get_citation_by_id(fact.citation_ids[0])
                if citation:
                    original_source = citation.url

            result = await self.verifier.verify_fact(fact.content, original_source)

            if result.get("verified"):
                fact.verified = True
                verified_count += 1
            else:
                self.memory.add_human_review_note(
                    fact.section,
                    f"Unverified fact: {fact.content[:100]}",
                    "verify"
                )

        self._log(f"  Verified {verified_count}/{sample_size} sampled facts")

    async def resume(self, project_id: str) -> str:
        """Resume an incomplete project."""

        state = self.memory.load_project(project_id)

        if not state:
            raise ValueError(f"Project not found: {project_id}")

        if state.status == ResearchStatus.COMPLETE:
            self._log("Project already complete")
            return self.file_manager.assemble_final_document()

        self._log(f"Resuming project: {project_id}")
        self._log(f"Status: {state.status.value}")
        self._log(f"Completed sections: {len(state.completed_sections)}")

        # Continue from where we left off
        if state.outline:
            pending = self.memory.get_pending_sections()
            self._log(f"Pending sections: {len(pending)}")

            for section in pending:
                await self._process_section(section)

        # Finish up
        await self._verify_document()

        return self.file_manager.assemble_final_document()


print("✅ DeepResearchOrchestrator defined")

✅ DeepResearchOrchestrator defined


In [41]:
# Recreate the orchestrator with updated SearchAgent
class DeepResearchOrchestratorV2(DeepResearchOrchestrator):
    """Updated orchestrator with proper search tool injection."""

    def __init__(
        self,
        med_gemma_caller: Callable,
        memory: PersistentMemory,
        file_manager: FileManager,
        search_tool,
        verbose: bool = True
    ):
        self.med_gemma = med_gemma_caller
        self.memory = memory
        self.file_manager = file_manager
        self.verbose = verbose

        # Initialize sub-agents with search tool
        self.planner = PlannerAgent(med_gemma_caller, verbose)
        self.searcher = SearchAgent(med_gemma_caller, search_tool, verbose)
        self.writer = WriterAgent(med_gemma_caller, verbose)
        self.verifier = VerifierAgent(med_gemma_caller, verbose)

        self._log("Deep Research Orchestrator V2 initialized")

"""
# Create updated orchestrator
orchestrator = DeepResearchOrchestratorV2(
    med_gemma_caller=run_medgemma,
    memory=memory,
    file_manager=file_manager,
    search_tool=tavily_search,
    verbose=True
)
"""
print("✅ Deep Research Orchestrator V2 created with proper search tool")

✅ Deep Research Orchestrator V2 created with proper search tool


In [42]:
# =============================================================================
# DEEP RESEARCH ORCHESTRATOR V3 - WITH ALL FIXES INTEGRATED
# =============================================================================

# FIX #5: Contextual Image Generator
class ContextualImageGenerator:
    """
    Enhanced image generator that uses full section content for context.
    """

    def __init__(self, gemini_generator, verbose: bool = True):
        self.generator = gemini_generator
        self.verbose = verbose

    def _log(self, msg: str):
        if self.verbose:
            print(f"[ContextualImageGen] {msg}")

    def _extract_key_concepts(self, content: str, max_concepts: int = 5) -> List[str]:
        """Extract key concepts from content for image generation."""
        import re
        capitalized = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', content)
        emphasized = re.findall(r'"([^"]+)"|\*\*([^*]+)\*\*', content)
        emphasized = [e[0] or e[1] for e in emphasized if any(e)]
        technical = re.findall(r'\b(?:AI|ML|CNN|RNN|LSTM|NLP|CT|MRI|FDA|WHO)\b', content)
        concepts = list(dict.fromkeys(capitalized + emphasized + technical))
        return concepts[:max_concepts]

    def _extract_statistics(self, content: str) -> List[str]:
        """Extract statistics and numbers from content."""
        import re
        percentages = re.findall(r'\d+(?:\.\d+)?%', content)
        numbers = re.findall(r'(\d+(?:\.\d+)?)\s*(million|billion|thousand|patients|studies|cases)', content)
        numbers = [f"{n[0]} {n[1]}" for n in numbers]
        return percentages + numbers

    def _determine_image_type(self, content: str, title: str) -> tuple:
        """Determine if image is needed and what type."""
        content_lower = content.lower()
        title_lower = title.lower()
        combined = f"{content_lower} {title_lower}"

        key_concepts = self._extract_key_concepts(content)
        statistics = self._extract_statistics(content)

        if any(w in combined for w in ['process', 'workflow', 'pipeline', 'steps', 'phases', 'cycle']):
            image_type = "flowchart"
            prompt = f"the workflow process for {title}"
            if key_concepts:
                prompt += f" featuring {', '.join(key_concepts[:3])}"
            return True, image_type, prompt

        elif any(w in combined for w in ['anatomy', 'structure', 'organ', 'tissue', 'cell', 'body']):
            image_type = "illustration"
            prompt = f"anatomical structure: {title}"
            if key_concepts:
                prompt += f" showing {', '.join(key_concepts[:3])}"
            return True, image_type, prompt

        elif any(w in combined for w in ['statistics', 'data', 'percentage', 'rate', 'survey', 'prevalence']):
            image_type = "infographic"
            prompt = f"key statistics about {title}"
            if statistics:
                prompt += f" showing data: {', '.join(statistics[:3])}"
            return True, image_type, prompt

        elif any(w in combined for w in ['comparison', 'versus', 'difference', 'compare', 'contrast']):
            image_type = "diagram"
            prompt = f"comparison diagram for {title}"
            return True, image_type, prompt

        elif any(w in combined for w in ['algorithm', 'model', 'neural network', 'architecture', 'framework']):
            image_type = "diagram"
            prompt = f"AI/ML architecture diagram: {title}"
            if key_concepts:
                prompt += f" showing components: {', '.join(key_concepts[:4])}"
            return True, image_type, prompt

        return False, "", ""

    def generate_for_section(
        self,
        section_title: str,
        section_content: str,
        output_dir: str
    ) -> Optional[str]:
        """Generate contextually appropriate image for a section."""
        self._log(f"Analyzing: {section_title}")

        image_needed, image_type, prompt = self._determine_image_type(
            section_content, section_title
        )

        if not image_needed:
            self._log(f"  No image needed for this section")
            return None

        self._log(f"  Type: {image_type}")
        self._log(f"  Prompt: {prompt[:80]}...")

        safe_title = "".join(c for c in section_title if c.isalnum() or c in " -_")[:50]
        image_filename = f"{safe_title.replace(' ', '_')}.png"
        image_path = f"{output_dir}/images/{image_filename}"

        os.makedirs(f"{output_dir}/images", exist_ok=True)
        image_generator_prompt = f"Generate an image infographic for a research paper based on the contents provided here: {section_content}"

        result = self.generator.generate_image(
            prompt=image_generator_prompt,
            image_type="infographic", # image_type,
            style="scientific",
            save_path=image_path
        )

        if result.get("success"):
            self._log(f"  ✓ Generated: {image_filename}")
            return image_path
        else:
            self._log(f"  ✗ Failed: {result.get('error')}")
            return None


class DeepResearchOrchestratorV3(DeepResearchOrchestratorV2):
    """
    Deep Research Orchestrator with full image generation support.

    Enhancements:
    - allow_image_generation parameter to toggle image generation
    - Contextual image generator (FIX #5)
    - Overlap checker for plagiarism detection (FIX #4)
    - Output cleaner for template leak prevention (FIX #6)
    """

    def __init__(
        self,
        med_gemma_caller: Callable,
        memory: PersistentMemory,
        file_manager: FileManager,
        search_tool,
        image_generator: Gemini3ProImageGenerator,
        allow_image_generation: bool = True,  # NEW PARAMETER
        verbose: bool = True
    ):
        super().__init__(
            med_gemma_caller=med_gemma_caller,
            memory=memory,
            file_manager=file_manager,
            search_tool=search_tool,
            verbose=verbose
        )

        self.allow_image_generation = allow_image_generation
        self.image_generator_base = image_generator

        # FIX #5: Use contextual image generator
        self.image_generator = ContextualImageGenerator(
            gemini_generator=image_generator,
            verbose=verbose
        ) if allow_image_generation else None

        # FIX #4: Initialize overlap checker
        self.overlap_checker = OverlapChecker(
            word_overlap_threshold=0.4,
            use_embeddings=True
        )

        # FIX #6: Initialize output cleaner
        self.output_cleaner = OutputCleaner(verbose=False)

        if allow_image_generation:
            self._log("Image generation enabled with Contextual Image Generator")
        else:
            self._log("Image generation DISABLED")

    async def _process_section(self, section: SectionOutline):
        """Process a single section: research, write, verify, optionally generate image."""

        # STEP 1: Research the topic
        self._log(f"  [1/5] Researching...")

        sources = await self.searcher.research_topic(
            section.title,
            context=section.description,
            num_queries=3
        )

        # Add sources as citations
        section_citations = []
        for source in sources:
            citation = self.memory.add_citation(
                title=source["title"],
                url=source["url"],
                authors="",
                date=source.get("published_date", ""),
                snippet=source.get("content", "")[:500],
                reliability_score=source.get("reliability", 0.5)
            )
            section_citations.append(citation)

        self._log(f"  [1/5] Added {len(section_citations)} citations")

        # STEP 2: Extract facts
        self._log(f"  [2/5] Extracting facts...")

        facts = await self.searcher.extract_facts(sources, section.title)

        # Store facts
        for fact_data in facts:
            matching_citations = [c for c in section_citations if c.url == fact_data["source"]["url"]]
            citation_ids = [c.id for c in matching_citations]

            self.memory.add_fact(
                content=fact_data["fact"],
                citation_ids=citation_ids,
                confidence=fact_data.get("confidence", 0.7),
                section=section.section_id
            )
            # ❌ MISSING — writer never sees this
            fact_data["citation_ids"] = citation_ids

        self._log(f"  [2/5] Extracted {len(facts)} facts")

        # STEP 3: Write section
        self._log(f"  [3/5] Writing...")

        self.memory.set_status(ResearchStatus.WRITING)

        content = await self.writer.write_section(section, facts, section_citations)

        # FIX #6: Clean output for template leakage
        content = self.output_cleaner.clean(content, section.title)

        # Check for leakage
        leakage_issues = self.output_cleaner.detect_leakage(content)
        if leakage_issues:
            self._log(f"  ⚠️ Detected {len(leakage_issues)} template leakage issues")

        # FIX #4: Check for subsection overlap
        subsections = self.overlap_checker.extract_subsections(content)
        if len(subsections) > 1:
            overlap_issues = self.overlap_checker.check_all_subsections(subsections)
            if overlap_issues:
                self._log(f"  ⚠️ Overlap detected between subsections:")
                for issue in overlap_issues[:2]:
                    self._log(f"    - {issue['text1_label']} vs {issue['text2_label']}")
                # Flag for human review
                self.memory.add_human_review_note(
                    section.section_id,
                    f"Subsection overlap detected: {overlap_issues[0]['recommendation']}",
                    "review"
                )

        # STEP 4: Generate image (if enabled and relevant)
        self._log(f"  [4/5] Checking for image generation...")

        image_path = None
        if self.allow_image_generation and self.image_generator and self.memory.current_state:
            # FIX #5: Use contextual image generator with full content
            image_path = self.image_generator.generate_for_section(
                section.title,
                content,  # Pass full content for context
                self.memory.current_state.output_dir
            )

        # STEP 5: Verify section
        self._log(f"  [5/5] Verifying...")

        self.memory.set_status(ResearchStatus.VERIFYING)

        verification = await self.verifier.verify_section(content, section_citations)

        if verification.get("issues"):
            for issue in verification["issues"][:3]:
                self.memory.add_human_review_note(
                    section.section_id,
                    f"Verification: {issue}",
                    "verify"
                )

        # Save section to file
        citation_ids = [c.id for c in section_citations]
        self.file_manager.save_section(
            section.section_id,
            section.title,
            content,
            citation_ids,
            image_path
        )

        word_count = len(content.split())
        self._log(f"  ✓ Section complete: {word_count} words")
        if image_path:
            self._log(f"  ✓ Image: {os.path.basename(image_path)}")


print("✅ DeepResearchOrchestratorV3 with allow_image_generation and all fixes integrated")


✅ DeepResearchOrchestratorV3 with allow_image_generation and all fixes integrated


In [43]:
# =============================================================================
# DEEP RESEARCH ORCHESTRATOR V4 - WITH PAPER FORMATTING AND LLM ABSTRACT
# =============================================================================

class DeepResearchOrchestratorV4(DeepResearchOrchestratorV3):
    """
    Deep Research Orchestrator with proper paper formatting.

    Features:
    - LLM-generated abstract (200-300 words)
    - Page breaks between Abstract/Keywords and Table of Contents
    - allow_image_generation toggle
    - All fixes from V3 (overlap checker, output cleaner, contextual images)
    """

    def __init__(
        self,
        med_gemma_caller: Callable,
        memory: PersistentMemory,
        file_manager: FileManager,
        search_tool,
        image_generator,
        paper_formatter: ResearchPaperFormatter,
        allow_image_generation: bool = True,  # Toggle for image generation
        verbose: bool = True
    ):
        super().__init__(
            med_gemma_caller=med_gemma_caller,
            memory=memory,
            file_manager=file_manager,
            search_tool=search_tool,
            image_generator=image_generator,
            allow_image_generation=allow_image_generation,
            verbose=verbose
        )

        # Create paper formatter with LLM caller for abstract generation
        # self.paper_formatter = ResearchPaperFormatter(memory, med_gemma_caller) # bug: image vs section folder mismatch
        if paper_formatter is not None:
            self.paper_formatter = paper_formatter

        else:
            self.paper_formatter = ResearchPaperFormatter(memory, med_gemma_caller)

        self._log("Paper formatting enabled with LLM abstract generation")

        if allow_image_generation:
            self._log("Image generation: ENABLED")
        else:
            self._log("Image generation: DISABLED")

    async def research(self, topic: str, target_pages: int = 80) -> str:
        """
        Conduct research and produce formatted paper.
        """
        # Run base research process
        self._log(f"Starting research: {topic}")
        self._log(f"Target: {target_pages} pages")
        self._log(f"Image generation: {'ENABLED' if self.allow_image_generation else 'DISABLED'}")

        # Call parent's research method for phases 1-4
        await self._run_research_phases(topic, target_pages)

        # Phase 5: Format as proper research paper with LLM abstract
        self._log("\n" + "="*70)
        self._log("PHASE 5: FORMATTING RESEARCH PAPER (with LLM Abstract)")
        self._log("="*70)

        final_path = self.paper_formatter.format_final_document()

        self.memory.set_status(ResearchStatus.COMPLETE)

        return final_path

    async def _run_research_phases(self, topic: str, target_pages: int):
        """Run phases 1-4 of research."""

        # Phase 1: Initialize
        self._log("\n" + "="*70)
        self._log("PHASE 1: PROJECT INITIALIZATION")
        self._log("="*70)

        state = self.memory.create_project(topic)
        self.memory.set_status(ResearchStatus.PLANNING)

        # Phase 2: Planning
        self._log("\n" + "="*70)
        self._log("PHASE 2: PLANNING")
        self._log("="*70)

        outline = await self.planner.create_outline(topic, target_pages)
        self.memory.set_outline(outline)

        self._log(f"Outline: {len(outline.sections)} sections")

        # Phase 3: Research & Write
        self._log("\n" + "="*70)
        self._log("PHASE 3: RESEARCH AND WRITING")
        self._log("="*70)

        self.memory.set_status(ResearchStatus.RESEARCHING)

        for i, section in enumerate(outline.sections):
            self._log(f"\n--- Section {i+1}/{len(outline.sections)}: {section.title} ---")

            if section.status == "complete":
                self._log("  Already complete, skipping")
                continue

            await self._process_section(section)

        # Phase 4: Final Verification
        self._log("\n" + "="*70)
        self._log("PHASE 4: FINAL VERIFICATION")
        self._log("="*70)

        self.memory.set_status(ResearchStatus.VERIFYING)

        # Check for any pending issues
        pending_reviews = [n for n in self.memory.current_state.human_review_notes if not n.get("resolved")]
        if pending_reviews:
            self._log(f"⚠️ {len(pending_reviews)} items pending human review")
            self.memory.set_status(ResearchStatus.NEEDS_REVIEW)
        else:
            self._log("✓ All sections verified")

        # Progress report
        self._log(f"\n{self.file_manager.get_progress_report()}")


print("✅ DeepResearchOrchestratorV4 with LLM abstract and allow_image_generation parameter")


✅ DeepResearchOrchestratorV4 with LLM abstract and allow_image_generation parameter


In [44]:
# =============================================================================
# CREATE FINAL ORCHESTRATOR WITH ALL FIXES
# =============================================================================

# Create paper formatter with LLM caller
paper_formatter = ResearchPaperFormatter(memory, run_medgemma)

# Create final orchestrator with all enhancements
# Set allow_image_generation=True to enable image generation
# Set allow_image_generation=False to disable image generation
orchestrator = DeepResearchOrchestratorV4(
    med_gemma_caller=run_medgemma,
    memory=memory,
    file_manager=file_manager,
    search_tool=tavily_search,
    image_generator=image_generator,
    paper_formatter=paper_formatter,
    allow_image_generation=True,  # Toggle image generation here
    verbose=True
)

# Update writer with enhanced version (includes all fixes)
orchestrator.writer = WriterAgent(run_medgemma, verbose=True)
print("✅ WriterAgent updated with PhD-level prompts and all fixes")

# Update searcher with fixed version
orchestrator.searcher = SearchAgent(run_medgemma, tavily_search, verbose=True)
print("✅ SearchAgent updated with concise query generation")

print("\n" + "="*70)
print("✅ DEEP RESEARCH ORCHESTRATOR V4 READY")
print("="*70)
print("""
FIXES APPLIED:
  ✓ Fix #1: Temperature scaling (writing=0.8, planning=0.3, factual=0.2)
  ✓ Fix #2: Allow unverified facts with confidence threshold
  ✓ Fix #3: PhD-level writing prompts with cohesive devices
  ✓ Fix #4: Overlap/plagiarism detection between subsections
  ✓ Fix #5: Contextual image generation with full content analysis
  ✓ Fix #6: Template leak prevention with 30+ patterns

ENHANCEMENTS:
  ✓ LLM-generated abstract (200-300 words)
  ✓ Page breaks between Abstract and Table of Contents
  ✓ allow_image_generation toggle parameter
  ✓ No duplicate image labels in output
""")

# Sample test topics
TEST_TOPICS = [
    "Recent Advances in Artificial Intelligence for Medical Diagnosis: A Comprehensive Review",
    "Mechanical Ventilation in the ICU: A Literature Review of Frontier Innovations for 2010-2026",
    "CRISPR Gene Editing in Clinical Medicine: Current Applications and Future Directions",
    "The Role of Large Language Models in Healthcare: Opportunities, Challenges, and Ethical Considerations",
    "Immunotherapy in Oncology: Breakthrough Treatments and Emerging Resistance Mechanisms",
    "Digital Health and Remote Patient Monitoring: Transforming Chronic Disease Management",
    "Antibiotic Resistance: Global Trends, Mechanisms, and Novel Therapeutic Strategies",
    "Precision Medicine in Cardiovascular Disease: From Genomics to Clinical Practice",
    "Mental Health in the Post-Pandemic Era: Epidemiology, Treatment Advances, and Healthcare System Response",
    "Regenerative Medicine and Stem Cell Therapy: Current Evidence and Clinical Applications"
]

print(f"\n📚 {len(TEST_TOPICS)} sample research topics available")
for i, topic in enumerate(TEST_TOPICS, 1):
    print(f"   {i}. {topic[:60]}...")


[Orchestrator] Deep Research Orchestrator V2 initialized


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[OverlapChecker] ✓ Sentence embeddings enabled
[Orchestrator] Image generation enabled with Contextual Image Generator
[Orchestrator] Paper formatting enabled with LLM abstract generation
[Orchestrator] Image generation: ENABLED
✅ WriterAgent updated with PhD-level prompts and all fixes
✅ SearchAgent updated with concise query generation

✅ DEEP RESEARCH ORCHESTRATOR V4 READY

FIXES APPLIED:
  ✓ Fix #1: Temperature scaling (writing=0.8, planning=0.3, factual=0.2)
  ✓ Fix #2: Allow unverified facts with confidence threshold
  ✓ Fix #3: PhD-level writing prompts with cohesive devices
  ✓ Fix #4: Overlap/plagiarism detection between subsections
  ✓ Fix #5: Contextual image generation with full content analysis
  ✓ Fix #6: Template leak prevention with 30+ patterns

ENHANCEMENTS:
  ✓ LLM-generated abstract (200-300 words)
  ✓ Page breaks between Abstract and Table of Contents
  ✓ allow_image_generation toggle parameter
  ✓ No duplicate image labels in output


📚 10 sample research topics a

In [45]:
# =============================================================================
# RUN FULL RESEARCH WITH ALL FIXES
# =============================================================================

async def run_deep_research(
    topic_index: int = 0,
    target_pages: int = 80,
    allow_image_generation: bool = True
):
    """
    Run deep research on a selected topic.

    Args:
        topic_index: Index of topic from TEST_TOPICS list
        target_pages: Target number of pages (default 80)
        allow_image_generation: Whether to generate images (default True)
    """

    topic = TEST_TOPICS[topic_index]

    print(f"\n{'#'*70}")
    print(f"# DEEP RESEARCH WITH ALL FIXES")
    print(f"# Topic: {topic[:60]}...")
    print(f"# Target: {target_pages} pages")
    print(f"# Image Generation: {'ENABLED' if allow_image_generation else 'DISABLED'}")
    print(f"{'#'*70}")

    # Recreate orchestrator with specified image generation setting
    global orchestrator
    """
    orchestrator = DeepResearchOrchestratorV4(
        med_gemma_caller=run_medgemma,
        memory=memory,
        file_manager=file_manager,
        search_tool=tavily_search,
        image_generator=image_generator,
        paper_formatter=ResearchPaperFormatter(memory, run_medgemma),
        allow_image_generation=allow_image_generation,
        verbose=True
    )

    # Update sub-agents
    orchestrator.writer = WriterAgent(run_medgemma, verbose=True)
    orchestrator.searcher = SearchAgent(run_medgemma, tavily_search, verbose=True)
    """
    try:
        final_path = await orchestrator.research(topic, target_pages)

        print(f"\n{'='*70}")
        print("✅ RESEARCH COMPLETE")
        print(f"{'='*70}")
        print(f"\nOutput: {final_path}")
        print(f"\n{file_manager.get_progress_report()}")

        return final_path

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

        if memory.current_state:
            memory.log_error(str(e))
            print(f"\nProject saved. Resume with:")
            print(f"  await orchestrator.resume('{memory.current_state.project_id}')")

        return None


print("✅ run_deep_research function ready")
print("""
USAGE:
  # With images (default)
  result = await run_deep_research(topic_index=0, target_pages=20)

  # Without images
  result = await run_deep_research(topic_index=0, target_pages=20, allow_image_generation=False)
""")


✅ run_deep_research function ready

USAGE:
  # With images (default)
  result = await run_deep_research(topic_index=0, target_pages=20)

  # Without images
  result = await run_deep_research(topic_index=0, target_pages=20, allow_image_generation=False)



In [46]:
# Run this BEFORE your long research cell
from google.colab import output
output.eval_js('''
    setInterval(() => {
        document.querySelector("colab-connect-button")?.click();
        console.log("Keep alive: " + new Date().toLocaleTimeString());
    }, 60000);
''')

1

##### **Cell 8: Human Review**

In [48]:
# =============================================================================
# HUMAN REVIEW SYSTEM (PATCHED — regeneration now resets project status)
# =============================================================================

class HumanReviewSystem:
    """
    Comprehensive human review interface for research documents.

    Workflow:
    1. Agent completes research → flags uncertain items automatically
    2. Human reviews flagged items + can add own notes
    3. Human marks items resolved OR requests regeneration
    4. Agent can resume and implement changes

    Review Types:
    - VERIFY: Fact needs verification (auto-flagged by VerifierAgent)
    - REVIEW: Section quality concern (auto-flagged)
    - CORRECTION: Human-added correction needed
    - REGENERATE: Section needs rewriting
    - COMMENT: General feedback (no action required)
    """

    def __init__(self, memory: PersistentMemory):
        self.memory = memory

    def _get_state(self) -> Optional[ResearchState]:
        """Get current state or raise error."""
        if not self.memory.current_state:
            print("❌ No active project. Load one first with: review.load_project('project_id')")
            return None
        return self.memory.current_state

    # =========================================================================
    # PROJECT MANAGEMENT
    # =========================================================================

    def list_projects(self):
        """List all available research projects."""
        projects_dir = Path(self.memory.base_dir)

        if not projects_dir.exists():
            print("No projects found.")
            return

        print("\n" + "="*70)
        print("📁 AVAILABLE RESEARCH PROJECTS")
        print("="*70)

        for project_dir in sorted(projects_dir.iterdir()):
            if project_dir.is_dir():
                state_file = project_dir / "state.json"
                if state_file.exists():
                    with open(state_file) as f:
                        state = json.load(f)

                    status_icon = {
                        "complete": "✅",
                        "needs_review": "⚠️",
                        "researching": "🔄",
                        "writing": "✍️",
                        "planning": "📋",
                        "not_started": "⏸️"
                    }.get(state.get("status", ""), "❓")

                    review_count = len([n for n in state.get("human_review_notes", [])
                                       if not n.get("resolved")])

                    print(f"\n{status_icon} {state['project_id']}")
                    print(f"   Topic: {state['topic'][:60]}...")
                    print(f"   Status: {state['status']}")
                    print(f"   Pending Reviews: {review_count}")
                    print(f"   Updated: {state.get('updated_at', 'Unknown')[:19]}")

    def load_project(self, project_id: str):
        """Load a project for review."""
        state = self.memory.load_project(project_id)
        if state:
            print(f"\n✅ Loaded project: {project_id}")
            self.show_summary()

    # =========================================================================
    # VIEW FUNCTIONS
    # =========================================================================

    def show_summary(self):
        """Show project summary and review status."""
        state = self._get_state()
        if not state:
            return

        print("\n" + "="*70)
        print("📊 PROJECT SUMMARY")
        print("="*70)

        print(f"\n📌 Topic: {state.topic}")
        print(f"📍 Status: {state.status.value}")
        print(f"📁 Output: {state.output_dir}")

        # Progress
        if state.outline:
            total_sections = len(state.outline.sections)
            completed = len(state.completed_sections)
            print(f"\n📈 Progress: {completed}/{total_sections} sections")

            total_words = sum(s.word_count for s in state.outline.sections)
            print(f"📝 Words: {total_words:,}")

        print(f"📚 Citations: {len(state.citations)}")
        print(f"🔍 Facts: {len(state.facts)} ({len([f for f in state.facts if f.verified])} verified)")

        # Review status
        pending = [n for n in state.human_review_notes if not n.get("resolved")]
        resolved = [n for n in state.human_review_notes if n.get("resolved")]

        print(f"\n📋 Reviews: {len(pending)} pending, {len(resolved)} resolved")

        if pending:
            print("\n⚠️  PENDING REVIEWS:")
            for i, note in enumerate(pending[:5], 1):
                print(f"   {i}. [{note['section']}] {note['note'][:50]}...")
            if len(pending) > 5:
                print(f"   ... and {len(pending) - 5} more")

    def show_pending_reviews(self):
        """Show all pending review items in detail."""
        state = self._get_state()
        if not state:
            return

        pending = [n for n in state.human_review_notes if not n.get("resolved")]

        if not pending:
            print("\n✅ No pending reviews!")
            return

        print("\n" + "="*70)
        print(f"⚠️  PENDING REVIEWS ({len(pending)})")
        print("="*70)

        for i, note in enumerate(pending, 1):
            action_icon = {
                "verify": "🔍",
                "review": "📋",
                "correction": "✏️",
                "regenerate": "🔄",
                "comment": "💬"
            }.get(note.get("action", "review"), "❓")

            print(f"\n{action_icon} [{i}] Section: {note['section']}")
            print(f"   Action: {note['action']}")
            print(f"   Note: {note['note']}")
            print(f"   Flagged: {note['timestamp'][:19]}")

        print("\n" + "-"*70)
        print("Commands:")
        print("  review.view_section('section_id')  - View section content")
        print("  review.resolve(1)                  - Mark item #1 as resolved")
        print("  review.resolve(1, 'your notes')    - Resolve with notes")
        print("  review.regenerate('section_id')    - Mark for regeneration")

    def show_sections(self):
        """List all sections with their status."""
        state = self._get_state()
        if not state or not state.outline:
            return

        print("\n" + "="*70)
        print("📑 SECTIONS")
        print("="*70)

        for section in state.outline.sections:
            status_icon = "✅" if section.status == "complete" else "⏳"
            review_count = len([n for n in state.human_review_notes
                               if n['section'] == section.section_id and not n.get('resolved')])

            review_flag = f" ⚠️({review_count})" if review_count > 0 else ""

            print(f"\n{status_icon} {section.section_id}: {section.title}{review_flag}")
            print(f"   Words: {section.word_count}/{section.target_word_count}")
            if section.file_path:
                print(f"   File: {Path(section.file_path).name}")

    def view_section(self, section_id: str):
        """View the content of a specific section."""
        state = self._get_state()
        if not state:
            return

        # Find section file
        sections_dir = Path(state.output_dir) / "sections"

        for file in sections_dir.glob(f"{section_id}*.md"):
            print(f"\n📄 {file.name}")
            print("="*70)

            with open(file, 'r') as f:
                content = f.read()

            print(content)

            # Show any reviews for this section
            reviews = [n for n in state.human_review_notes
                      if n['section'] == section_id and not n.get('resolved')]

            if reviews:
                print("\n" + "-"*70)
                print(f"⚠️ PENDING REVIEWS FOR THIS SECTION ({len(reviews)}):")
                for r in reviews:
                    print(f"   - [{r['action']}] {r['note']}")

            return content

        print(f"❌ Section not found: {section_id}")
        return None

    def view_citations(self, limit: int = 20):
        """View all citations."""
        state = self._get_state()
        if not state:
            return

        print("\n" + "="*70)
        print(f"📚 CITATIONS ({len(state.citations)} total)")
        print("="*70)

        for citation in state.citations[:limit]:
            print(f"\n[{citation.id}] {citation.title[:60]}...")
            print(f"    URL: {citation.url[:60]}...")
            print(f"    Reliability: {citation.reliability_score:.1f}")

        if len(state.citations) > limit:
            print(f"\n... and {len(state.citations) - limit} more")

    def view_facts(self, section_id: str = None, verified_only: bool = False):
        """View facts, optionally filtered."""
        state = self._get_state()
        if not state:
            return

        facts = state.facts

        if section_id:
            facts = [f for f in facts if f.section == section_id]

        if verified_only:
            facts = [f for f in facts if f.verified]

        print(f"\n📋 FACTS ({len(facts)})")
        print("="*70)

        for fact in facts[:20]:
            verified_icon = "✅" if fact.verified else "❓"
            print(f"\n{verified_icon} [{fact.section}] {fact.content[:80]}...")
            print(f"   Confidence: {fact.confidence:.0%} | Citations: {fact.citation_ids}")

        if len(facts) > 20:
            print(f"\n... and {len(facts) - 20} more")

    # =========================================================================
    # ADD REVIEW COMMENTS
    # =========================================================================

    def add_comment(self, section_id: str, comment: str):
        """Add a general comment to a section (no action required)."""
        self.memory.add_human_review_note(
            section=section_id,
            note=comment,
            action="comment"
        )
        print(f"✅ Comment added to {section_id}")

    def add_correction(self, section_id: str, correction: str):
        """Add a correction that needs to be made."""
        self.memory.add_human_review_note(
            section=section_id,
            note=f"CORRECTION NEEDED: {correction}",
            action="correction"
        )
        print(f"✅ Correction flagged for {section_id}")

    def flag_for_verification(self, section_id: str, fact: str):
        """Flag a specific fact for verification."""
        self.memory.add_human_review_note(
            section=section_id,
            note=f"VERIFY: {fact}",
            action="verify"
        )
        print(f"✅ Fact flagged for verification in {section_id}")

    def request_regeneration(self, section_id: str, reason: str = ""):
        """Request that a section be regenerated."""
        state = self._get_state()
        if not state or not state.outline:
            return

        # Find section and mark as pending
        for section in state.outline.sections:
            if section.section_id == section_id:
                section.status = "pending"

                if section_id in state.completed_sections:
                    state.completed_sections.remove(section_id)

                # ═══════════════════════════════════════════════
                # FIX: Reset project status so resume() doesn't
                # short-circuit on "complete"
                # ═══════════════════════════════════════════════
                state.status = ResearchStatus.WRITING

                self.memory.add_human_review_note(
                    section=section_id,
                    note=f"REGENERATE: {reason}" if reason else "REGENERATE: Requested by reviewer",
                    action="regenerate"
                )

                self.memory._save_state()
                print(f"✅ Section {section_id} marked for regeneration")
                print(f"   Status reset to: {state.status.value}")
                print(f"   Run: await orchestrator.resume('{state.project_id}')")
                return

        print(f"❌ Section not found: {section_id}")

    # =========================================================================
    # RESOLVE REVIEWS
    # =========================================================================

    def resolve(self, review_index: int, resolution_notes: str = "Reviewed and approved"):
        """
        Resolve a pending review item.

        Args:
            review_index: 1-based index from show_pending_reviews()
            resolution_notes: Notes about how it was resolved
        """
        state = self._get_state()
        if not state:
            return

        pending = [n for n in state.human_review_notes if not n.get("resolved")]

        if review_index < 1 or review_index > len(pending):
            print(f"❌ Invalid index. Use 1-{len(pending)}")
            return

        # Find the item in original list
        item_to_resolve = pending[review_index - 1]

        for note in state.human_review_notes:
            if note['timestamp'] == item_to_resolve['timestamp']:
                note['resolved'] = True
                note['resolution'] = resolution_notes
                note['resolved_at'] = datetime.now().isoformat()
                note['resolved_by'] = "human_reviewer"
                break

        self.memory._save_state()

        print(f"✅ Resolved: {item_to_resolve['note'][:50]}...")
        print(f"   Resolution: {resolution_notes}")

    def resolve_all(self, resolution_notes: str = "Batch approved"):
        """Resolve all pending reviews at once."""
        state = self._get_state()
        if not state:
            return

        pending = [n for n in state.human_review_notes if not n.get("resolved")]

        if not pending:
            print("No pending reviews to resolve.")
            return

        count = 0
        for note in state.human_review_notes:
            if not note.get("resolved"):
                note['resolved'] = True
                note['resolution'] = resolution_notes
                note['resolved_at'] = datetime.now().isoformat()
                note['resolved_by'] = "human_reviewer"
                count += 1

        self.memory._save_state()
        print(f"✅ Resolved {count} reviews")

    # =========================================================================
    # EXPORT REVIEWS
    # =========================================================================

    def export_review_report(self) -> str:
        """Export all reviews to a markdown file."""
        state = self._get_state()
        if not state:
            return ""

        lines = [
            f"# Review Report: {state.topic}",
            "",
            f"**Project ID:** {state.project_id}",
            f"**Generated:** {datetime.now().isoformat()}",
            "",
            "---",
            "",
            "## Summary",
            "",
        ]

        pending = [n for n in state.human_review_notes if not n.get("resolved")]
        resolved = [n for n in state.human_review_notes if n.get("resolved")]

        lines.append(f"- **Pending Reviews:** {len(pending)}")
        lines.append(f"- **Resolved Reviews:** {len(resolved)}")
        lines.append("")

        if pending:
            lines.append("## Pending Reviews")
            lines.append("")
            for i, note in enumerate(pending, 1):
                lines.append(f"### {i}. [{note['section']}] {note['action'].upper()}")
                lines.append(f"- **Note:** {note['note']}")
                lines.append(f"- **Flagged:** {note['timestamp']}")
                lines.append("")

        if resolved:
            lines.append("## Resolved Reviews")
            lines.append("")
            for i, note in enumerate(resolved, 1):
                lines.append(f"### {i}. [{note['section']}] {note['action'].upper()}")
                lines.append(f"- **Note:** {note['note']}")
                lines.append(f"- **Resolution:** {note.get('resolution', 'N/A')}")
                lines.append(f"- **Resolved:** {note.get('resolved_at', 'N/A')}")
                lines.append("")

        # Save to file
        report_path = Path(state.output_dir) / "REVIEW_REPORT.md"
        content = "\n".join(lines)

        with open(report_path, 'w') as f:
            f.write(content)

        print(f"✅ Review report exported: {report_path}")
        return str(report_path)


# Create review system instance
review = HumanReviewSystem(memory)

print("✅ HumanReviewSystem created")
print("\nQuick Start:")
print("  review.list_projects()           # See all projects")
print("  review.load_project('id')        # Load a project")
print("  review.show_summary()            # Project overview")
print("  review.show_pending_reviews()    # Items needing attention")

✅ HumanReviewSystem created

Quick Start:
  review.list_projects()           # See all projects
  review.load_project('id')        # Load a project
  review.show_summary()            # Project overview
  review.show_pending_reviews()    # Items needing attention


##### **MarkDown to PDF Tool**

In [49]:
# =============================================================================
# PDF CONVERSION - MULTIPLE METHODS
# =============================================================================

# Method 1: Install dependencies for pypandoc
!apt-get update -qq
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic -qq

# Install pypandoc
!pip install pypandoc weasyprint markdown pdfkit -q

print("✅ PDF dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../01-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../02-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../03-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unselect

In [50]:
# =============================================================================
# PDF CONVERTER WITH MULTIPLE FALLBACK METHODS
# =============================================================================

import subprocess
from pathlib import Path

class ResearchPDFConverter:
    """
    Converts research markdown to PDF using multiple methods.
    Falls back through methods if one fails.
    """

    def __init__(self):
        self.methods = [
            ("pandoc", self._convert_with_pandoc),
            ("weasyprint", self._convert_with_weasyprint),
            ("md_to_pdf", self._convert_with_md2pdf),
        ]

    def convert(self, md_path: str, pdf_path: str = None, method: str = "auto") -> str:
        """
        Convert markdown to PDF.

        Args:
            md_path: Path to markdown file
            pdf_path: Output PDF path (optional, auto-generated if not provided)
            method: "auto", "pandoc", "weasyprint", or "md_to_pdf"

        Returns:
            Path to generated PDF
        """
        md_path = Path(md_path)

        if not md_path.exists():
            raise FileNotFoundError(f"Markdown file not found: {md_path}")

        # Generate output path if not provided
        if pdf_path is None:
            pdf_path = md_path.with_suffix('.pdf')
        else:
            pdf_path = Path(pdf_path)

        print(f"📄 Converting: {md_path.name}")
        print(f"📥 Output: {pdf_path}")

        if method == "auto":
            # Try each method until one works
            for method_name, converter in self.methods:
                try:
                    print(f"   Trying {method_name}...")
                    result = converter(str(md_path), str(pdf_path))
                    if result and Path(pdf_path).exists():
                        print(f"✅ Success with {method_name}")
                        return str(pdf_path)
                except Exception as e:
                    print(f"   ❌ {method_name} failed: {e}")
                    continue

            raise RuntimeError("All PDF conversion methods failed")
        else:
            # Use specific method
            converter = dict(self.methods).get(method)
            if not converter:
                raise ValueError(f"Unknown method: {method}")
            return converter(str(md_path), str(pdf_path))

    def _convert_with_pandoc(self, md_path: str, pdf_path: str) -> str:
        """Convert using pandoc with LaTeX."""
        import pypandoc

        # Academic paper styling
        extra_args = [
            '--pdf-engine=xelatex',
            '-V', 'geometry:margin=1in',
            '-V', 'fontsize=11pt',
            '-V', 'documentclass=article',
            '-V', 'linkcolor=blue',
            '-V', 'toccolor=black',
            '--toc',                        # Table of contents
            '--toc-depth=3',
            '-V', 'toc-title=Table of Contents',
            '--highlight-style=tango',      # Code highlighting
            '-V', 'mainfont=DejaVu Serif',
            '-V', 'sansfont=DejaVu Sans',
            '-V', 'monofont=DejaVu Sans Mono',
        ]

        pypandoc.convert_file(
            md_path,
            'pdf',
            outputfile=pdf_path,
            extra_args=extra_args
        )

        return pdf_path

    def _convert_with_weasyprint(self, md_path: str, pdf_path: str) -> str:
        """Convert using WeasyPrint (HTML intermediate)."""
        import markdown
        from weasyprint import HTML, CSS

        # Read markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            md_content = f.read()

        # Convert to HTML
        html_content = markdown.markdown(
            md_content,
            extensions=['tables', 'fenced_code', 'toc', 'meta']
        )

        # Academic paper CSS
        css = CSS(string='''
            @page {
                size: letter;
                margin: 1in;
                @top-center { content: "Research Paper"; }
                @bottom-center { content: counter(page); }
            }

            body {
                font-family: "Times New Roman", Times, serif;
                font-size: 11pt;
                line-height: 1.6;
                color: #333;
            }

            h1 {
                font-size: 18pt;
                color: #000;
                border-bottom: 2px solid #333;
                padding-bottom: 10px;
                margin-top: 30px;
            }

            h2 {
                font-size: 14pt;
                color: #222;
                margin-top: 25px;
                border-bottom: 1px solid #ccc;
            }

            h3 {
                font-size: 12pt;
                color: #444;
                margin-top: 20px;
            }

            p {
                text-align: justify;
                margin-bottom: 12px;
            }

            img {
                max-width: 100%;
                height: auto;
                display: block;
                margin: 20px auto;
            }

            code {
                background-color: #f4f4f4;
                padding: 2px 5px;
                font-family: monospace;
                font-size: 10pt;
            }

            blockquote {
                border-left: 3px solid #ccc;
                padding-left: 15px;
                color: #666;
                font-style: italic;
            }

            table {
                border-collapse: collapse;
                width: 100%;
                margin: 20px 0;
            }

            th, td {
                border: 1px solid #ddd;
                padding: 8px;
                text-align: left;
            }

            th {
                background-color: #f2f2f2;
            }

            /* Citation styling */
            a {
                color: #0066cc;
                text-decoration: none;
            }

            /* Figure captions */
            em {
                display: block;
                text-align: center;
                font-size: 10pt;
                color: #666;
            }
        ''')

        # Wrap in full HTML
        full_html = f'''
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="utf-8">
            <title>Research Paper</title>
        </head>
        <body>
            {html_content}
        </body>
        </html>
        '''

        # Convert to PDF
        HTML(string=full_html).write_pdf(pdf_path, stylesheets=[css])

        return pdf_path

    def _convert_with_md2pdf(self, md_path: str, pdf_path: str) -> str:
        """Convert using md2pdf (simpler method)."""
        from md2pdf.core import md2pdf as convert_md2pdf

        convert_md2pdf(
            pdf_path,
            md_file_path=md_path,
            css_file_path=None,
            base_url=str(Path(md_path).parent)
        )

        return pdf_path


# Create converter instance
pdf_converter = ResearchPDFConverter()

print("✅ ResearchPDFConverter ready")

✅ ResearchPDFConverter ready


In [51]:
# =============================================================================
# PDF CONVERTER WITH EMBEDDED IMAGES
# =============================================================================

import base64
import re
from pathlib import Path
import markdown
from weasyprint import HTML, CSS

class ResearchPDFConverterWithImages:
    """
    Converts research markdown to PDF with properly embedded images.
    Images are converted to base64 and embedded directly in the HTML.
    """

    def __init__(self):
        pass

    def convert(self, md_path: str, pdf_path: str = None) -> str:
        """
        Convert markdown to PDF with embedded images.

        Args:
            md_path: Path to markdown file
            pdf_path: Output PDF path

        Returns:
            Path to generated PDF
        """
        md_path = Path(md_path)

        if not md_path.exists():
            raise FileNotFoundError(f"Markdown file not found: {md_path}")

        if pdf_path is None:
            pdf_path = md_path.with_suffix('.pdf')
        else:
            pdf_path = Path(pdf_path)

        print(f"📄 Converting: {md_path.name}")

        # Read markdown content
        with open(md_path, 'r', encoding='utf-8') as f:
            md_content = f.read()

        # Get the directory containing the markdown (for resolving relative paths)
        base_dir = md_path.parent

        # Process images - convert to base64
        md_content, image_count = self._embed_images(md_content, base_dir)
        print(f"🖼️  Embedded {image_count} images")

        # Convert markdown to HTML
        html_body = markdown.markdown(
            md_content,
            extensions=['tables', 'fenced_code', 'toc', 'md_in_html']
        )

        # Build full HTML document with academic styling
        html_doc = self._build_html_document(html_body)

        # Convert to PDF
        HTML(string=html_doc).write_pdf(str(pdf_path))

        # Get file size
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        print(f"✅ PDF saved: {pdf_path}")
        print(f"📦 Size: {size_mb:.2f} MB")

        return str(pdf_path)

    def _embed_images(self, md_content: str, base_dir: Path) -> tuple:
        """
        Find all images in markdown and embed as base64.

        Returns:
            (modified_content, image_count)
        """
        # Pattern to match markdown images: ![alt](path)
        img_pattern = r'!\[([^\]]*)\]\(([^)]+)\)'

        image_count = 0

        def replace_image(match):
            nonlocal image_count

            alt_text = match.group(1)
            img_path = match.group(2)

            # Skip if already a URL or data URI
            if img_path.startswith(('http://', 'https://', 'data:')):
                return match.group(0)

            # Resolve the image path
            if img_path.startswith('/'):
                # Absolute path
                full_path = Path(img_path)
            else:
                # Relative path - resolve from markdown file location
                full_path = (base_dir / img_path).resolve()

            # Also try from sections directory (common structure)
            if not full_path.exists():
                # Try relative to project root
                project_root = base_dir.parent if base_dir.name == 'sections' else base_dir
                full_path = (project_root / img_path.lstrip('./')).resolve()

            if not full_path.exists():
                # Try images directory directly
                full_path = base_dir / 'images' / Path(img_path).name

            if not full_path.exists():
                # Try parent's images directory
                full_path = base_dir.parent / 'images' / Path(img_path).name

            if full_path.exists():
                try:
                    # Read and encode image
                    with open(full_path, 'rb') as f:
                        img_data = f.read()

                    # Determine MIME type
                    suffix = full_path.suffix.lower()
                    mime_types = {
                        '.png': 'image/png',
                        '.jpg': 'image/jpeg',
                        '.jpeg': 'image/jpeg',
                        '.gif': 'image/gif',
                        '.webp': 'image/webp',
                        '.svg': 'image/svg+xml'
                    }
                    mime_type = mime_types.get(suffix, 'image/png')

                    # Encode to base64
                    b64_data = base64.b64encode(img_data).decode('utf-8')
                    data_uri = f"data:{mime_type};base64,{b64_data}"

                    image_count += 1
                    print(f"   ✓ Embedded: {full_path.name}")

                    return f'![{alt_text}]({data_uri})'

                except Exception as e:
                    print(f"   ⚠️ Failed to embed {full_path.name}: {e}")
                    return match.group(0)
            else:
                print(f"   ⚠️ Image not found: {img_path}")
                return match.group(0)

        # Replace all images
        modified_content = re.sub(img_pattern, replace_image, md_content)

        return modified_content, image_count

    def _build_html_document(self, html_body: str) -> str:
        """Build complete HTML document with academic styling."""

        return f'''
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Research Paper</title>
    <style>
        @page {{
            size: letter;
            margin: 0.75in 1in;

            @top-center {{
                content: string(paper-title);
                font-size: 9pt;
                color: #666;
            }}

            @bottom-center {{
                content: counter(page);
                font-size: 10pt;
            }}
        }}

        @page :first {{
            @top-center {{ content: none; }}
        }}

        body {{
            font-family: "Times New Roman", Times, Georgia, serif;
            font-size: 11pt;
            line-height: 1.6;
            color: #1a1a1a;
            text-align: justify;
            hyphens: auto;
        }}

        /* Title */
        h1 {{
            string-set: paper-title content();
            font-size: 20pt;
            font-weight: bold;
            text-align: center;
            color: #000;
            margin-bottom: 20px;
            border-bottom: none;
            page-break-after: avoid;
        }}

        /* Main sections */
        h2 {{
            font-size: 14pt;
            font-weight: bold;
            color: #000;
            margin-top: 24pt;
            margin-bottom: 12pt;
            border-bottom: 1px solid #ccc;
            padding-bottom: 4pt;
            page-break-after: avoid;
        }}

        /* Subsections */
        h3 {{
            font-size: 12pt;
            font-weight: bold;
            color: #222;
            margin-top: 18pt;
            margin-bottom: 8pt;
            page-break-after: avoid;
        }}

        /* Paragraphs */
        p {{
            margin-bottom: 10pt;
            text-indent: 0;
        }}

        /* First paragraph after heading - no indent */
        h1 + p, h2 + p, h3 + p {{
            text-indent: 0;
        }}

        /* Images */
        img {{
            max-width: 85%;
            height: auto;
            display: block;
            margin: 20pt auto;
            border: 1px solid #ddd;
            padding: 5px;
            background: #fff;
        }}

        /* Figure captions (em after images) */
        p > em:only-child,
        img + em {{
            display: block;
            text-align: center;
            font-size: 10pt;
            color: #555;
            margin-top: 8pt;
            margin-bottom: 16pt;
            font-style: italic;
        }}

        /* Code */
        code {{
            font-family: "Courier New", Courier, monospace;
            font-size: 9pt;
            background-color: #f5f5f5;
            padding: 1px 4px;
            border-radius: 2px;
        }}

        pre {{
            background-color: #f5f5f5;
            padding: 10pt;
            border: 1px solid #ddd;
            border-radius: 3px;
            overflow-x: auto;
            font-size: 9pt;
            line-height: 1.4;
        }}

        pre code {{
            background: none;
            padding: 0;
        }}

        /* Block quotes */
        blockquote {{
            margin: 15pt 30pt;
            padding-left: 15pt;
            border-left: 3px solid #ccc;
            color: #444;
            font-style: italic;
        }}

        /* Tables */
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 15pt 0;
            font-size: 10pt;
        }}

        th, td {{
            border: 1px solid #999;
            padding: 6pt 10pt;
            text-align: left;
        }}

        th {{
            background-color: #f0f0f0;
            font-weight: bold;
        }}

        tr:nth-child(even) {{
            background-color: #fafafa;
        }}

        /* Lists */
        ul, ol {{
            margin: 10pt 0;
            padding-left: 25pt;
        }}

        li {{
            margin-bottom: 5pt;
        }}

        /* Links / Citations */
        a {{
            color: #0066cc;
            text-decoration: none;
        }}

        /* Horizontal rules */
        hr {{
            border: none;
            border-top: 1px solid #ccc;
            margin: 20pt 0;
        }}

        /* Abstract styling */
        h2:first-of-type {{
            margin-top: 15pt;
        }}

        /* Keywords */
        p > strong:first-child {{
            font-weight: bold;
        }}

        /* References section */
        h2:last-of-type {{
            margin-top: 30pt;
        }}

        /* Page breaks */
        h1 {{
            page-break-before: avoid;
        }}

        h2 {{
            page-break-before: auto;
        }}

        /* Prevent orphans and widows */
        p {{
            orphans: 3;
            widows: 3;
        }}

        /* Keep figures together */
        figure, img {{
            page-break-inside: avoid;
        }}
    </style>
</head>
<body>
    {html_body}
</body>
</html>
'''


# Create converter instance
pdf_converter = ResearchPDFConverterWithImages()

print("✅ ResearchPDFConverterWithImages ready")

✅ ResearchPDFConverterWithImages ready


In [52]:
from google.colab import drive
drive.mount('/content/drive')

# Create a zip archive of the project folder
!zip -r /content/research_projects.zip /content/research_projects

# Copy the zip file to your Google Drive
!cp /content/research_projects.zip /content/drive/MyDrive/

Mounted at /content/drive
  adding: content/research_projects/ (stored 0%)


In [55]:
# ── Fix: patch SubAgent.execute to await async med_gemma callers ──
import inspect

async def _patched_execute(self, task, context=None):
    self._log(f"Task: {task[:80]}...")

    full_prompt = f"""{self.system_prompt}

TASK: {task}

{f"CONTEXT: {json.dumps(context)}" if context else ""}

Think through this task step by step, then execute the appropriate tools."""

    messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]

    response = self.med_gemma(
        messages=messages,
        max_new_tokens=self.gen_config.max_new_tokens,
        temperature=self.gen_config.temperature,
    )

    # Await if async caller (run_medgemma_async), call directly if sync (run_medgemma)
    if inspect.iscoroutine(response):
        response = await response

    response = self._clean_response(response)
    self._log(f"Response: {response[:200]}...")
    return response

SubAgent.execute = _patched_execute
print("✅ SubAgent.execute patched to support async med_gemma callers")

✅ SubAgent.execute patched to support async med_gemma callers


In [56]:
# ============================================================
# BENCHMARK RUNNER — All Topics, Concurrent I/O, Auto-Save
# ============================================================
#
# Strategy:
#   - Each topic gets its OWN memory + file_manager + orchestrator
#     so there are zero shared-state collisions.
#   - run_medgemma is wrapped in asyncio.to_thread() so async
#     tasks can yield during web searches.
#   - GPU_LOCK (Semaphore 1) ensures only ONE topic calls the
#     model at a time — no CUDA OOM, no serialization surprises.
#   - Web search calls (Tavily) run freely in parallel since
#     they are pure network I/O.
#   - Each completed topic auto-exports PDF → Google Drive.
#
# ============================================================

import asyncio
import time
import traceback
from datetime import datetime
from pathlib import Path
from google.colab import files, drive

# ── Config ────────────────────────────────────────────────────
TARGET_PAGES      = 20          # pages per report (20 ≈ fast; 80 = full)
ALLOW_IMAGES      = False       # set True if you want Gemini image gen
MAX_CONCURRENCY   = 3           # how many topics share the event loop
                                # (GPU still serialised; just overlaps I/O)
BASE_OUTPUT_DIR   = "/content/research_projects"
DRIVE_OUTPUT_DIR  = "/content/drive/MyDrive/DeepResearch_Benchmark"

# ── GPU lock — only one LLM call at a time ────────────────────
GPU_LOCK = asyncio.Semaphore(1)

# ── Wrap the blocking pipeline call so it yields to the loop ──
async def run_medgemma_async(messages, max_new_tokens=512, temperature=0.7, **kwargs):
    async with GPU_LOCK:
        return await asyncio.to_thread(
            run_medgemma,
            messages=messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            **kwargs
        )

# ── Progress tracker ──────────────────────────────────────────
class BenchmarkTracker:
    def __init__(self, topics):
        self.topics   = topics
        self.status   = {t: "⏳ queued"   for t in topics}
        self.start_t  = {t: None         for t in topics}
        self.end_t    = {t: None         for t in topics}
        self.pdf_path = {t: None         for t in topics}
        self.errors   = {t: None         for t in topics}
        self._lock    = asyncio.Lock()

    async def set(self, topic, status, pdf_path=None, error=None):
        async with self._lock:
            self.status[topic] = status
            if pdf_path: self.pdf_path[topic] = pdf_path
            if error:    self.errors[topic]   = error
            self._print()

    def _print(self):
        print("\033[2J\033[H", end="")   # clear screen
        print("=" * 70)
        print(f"  DEEP RESEARCH BENCHMARK  |  {datetime.now().strftime('%H:%M:%S')}")
        print("=" * 70)
        for i, t in enumerate(self.topics):
            label  = f"[{i+1:02d}]"
            title  = t[:50] + "…" if len(t) > 50 else t
            status = self.status[t]
            elapsed = ""
            if self.start_t[t]:
                secs = (self.end_t[t] or time.time()) - self.start_t[t]
                elapsed = f"  ({secs/60:.1f} min)"
            print(f"  {label} {status:<22} {title}{elapsed}")
        done    = sum(1 for s in self.status.values() if "✅" in s)
        failed  = sum(1 for s in self.status.values() if "❌" in s)
        running = sum(1 for s in self.status.values() if "🔄" in s)
        print("=" * 70)
        print(f"  Done: {done}/{len(self.topics)}  |  Running: {running}  |  Failed: {failed}")
        print("=" * 70)


# ── Per-topic research coroutine ──────────────────────────────
async def research_one_topic(topic: str, topic_idx: int, tracker: BenchmarkTracker):
    """
    Runs the full pipeline for a single topic:
      1. Creates its own isolated memory / file_manager / orchestrator.
      2. Uses run_medgemma_async so GPU calls are serialised globally.
      3. Exports PDF and saves to Drive on success.
    """
    tracker.start_t[topic] = time.time()
    await tracker.set(topic, "🔄 planning…")

    try:
        # ── Isolated state for this topic ──
        topic_memory      = PersistentMemory(base_dir=BASE_OUTPUT_DIR)
        topic_file_mgr    = FileManager(topic_memory)
        topic_formatter   = ResearchPaperFormatter(topic_memory, run_medgemma_async)
        topic_img_gen     = Gemini3ProImageGenerator(verbose=False) if ALLOW_IMAGES else None
        topic_ctx_img_gen = ContextualImageGenerator(topic_img_gen, verbose=False) if ALLOW_IMAGES else None

        topic_orchestrator = DeepResearchOrchestratorV4(
            med_gemma_caller      = run_medgemma_async,
            memory                = topic_memory,
            file_manager          = topic_file_mgr,
            search_tool           = TavilySearchTool(verbose=False),   # own instance
            image_generator       = topic_ctx_img_gen,
            paper_formatter       = topic_formatter,
            allow_image_generation= ALLOW_IMAGES,
            verbose               = False    # keep output clean; tracker handles display
        )

        # ── Run research ──
        await tracker.set(topic, "🔄 researching…")
        final_md_path = await topic_orchestrator.research(topic, target_pages=TARGET_PAGES)

        # ── Convert to PDF ──
        await tracker.set(topic, "🔄 exporting PDF…")
        pdf_converter = ResearchPDFConverterWithImages()
        pdf_path = str(Path(final_md_path).with_suffix(".pdf"))
        pdf_path = await asyncio.to_thread(pdf_converter.convert, final_md_path, pdf_path)

        # ── Save to Google Drive ──
        await asyncio.to_thread(_save_to_drive, pdf_path, topic_idx, topic)

        tracker.end_t[topic] = time.time()
        elapsed = (tracker.end_t[topic] - tracker.start_t[topic]) / 60
        await tracker.set(topic, f"✅ done ({elapsed:.1f}m)", pdf_path=pdf_path)

        return pdf_path

    except Exception as e:
        tracker.end_t[topic] = time.time()
        err_msg = str(e)[:80]
        await tracker.set(topic, f"❌ failed", error=err_msg)
        print(f"\n[ERROR] Topic {topic_idx}: {e}")
        traceback.print_exc()
        return None


# ── Helper: save PDF to Drive ─────────────────────────────────
def _save_to_drive(pdf_path: str, idx: int, topic: str):
    """Copy finished PDF to Google Drive (Drive must be mounted)."""
    import shutil
    dest_dir = Path(DRIVE_OUTPUT_DIR)
    dest_dir.mkdir(parents=True, exist_ok=True)
    safe_name = f"{idx+1:02d}_{topic[:40].replace(' ', '_').replace('/', '-')}.pdf"
    dest = dest_dir / safe_name
    shutil.copy2(pdf_path, dest)
    print(f"[Drive] Saved → {dest}")


# ── Main benchmark runner ─────────────────────────────────────
async def run_benchmark(topics=None, max_concurrent=MAX_CONCURRENCY):
    """
    Run all topics concurrently up to `max_concurrent` at a time.
    GPU calls are globally serialised via GPU_LOCK.
    Web-search I/O genuinely overlaps between topics.
    """
    if topics is None:
        topics = TEST_TOPICS

    print(f"\n🚀 Starting benchmark: {len(topics)} topics, max {max_concurrent} concurrent")
    print(f"   Target pages: {TARGET_PAGES}  |  Images: {ALLOW_IMAGES}")
    print(f"   Output: {BASE_OUTPUT_DIR}")
    print(f"   Drive:  {DRIVE_OUTPUT_DIR}\n")

    # Mount Drive once upfront
    try:
        drive.mount("/content/drive", force_remount=False)
        print("✅ Google Drive mounted")
    except Exception as e:
        print(f"⚠️  Drive mount failed ({e}) — PDFs will still save locally")

    tracker  = BenchmarkTracker(topics)
    semaphore = asyncio.Semaphore(max_concurrent)   # limits how many topics are active
    results  = {}

    async def bounded_task(topic, idx):
        async with semaphore:
            result = await research_one_topic(topic, idx, tracker)
            results[topic] = result

    # Launch all tasks; semaphore caps active count
    benchmark_start = time.time()
    await asyncio.gather(*[bounded_task(t, i) for i, t in enumerate(topics)])
    total_time = (time.time() - benchmark_start) / 60

    # ── Final summary ──
    print("\n\n" + "=" * 70)
    print("  BENCHMARK COMPLETE")
    print("=" * 70)
    succeeded = [(t, p) for t, p in results.items() if p]
    failed    = [(t, tracker.errors[t]) for t, p in results.items() if not p]

    print(f"\n✅ Succeeded ({len(succeeded)}/{len(topics)}):")
    for topic, path in succeeded:
        print(f"   • {topic[:60]}")
        print(f"     → {path}")

    if failed:
        print(f"\n❌ Failed ({len(failed)}/{len(topics)}):")
        for topic, err in failed:
            print(f"   • {topic[:60]}")
            print(f"     Error: {err}")

    print(f"\n⏱️  Total wall time: {total_time:.1f} minutes")

    # ── Zip everything to Drive ──
    print("\n📦 Creating zip archive of all projects…")
    zip_path = "/content/benchmark_results.zip"
    await asyncio.to_thread(
        lambda: __import__('subprocess').run(
            ["zip", "-r", zip_path, BASE_OUTPUT_DIR], capture_output=True
        )
    )
    drive_zip = Path(DRIVE_OUTPUT_DIR) / f"benchmark_{datetime.now().strftime('%Y%m%d_%H%M')}.zip"
    if Path(zip_path).exists():
        import shutil
        shutil.copy2(zip_path, drive_zip)
        print(f"✅ Archive saved to Drive: {drive_zip}")
    else:
        print("⚠️  Zip failed — download manually from /content/research_projects")

    return results


# ── Convenience: run a subset for quick testing ───────────────
async def run_benchmark_subset(indices: list, **kwargs):
    """
    Run only specific topics by index.
    Example: await run_benchmark_subset([0, 1, 2])  # first 3 topics
    """
    subset = [TEST_TOPICS[i] for i in indices]
    return await run_benchmark(topics=subset, **kwargs)


# ============================================================
# TO RUN:
# ============================================================
#
# All 10 topics (full benchmark):
#   results = await run_benchmark()
#
# Subset test (topics 0, 1, 2):
#   results = await run_benchmark_subset([0, 1, 2])
#
# Custom topics list:
#   MY_TOPICS = ["topic A", "topic B"]
#   results = await run_benchmark(topics=MY_TOPICS, max_concurrent=2)
#
# ============================================================

print("✅ Benchmark runner ready.")
print("\nQuick start:")
print("  results = await run_benchmark()               # all 10 topics")
print("  results = await run_benchmark_subset([0,1,2]) # first 3 only")

✅ Benchmark runner ready.

Quick start:
  results = await run_benchmark()               # all 10 topics
  results = await run_benchmark_subset([0,1,2]) # first 3 only


In [57]:
results = await run_benchmark_subset([1, 2, 6, 9])


🚀 Starting benchmark: 4 topics, max 3 concurrent
   Target pages: 20  |  Images: False
   Output: /content/research_projects
   Drive:  /content/drive/MyDrive/DeepResearch_Benchmark

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
  DEEP RESEARCH BENCHMARK  |  17:32:21
  [01] 🔄 planning…            Mechanical Ventilation in the ICU: A Literature Re…  (0.0 min)
  [02] ⏳ queued               CRISPR Gene Editing in Clinical Medicine: Current …
  [03] ⏳ queued               Antibiotic Resistance: Global Trends, Mechanisms, …
  [04] ⏳ queued               Regenerative Medicine and Stem Cell Therapy: Curre…
  Done: 0/4  |  Running: 1  |  Failed: 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[OverlapChecker] ✓ Sentence embeddings enabled
  DEEP RESEARCH BENCHMARK  |  17:32:25
  [01] 🔄 researching…         Mechanical Ventilation in the ICU: A Literature Re…  (0.1 min)
  [02] ⏳ queued               CRISPR Gene Editing in Clinical Medicine: Current …
  [03] ⏳ queued               Antibiotic Resistance: Global Trends, Mechanisms, …
  [04] ⏳ queued               Regenerative Medicine and Stem Cell Therapy: Curre…
  Done: 0/4  |  Running: 1  |  Failed: 0
✅ Created project: 938a06ded1b5
   Output directory: /content/research_projects/938a06ded1b5
  DEEP RESEARCH BENCHMARK  |  17:32:25
  [01] 🔄 researching…         Mechanical Ventilation in the ICU: A Literature Re…  (0.1 min)
  [02] 🔄 planning…            CRISPR Gene Editing in Clinical Medicine: Current …  (0.0 min)
  [03] ⏳ queued               Antibiotic Resistance: Global Trends, Mechanisms, …
  [04] ⏳ queued               Regenerative Medicine and Stem Cell Therapy: Curre…
  Done: 0/4  |  Running: 2  |  Failed: 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

✅ Saved section: /content/research_projects/32a429fa2b50/sections/sec_05_The_Regulatory_Maze_Comparing_Global_Approaches_to.md
   Words: 733


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:1 for

✅ Saved section: /content/research_projects/32a429fa2b50/sections/sec_06_Economic_Realities_Comparing_Market_Dynamics_and_B.md
   Words: 679
📄 Formatting 6 sections from: /content/research_projects/32a429fa2b50/sections
[Formatter] ⚠️ LLM abstract generation failed: 'coroutine' object has no attribute 'strip', using fallback
[Keywords] LLM extraction failed: 'coroutine' object has no attribute 'strip'

✅ RESEARCH PAPER FORMATTED
📄 Output: /content/research_projects/32a429fa2b50/RESEARCH_PAPER.md
📊 Words: 11,469 (~22.9 pages)
📚 Citations: 54
🖼️  Figures: 0
  DEEP RESEARCH BENCHMARK  |  06:48:10
  [01] ✅ done (722.3m)        Mechanical Ventilation in the ICU: A Literature Re…  (722.3 min)
  [02] ✅ done (565.1m)        CRISPR Gene Editing in Clinical Medicine: Current …  (565.1 min)
  [03] ✅ done (669.5m)        Antibiotic Resistance: Global Trends, Mechanisms, …  (669.5 min)
  [04] 🔄 exporting PDF…       Regenerative Medicine and Stem Cell Therapy: Curre…  (230.7 min)
  Done: 3/4  |  Run

DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.001s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.003s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

✅ PDF saved: /content/research_projects/32a429fa2b50/RESEARCH_PAPER.pdf
📦 Size: 0.12 MB
[Drive] Saved → /content/drive/MyDrive/DeepResearch_Benchmark/04_Regenerative_Medicine_and_Stem_Cell_Ther.pdf
  DEEP RESEARCH BENCHMARK  |  06:48:12
  [01] ✅ done (722.3m)        Mechanical Ventilation in the ICU: A Literature Re…  (722.3 min)
  [02] ✅ done (565.1m)        CRISPR Gene Editing in Clinical Medicine: Current …  (565.1 min)
  [03] ✅ done (669.5m)        Antibiotic Resistance: Global Trends, Mechanisms, …  (669.5 min)
  [04] ✅ done (230.7m)        Regenerative Medicine and Stem Cell Therapy: Curre…  (230.7 min)
  Done: 4/4  |  Running: 0  |  Failed: 0


  BENCHMARK COMPLETE

✅ Succeeded (4/4):
   • CRISPR Gene Editing in Clinical Medicine: Current Applicatio
     → /content/research_projects/c4f7f0da116c/RESEARCH_PAPER.pdf
   • Antibiotic Resistance: Global Trends, Mechanisms, and Novel 
     → /content/research_projects/b1d6aff87edf/RESEARCH_PAPER.pdf
   • Mechanical Ventilation in the 

##### **Human Review Module Documentation**

###### **Quick Human review reference card**
```
# =============================================================================
# HUMAN REVIEW QUICK REFERENCE
# =============================================================================

# --- PROJECT MANAGEMENT ---
review.list_projects()                    # List all projects
review.load_project('project_id')         # Load project for review

# --- VIEW CONTENT ---
review.show_summary()                     # Project overview
review.show_pending_reviews()             # Items needing attention
review.show_sections()                    # All sections with status
review.view_section('sec_01')             # Read section content
review.view_citations()                   # All citations
review.view_facts('sec_01')               # Facts for a section
review.view_facts(verified_only=True)     # Only verified facts

# --- ADD FEEDBACK ---
review.add_comment('sec_01', 'Great intro!')              # General comment
review.add_correction('sec_02', 'Fix date: 2024→2025')    # Correction needed
review.flag_for_verification('sec_03', 'Check 95% stat')  # Verify a fact
review.request_regeneration('sec_04', 'Too shallow')      # Rewrite section

# --- RESOLVE REVIEWS ---
review.resolve(1)                         # Approve item #1
review.resolve(1, 'Verified via PubMed')  # Approve with notes
review.resolve_all()                      # Approve all pending
review.resolve_all('Batch reviewed')      # Approve all with note

# --- EXPORT ---
review.export_review_report()             # Save review log to file

# --- RESUME AFTER CHANGES ---
await orchestrator.resume('project_id')   # Regenerate flagged sections
```

Fixes to Attempt

In [ ]:
# Inline citations Verified
# BEFORE (broken — filters on a key that doesn't exist yet)
filtered_facts = self.fact_filter.filter_facts(facts_as_dicts, section.section_id)

# AFTER (correct — skip section filtering, facts are already section-scoped)
filtered_facts = self.fact_filter.filter_facts(facts_as_dicts)

In [ ]:
# Can remove this now — filtered_facts won't be empty due to section mismatch
# if not filtered_facts and facts_as_dicts:
#     self._log(" [WriterAgent] Section filter returns Nothing. Using all facts")
#     filtered_facts = facts_as_dicts

In [ ]:
"""
# use instead:
if not filtered_facts and facts_as_dicts:
    self._log("  ⚠️ All facts filtered out by confidence/citation rules. Using unfiltered.")
    filtered_facts = facts_as_dicts
"""